In [1]:
!pip install mordred==1.2.0       --no-deps -q
!pip install scipy==1.17.0        --no-deps -q
!pip install scikit-learn==1.8.0  --no-deps -q
!pip install imbalanced-learn==0.14.1 --no-deps -q
!pip install umap-learn           --no-deps -q
!pip install rdkit                --no-deps -q
!pip install lightgbm             --no-deps -q
!pip install mendeleev            --no-deps -q
!pip install morfeus-ml           --no-deps -q
!pip install networkx             --no-deps -q
!pip install optuna               --no-deps -q
!pip install colorlog             --no-deps -q
!pip install drfp                 --no-deps -q
!pip install dscribe              --no-deps -q

!pip install "numpy==2.0.2" -q

In [ ]:
# Run this cell ALONE first
import os
os.kill(os.getpid(), 9)  # force-kills the runtime, equivalent to "Restart session"


# Featurization Block Summary

This notebook constructs a comprehensive feature matrix for MOF synthesis prediction by combining multiple distinct descriptor blocks. Below is a summary of each block:

### 1. Metal Center Features (Block A)
*   **Source:** `mendeleev` + heuristic logic
*   **Description:** Captures the fundamental atomic properties of the metal node.
*   **Key Features:** Atomic number, period, group, electronegativity (Pauling/Allen), atomic/covalent/vdW radii, ionization energies, valence electrons, oxidation state (parsed from IUPAC), d-electron count, and geometry flags (square planar, tetrahedral, octahedral).

### 2. Co-ligand Inventory (Block B)
*   **Source:** RDKit + Lookup Tables
*   **Description:** Characterizes the simple ligands attached to the metal precursor (e.g., halides, CO, phosphines).
*   **Key Features:** Counts and properties of specific ligands (CO, Cl, Br, I, PPh3), electronic parameters (sigma-donor, pi-acceptor strength), and net charge of the co-ligand sphere.

### 3. Complex-level Descriptors (Block C)
*   **Source:** RDKit
*   **Description:** Global properties of the metal precursor complex.
*   **Key Features:** Dimer/cluster flags, total coordination number, homoleptic status, estimated precursor charge, number of unique ligand types, and presence of bridging halides or metal-metal bonds.

### 4. Revised Autocorrelation (RAC) Descriptors
*   **Source:** `mordred`
*   **Description:** Encodes topological and electronic structure of ligands.
*   **Key Features:** Weighted sum of Autocorrelation descriptors (ATS, MATS, GATS) for all precursor ligands and modulators. Includes missingness indicators for unparseable ligands.

### 5. Physicochemical Descriptors
*   **Source:** `rdkit.Chem.Descriptors`
*   **Description:** General molecular properties for linkers and modulators.
*   **Key Features:** Molecular weight, LogP, TPSA, number of rotatable bonds, H-bond donors/acceptors, Hall-Kier alpha, aromatic ring count, and Gasteiger partial charges.

### 6. Tolman Electronic Parameter (TEP)
*   **Source:** `LGBMRegressor` (pre-trained model from Daniel Ess lab)
*   **Description:** Predicts the electronic donating ability of phosphine and N-heterocyclic carbene ligands.
*   **Key Features:** Predicted TEP value ($cm^{-1}$) for modulators, linkers, and precursor ligands. Includes flags for missing/non-applicable values.

### 7. Steric Descriptors (Cone Angle & Buried Volume)
*   **Source:** `morfeus`
*   **Description:** Captures the spatial bulk of ligands, critical for surface chemistry and pore formation.
*   **Key Features:** Exact Cone Angle and Percent Buried Volume (%V_bur) for phosphine-containing linkers, modulators, and precursor ligands.

### 8. ChemBERTa-2 Embeddings
*   **Source:** `DeepChem/ChemBERTa-77M-MTR` (Hugging Face)
*   **Description:** Transformer-based embeddings pre-trained on 77 million molecules.
*   **Key Features:** 384-dimensional CLS token vector providing a dense, context-aware representation of the linker and modulator molecular structures.

### 9. Extended RDKit Descriptors
*   **Source:** `rdkit`
*   **Description:** A curated suite of 2D descriptors focusing on complexity, topology, and functional groups.
*   **Key Features:** BertzCT, MolMR, LabuteASA, FractionCSP3, Chi/Kappa connectivity indices, ring counts (aliphatic/saturated), partial charge statistics, and counts of specific MOF-relevant SMARTS patterns (e.g., COOH, pyridyl N).

### 10. 3D Shape Descriptors
*   **Source:** `rdkit.Chem.rdMolDescriptors` (via ETKDGv3 conformers)
*   **Description:** Captures the 3D geometry of the organic linkers/modulators.
*   **Key Features:** Principal Moments of Inertia (PMI), Normalized Principal Moments Ratio (NPR), Asphericity, Eccentricity, Spherocity Index, and Radius of Gyration.

### 11. Tetratopic Phosphine (TTP) Topology
*   **Source:** Custom graph traversal logic
*   **Description:** Specifically designed for Group 14 (Si, Ge, Sn, C) centered tetratopic linkers.
*   **Key Features:** Hub element identity, arm length statistics (min/max/mean), arm backbone composition (alkyl/aryl/alkynyl), topicity (number of P-arms), and hub graph eccentricity.

### 12. Reaction Fingerprint (DRFP)
*   **Source:** `drfp`
*   **Description:** Encodes the reaction as a whole (precursor + linker + modulator).
*   **Key Features:** 2048-bit hashed fingerprint capturing the chemical transformation and local environments of all reactants simultaneously.


In [1]:
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Global seed set to {SEED}")

Global seed set to 42


In [2]:
import pandas as pd

# Read the Excel file
file_path = "/content/Experiments_with_Calculated_Properties_no_linker.xlsx"
df = pd.read_excel(file_path)

# Display the first 5 rows
print("First 5 rows of the dataset:")
display(df.head())

# Print the list of column names
print("\nList of column names:")
print(df.columns.tolist())

First 5 rows of the dataset:


,source_file,experiment_id,experiment_id.1,precursor_iupac_standardized,smiles_precursor,linker_structure,smiles_linker_1,smiles_linker_2,modulator,smiles_modulator,...,Mix_M2_Polarity,Mix_M3_Asymmetry,Mix_M_HB_Acc,Mix_M_HB_Don,temperature_k,Unnamed: 45,pxrd_comments,pxrd_manual,reaction_hours,pxrd_score
0,2021-RES-General/2021-RES-General/experiments/...,RES-3-004a,RES-3-004a,tetrakis(triphenylphosphine)palladium(0),P(C1=CC=CC=C1)(C2=CC=CC=C2)C3=CC=CC=C3.P(C4=CC...,"1,4-bis((diphenylphosphaneyl)ethynyl)benzene",C1(C#CP(C2=CC=CC=C2)C3=CC=CC=C3)=CC=C(C#CP(C4=...,NaN,PPh3,P(C1=CC=CC=C1)(C2=CC=CC=C2)C3=CC=CC=C3,...,0.002372,-0.000005,0.0,0.009513,333.0,NaN,No solids formed; solutions remained dark oran...,True,18.0,0
1,2021-RES-General/2021-RES-General/experiments/...,RES-3-004b,RES-3-004b,tetrakis(triphenylphosphine)palladium(0),P(C1=CC=CC=C1)(C2=CC=CC=C2)C3=CC=CC=C3.P(C4=CC...,"1,4-bis((diphenylphosphaneyl)ethynyl)benzene",C1(C#CP(C2=CC=CC=C2)C3=CC=CC=C3)=CC=C(C#CP(C4=...,NaN,PPh3,P(C1=CC=CC=C1)(C2=CC=CC=C2)C3=CC=CC=C3,...,0.002372,-0.000005,0.0,0.009513,333.0,NaN,No solids formed; solutions remained dark oran...,True,18.0,0
2,2021-RES-General/2021-RES-General/experiments/...,RES-3-004c,RES-3-004c,tetrakis(triphenylphosphine)palladium(0),P(C1=CC=CC=C1)(C2=CC=CC=C2)C3=CC=CC=C3.P(C4=CC...,"1,4-bis((diphenylphosphaneyl)ethynyl)benzene",C1(C#CP(C2=CC=CC=C2)C3=CC=CC=C3)=CC=C(C#CP(C4=...,NaN,PPh3,P(C1=CC=CC=C1)(C2=CC=CC=C2)C3=CC=CC=C3,...,0.002372,-0.000005,0.0,0.009513,333.0,NaN,Isolated solid looked somewhat crystalline und...,True,48.0,0
3,2021-RES-General/2021-RES-General/experiments/...,RES-3-004d,RES-3-004d,tetrakis(triphenylphosphine)palladium(0),P(C1=CC=CC=C1)(C2=CC=CC=C2)C3=CC=CC=C3.P(C4=CC...,"1,4-bis((diphenylphosphaneyl)ethynyl)benzene",C1(C#CP(C2=CC=CC=C2)C3=CC=CC=C3)=CC=C(C#CP(C4=...,NaN,PPh3,P(C1=CC=CC=C1)(C2=CC=CC=C2)C3=CC=CC=C3,...,0.002372,-0.000005,0.0,0.009513,333.0,NaN,Solid (when briefly present) completely dissol...,True,48.0,1
4,2021-RES-General/2021-RES-General/experiments/...,RES-3-005a,RES-3-005a,Cu4I4(PPh3)4,I12[Cu]3([P](C4=CC=CC=C4)(C5=CC=CC=C5)C6=CC=CC...,"1,4-bis((diphenylphosphaneyl)ethynyl)benzene",C1(C#CP(C2=CC=CC=C2)C3=CC=CC=C3)=CC=C(C#CP(C4=...,NaN,PPh3,P(C1=CC=CC=C1)(C2=CC=CC=C2)C3=CC=CC=C3,...,0.002419,-0.000010,0.0,0.015217,298.0,NaN,No PXRD reported. Vial remained a clear soluti...,False,48.0,0



List of column names:
['source_file', 'experiment_id', 'experiment_id.1', 'precursor_iupac_standardized', 'smiles_precursor', 'linker_structure', 'smiles_linker_1', 'smiles_linker_2', 'modulator', 'smiles_modulator', 'umol_metal_precursor', 'umol_linker', 'umol_modulator', 'metal_conc', 'linker_conc', 'mod_conc', 'total_conc', 'total_solvent_volume_ml', 'metal_over_linker_ratio', 'equivalents', 'solvent_1', 'solvent_1_volume_ml', 'solvent_1_fraction', 'solvent_2', 'solvent_2_volume_ml', 'solvent_2_fraction', 'solvent_3', 'solvent_3_volume_ml', 'solvent_3_fraction', 'Min_Boiling_Point_K', 'Max_Boiling_Point_K', 'Weighted_Boiling_Point_K', 'Weighted_AN_mole', 'Weighted_DN_mole', 'Weighted_Dielectric_vol', 'Weighted_Polarity_vol', 'Weighted_sig_h_vol', 'Weighted_sig_d_vol', 'Weighted_sig_p_vol', 'Mix_M0_Area', 'Mix_M2_Polarity', 'Mix_M3_Asymmetry', 'Mix_M_HB_Acc', 'Mix_M_HB_Don', 'temperature_k', 'Unnamed: 45', 'pxrd_comments', 'pxrd_manual', 'reaction_hours', 'pxrd_score']


In [3]:
# After: df = pd.read_excel(...)
# Before: deconstruct_precursor loop, df.merge(...), fingerprints, RACs, etc.

COLMAP = {
    "id": "experiment_id",
    "precursor": "smiles_precursor",
    "linker1": "smiles_linker_1",
    "modulator": "smiles_modulator",
    "linker2": "smiles_linker_2",  # optional
}


In [4]:
from rdkit import Chem, rdBase
from collections import Counter

def canonicalize_smiles(smiles):
    """Converts a SMILES string to its unique, standard canonical form."""
    if pd.isna(smiles) or not isinstance(smiles, str) or smiles.strip() == "":
        return None
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol:
            return Chem.MolToSmiles(mol, canonical=True)
    except:
        pass
    return None


def deconstruct_precursor(smiles):
    """
    Deconstructs a precursor SMILES by removing metal atoms and counting the remaining ligand fragments.
    Explicitly handles coordination complexes by removing the metal node from the graph.
    """
    target_metals = {'Pd', 'Rh', 'Pt', 'Ag', 'Ir', 'Au', 'Cu', 'Co', 'Ni', 'Fe', 'Ru', 'Os'}

    if not isinstance(smiles, str):
        return None, {}

    # Disable RDKit error logs temporarily
    rdBase.DisableLog('rdApp.error')

    try:
        # Attempt to create molecule
        mol = Chem.MolFromSmiles(smiles)

        # Fallback to unsanitized if standard parsing fails
        if mol is None:
            mol = Chem.MolFromSmiles(smiles, sanitize=False)

        if mol is None:
            rdBase.EnableLog('rdApp.error')
            return None, {}

        # Convert to editable molecule
        rwmol = Chem.RWMol(mol)

        # Identify metal atoms
        metal_indices = []
        found_metal_symbol = None

        for atom in rwmol.GetAtoms():
            if atom.GetSymbol() in target_metals:
                metal_indices.append(atom.GetIdx())
                if found_metal_symbol is None:
                    found_metal_symbol = atom.GetSymbol()

        # Remove metal atoms (descending order)
        metal_indices.sort(reverse=True)
        for idx in metal_indices:
            rwmol.RemoveAtom(idx)

        # Extract remaining fragments
        # Convert back to Mol to ensure stability for GetMolFrags
        frag_mol = rwmol.GetMol()
        fragments = Chem.GetMolFrags(frag_mol, asMols=True, sanitizeFrags=False)

        # Convert fragments to SMILES and count
        ligand_counts = Counter()
        for frag in fragments:
            try:
                lig_smiles = Chem.MolToSmiles(frag, canonical=True)
                if lig_smiles:
                    ligand_counts[lig_smiles] += 1
            except:
                pass

        return found_metal_symbol, dict(ligand_counts)

    finally:
        # Ensure logs are re-enabled even if an error occurs
        rdBase.EnableLog('rdApp.error')

In [ ]:
inventory_data = []

# Loop through the dataset
for index, row in df.iterrows():
    # Deconstruct precursor using the helper function
    metal, precursor_ligands = deconstruct_precursor(row['smiles_precursor'])

    # Process modulator
    mod_smiles_raw = row['smiles_modulator']
    modulator_smiles = None

    # Attempt to canonicalize if it's a string
    if isinstance(mod_smiles_raw, str):
        # Temporarily disable logs to avoid clutter from invalid SMILES
        rdBase.DisableLog('rdApp.error')
        canon = canonicalize_smiles(mod_smiles_raw)
        rdBase.EnableLog('rdApp.error')

        if canon:
            modulator_smiles = canon
        else:
            # Fallback to raw value if parsing fails
            modulator_smiles = mod_smiles_raw
    else:
        # Use raw value if not a string (e.g., NaN or other types)
        modulator_smiles = mod_smiles_raw

    # Retrieve equivalents
    try:
        if pd.notnull(row['equivalents']):
            mod_equiv = float(row['equivalents'])
        else:
            mod_equiv = 0.0
    except (ValueError, TypeError):
        mod_equiv = 0.0

    # Initialize row dict
    row_dict = {
        'experiment_id': row['experiment_id'],
        'metal_atom': metal
    }

    # Add precursor ligands
    for lig, count in precursor_ligands.items():
        key = f'Total_{lig}'
        row_dict[key] = row_dict.get(key, 0.0) + count

    # Add modulator if valid
    # Check if it is a non-empty string and not NaN (pandas treat NaN as float usually, but check notnull just in case)
    if isinstance(modulator_smiles, str) and modulator_smiles:
        key = f'Total_{modulator_smiles}'
        row_dict[key] = row_dict.get(key, 0.0) + mod_equiv

    inventory_data.append(row_dict)

# Create DataFrame
df_inventory = pd.DataFrame(inventory_data)

# Fill NaNs with 0
df_inventory = df_inventory.fillna(0)

# Display the first 5 rows
print("Regenerated Ligand Inventory (Metal Removal Method). First 5 rows:")
display(df_inventory.head())

In [ ]:



import numpy as np
import pandas as pd
from mendeleev import element as mendeleev_element

if 'df_merged' not in globals():
    # 'df' and 'df_inventory' must exist from preceding cells.
    if 'df' not in globals():
        raise NameError("DataFrame 'df' is not defined. Please run the cell that reads the Excel file (e.g., cell 7c2fa441).")
    if 'df_inventory' not in globals():
        raise NameError("DataFrame 'df_inventory' is not defined. Please run the cell that creates the ligand inventory (e.g., cell 7b696307).")
    df_merged = pd.merge(df, df_inventory, on="experiment_id", how="left")

# -----------------------------------------------------------
# 1. Define the metals present in dataset
#    These are the same as deconstructprecursor's targetmetals
# -----------------------------------------------------------
TARGET_METALS = ['Pd', 'Rh', 'Pt', 'Ag', 'Ir', 'Au', 'Cu', 'Co', 'Ni', 'Fe', 'Ru', 'Os']

# -----------------------------------------------------------
# 2. Pre-build a cache of descriptors for every target metal
# -----------------------------------------------------------
def get_metal_descriptors(symbol: str) -> dict:
    """
    Fetch atomic descriptors for a metal element using mendeleev.
    Returns a flat dict of float values. Missing values → 0.0.
    All values are physically meaningful scalars — no hallucination risk.
    """
    try:
        el = mendeleev_element(symbol)
    except Exception:
        # Unknown symbol — return all zeros with missing flag
        return {
            'metal_atomic_number':       0.0,
            'metal_period':              0.0,
            'metal_group':               0.0,
            'metal_en_pauling':          0.0,
            'metal_en_allen':            0.0,
            'metal_covalent_radius_pm':  0.0,
            'metal_atomic_radius_pm':    0.0,
            'metal_vdw_radius_pm':       0.0,
            'metal_ionization_energy1_eV': 0.0,
            'metal_ionization_energy2_eV': 0.0,
            'metal_electron_affinity_eV': 0.0,
            'metal_nvalence':            0.0,
            'metal_max_oxidation_state': 0.0,
            'metal_min_oxidation_state': 0.0,
            'metal_common_oxidation_states_count': 0.0,
            'metal_mendeleev_number':    0.0,
            'metal_hardness_eV':         0.0,
            'metal_dipole_polarizability_bohr3': 0.0,
            'metal_missing_flag':        1.0,
        }

    # --- Helper: safe float conversion ---
    def safe(val):
        try:
            v = float(val)
            return v if np.isfinite(v) else 0.0
        except (TypeError, ValueError):
            return 0.0

    # --- Ionization energies (1st and 2nd) ---
    try:
        ie_dict = el.ionenergies  # {1: float, 2: float, ...}
        ie1 = safe(ie_dict.get(1, 0.0))
        ie2 = safe(ie_dict.get(2, 0.0))
    except Exception:
        ie1, ie2 = 0.0, 0.0

    # --- Oxidation states (main only) ---
    try:
        ox_states = [os.oxidation_state for os in el.oxidation_states
                     if os.category == 'main']
        if not ox_states:
            # Fallback to all if 'main' is empty
            ox_states = [os.oxidation_state for os in el.oxidation_states]
        max_ox = safe(max(ox_states)) if ox_states else 0.0
        min_ox = safe(min(ox_states)) if ox_states else 0.0
        n_ox   = float(len(ox_states))
    except Exception:
        max_ox, min_ox, n_ox = 0.0, 0.0, 0.0

    # --- Number of valence electrons ---
    try:
        nval = safe(el.nvalence())
    except Exception:
        nval = 0.0

    # --- Hardness (Pearson absolute hardness = (IE1 - EA) / 2) ---
    #     mendeleev computes this as el.hardness (eV)
    try:
        hardness = safe(el.hardness())
    except Exception:
        # Manual fallback
        ea = safe(el.electron_affinity) if el.electron_affinity is not None else 0.0
        hardness = (ie1 - ea) / 2.0 if ie1 > 0 else 0.0

    # --- Covalent radius: prefer Pyykko single-bond, fallback Bragg ---
    try:
        cov_r = safe(el.covalent_radius_pyykko)
        if cov_r == 0.0:
            cov_r = safe(el.covalent_radius_bragg)
    except Exception:
        cov_r = 0.0

    return {
        'metal_atomic_number':           safe(el.atomic_number),
        'metal_period':                  safe(el.period),
        'metal_group':                   safe(el.group_id),
        'metal_en_pauling':              safe(el.en_pauling),
        'metal_en_allen':                safe(el.en_allen),
        'metal_covalent_radius_pm':      cov_r,
        'metal_atomic_radius_pm':        safe(el.atomic_radius),
        'metal_vdw_radius_pm':           safe(el.vdw_radius),
        'metal_ionization_energy1_eV':   ie1,
        'metal_ionization_energy2_eV':   ie2,
        'metal_electron_affinity_eV':    safe(el.electron_affinity),
        'metal_nvalence':                nval,
        'metal_max_oxidation_state':     max_ox,
        'metal_min_oxidation_state':     min_ox,
        'metal_common_oxidation_states_count': n_ox,
        'metal_mendeleev_number':        safe(el.mendeleev_number),
        'metal_hardness_eV':             hardness,
        'metal_dipole_polarizability_bohr3': safe(el.dipole_polarizability),
        'metal_missing_flag':            0.0,
    }

# -----------------------------------------------------------
# 3. Build cache once for all target metals
# -----------------------------------------------------------
print("Building metal descriptor cache via mendeleev...")
metal_descriptor_cache = {}

for sym in TARGET_METALS:
    metal_descriptor_cache[sym] = get_metal_descriptors(sym)
    d = metal_descriptor_cache[sym]
    print(f"  {sym:3s} | Z={d['metal_atomic_number']:.0f} | "
          f"EN={d['metal_en_pauling']:.2f} | "
          f"IE1={d['metal_ionization_energy1_eV']:.2f} eV | "
          f"Cov.r={d['metal_covalent_radius_pm']:.0f} pm | "
          f"Valence={d['metal_nvalence']:.0f} | "
          f"Mendeleev#={d['metal_mendeleev_number']:.0f}")

# Descriptor names (for feature naming later)
METAL_DESCRIPTOR_NAMES = list(list(metal_descriptor_cache.values())[0].keys())
N_METAL_DESCRIPTORS = len(METAL_DESCRIPTOR_NAMES)
print(f"\n{N_METAL_DESCRIPTORS} metal descriptors per experiment: {METAL_DESCRIPTOR_NAMES}")

# -----------------------------------------------------------
# 4. Map descriptors onto df_merged using the metal_atom column
#    (metal_atom was assigned during deconstructprecursor)
# -----------------------------------------------------------

# Ensure df_merged has metal_atom from df_inventory
if 'metal_atom' not in df_merged.columns:
    df_merged = pd.merge(df_merged, df_inventory[['experiment_id', 'metal_atom']],
                        on='experiment_id', how='left')

# Unknown/missing metal → use zero vector with missing_flag=1
_zero_descriptor = get_metal_descriptors('UNKNOWN_METAL_SYMBOL_XYZ')

def lookup_metal_descriptors(metal_symbol):
    """Look up cached descriptors, return zero vector if unknown."""
    if pd.isna(metal_symbol) or not isinstance(metal_symbol, str):
        return _zero_descriptor
    sym = str(metal_symbol).strip()
    if sym in metal_descriptor_cache:
        return metal_descriptor_cache[sym]
    # Attempt live lookup for any metal not in TARGET_METALS
    try:
        desc = get_metal_descriptors(sym)
        metal_descriptor_cache[sym] = desc  # Cache for reuse
        return desc
    except Exception:
        return _zero_descriptor

# Build feature matrix rows
metal_descriptor_rows = df_merged['metal_atom'].apply(lookup_metal_descriptors)

# Convert to numpy array (shape: [n_samples, N_METAL_DESCRIPTORS])
X_metal = np.array([
    [row[name] for name in METAL_DESCRIPTOR_NAMES]
    for row in metal_descriptor_rows
], dtype=float)

print(f"\nX_metal shape: {X_metal.shape}")
print(f"Non-zero rows: {(X_metal.sum(axis=1) != 0).sum()} / {len(X_metal)}")
print(f"Missing flag count: {int(X_metal[:, -1].sum())} experiments with unknown metal")

# Sanity check: no NaN/Inf
bad = ~np.isfinite(X_metal)
if bad.any():
    print(f"WARNING: {bad.sum()} non-finite values found — replacing with 0.0")
    X_metal[bad] = 0.0
else:
    print("✓ All values finite")

# -----------------------------------------------------------
# 5. One-hot encode metal identity (keeps hard identity signal
#    separate from continuous descriptors)
# -----------------------------------------------------------
metal_ohe_df = pd.get_dummies(
    df_merged['metal_atom'].fillna('Unknown'),
    prefix='metal_is'
)

# Ensure all TARGET_METALS columns exist even if absent from this dataset
for sym in TARGET_METALS:
    col = f'metal_is_{sym}'
    if col not in metal_ohe_df.columns:
        metal_ohe_df[col] = 0

# Reorder columns consistently
ohe_cols = sorted([c for c in metal_ohe_df.columns if c.startswith('metal_is_')])
metal_ohe_df = metal_ohe_df[ohe_cols]
X_metal_ohe = metal_ohe_df.values.astype(float)
metal_names = METAL_DESCRIPTOR_NAMES + ohe_cols
print(f"\nX_metal_ohe shape: {X_metal_ohe.shape}")
print(f"OHE columns: {ohe_cols}")


In [ ]:
# ── Process Variable Audit ────────────────────────────────────────────────────
import pandas as pd
import numpy as np

process_cols = [
    'equivalents', 'total_solvent_volume_ml', 'solvent_1_fraction',
    'solvent_2_fraction', 'solvent_3_fraction', 'Min_Boiling_Point_K',
    'Max_Boiling_Point_K', 'Weighted_Boiling_Point_K', 'Weighted_AN_mole',
    'Weighted_DN_mole', 'Weighted_Dielectric_vol', 'Weighted_Polarity_vol',
    'Weighted_sig_h_vol', 'Weighted_sig_d_vol', 'Weighted_sig_p_vol',
    'Mix_M0_Area', 'Mix_M2_Polarity', 'Mix_M3_Asymmetry', 'Mix_M_HB_Acc',
    'Mix_M_HB_Don', 'temperature_k', 'metal_over_linker_ratio', 'reaction_hours',
    'umol_metal_precursor', 'umol_linker', 'umol_modulator',
    'metal_conc', 'linker_conc', 'mod_conc', 'total_conc',
]

process_cols_present = [c for c in process_cols if c in df_merged.columns]
process_df = df_merged[process_cols_present].apply(pd.to_numeric, errors='coerce')

# ── 1. Global NaN summary ─────────────────────────────────────────────────────
print("=" * 65)
print("  STEP 1 — NaN counts per column (raw, before any imputation)")
print("=" * 65)
nan_summary = process_df.isnull().sum()
total_nan = nan_summary.sum()
print(f"  Total NaN entries across all process vars: {total_nan}")
print()
for col, n in nan_summary.items():
    if n > 0:
        pct = 100 * n / len(process_df)
        print(f"  {col:<35} {n:>4} NaNs  ({pct:.1f}%)")
if total_nan == 0:
    print("  ✅ No NaNs found — issue is NOT missing values from the source file.")

# ── 2. Coercion failures — values that became NaN after to_numeric ─────────────
print()
print("=" * 65)
print("  STEP 2 — Values coerced to NaN (non-numeric strings in source)")
print("=" * 65)
raw_df = df_merged[process_cols_present]
coercion_failures = {}
for col in process_cols_present:
    original = raw_df[col]
    coerced  = pd.to_numeric(original, errors='coerce')
    bad_mask = coerced.isnull() & original.notna()  # was something, became NaN
    if bad_mask.any():
        coercion_failures[col] = df_merged.loc[bad_mask, 'experiment_id'].tolist()
        print(f"\n  ⚠️  {col} — {bad_mask.sum()} coercion failure(s):")
        print(f"      Raw values: {original[bad_mask].unique().tolist()}")
        print(f"      Experiments: {coercion_failures[col][:10]}")

if not coercion_failures:
    print("  ✅ No coercion failures — all values are numeric or truly NaN.")

# ── 3. Temperature-specific check ─────────────────────────────────────────────
print()
print("=" * 65)
print("  STEP 3 — Temperature sanity check (expected: 298–393 K)")
print("=" * 65)
if 'temperature_k' in process_df.columns:
    temp = process_df['temperature_k']
    low_mask  = temp.notna() & (temp < 200)
    high_mask = temp.notna() & (temp > 500)
    null_mask = temp.isnull()
    print(f"  NaN:            {null_mask.sum()} experiments")
    print(f"  < 200 K (cold): {low_mask.sum()} experiments")
    print(f"  > 500 K (hot):  {high_mask.sum()} experiments")
    if low_mask.any():
        print(f"\n  🔴 Suspiciously cold experiments:")
        display(df_merged.loc[low_mask, ['experiment_id', 'temperature_k',
                                          'precursor_iupac_standardized',
                                          'metal_atom']].head(20))
    if null_mask.any():
        print(f"\n  🔴 Experiments with no temperature recorded:")
        display(df_merged.loc[null_mask, ['experiment_id', 'temperature_k',
                                           'precursor_iupac_standardized',
                                           'metal_atom']].head(20))

# ── 4. Find ALL experiments with ANY bad process variable ─────────────────────
print()
print("=" * 65)
print("  STEP 4 — All experiments with ≥1 NaN process variable")
print("=" * 65)
any_nan_mask = process_df.isnull().any(axis=1)
print(f"  {any_nan_mask.sum()} / {len(process_df)} experiments have at least one NaN")

if any_nan_mask.any():
    # Which columns are NaN for each bad experiment
    bad_df = df_merged.loc[any_nan_mask, ['experiment_id', 'metal_atom',
                                           'precursor_iupac_standardized']].copy()
    bad_df['nan_columns'] = process_df[any_nan_mask].apply(
        lambda row: [c for c in row.index if pd.isnull(row[c])], axis=1
    )
    bad_df['n_nan_cols'] = bad_df['nan_columns'].apply(len)
    bad_df = bad_df.sort_values('n_nan_cols', ascending=False)
    print(f"\n  Worst offenders (most missing process vars):")
    display(bad_df.head(30))

    # ── 5. Pattern — are bad experiments clustered by metal or source file? ──
    print()
    print("=" * 65)
    print("  STEP 5 — Are NaN experiments clustered by metal or source?")
    print("=" * 65)
    print("\n  Metal breakdown of NaN experiments:")
    print(df_merged.loc[any_nan_mask, 'metal_atom'].value_counts().to_string())
    if 'source_file' in df_merged.columns:
        print("\n  Source file breakdown of NaN experiments:")
        print(df_merged.loc[any_nan_mask, 'source_file'].value_counts().to_string())


In [ ]:
worst_ids = ['RN175a', 'RN-96', 'RN-42', 'RN 158a', 'RN 158b', 'RN 158c']
worst_mask = df_merged['experiment_id'].isin(worst_ids)

solvent_cols = ['solvent_1', 'solvent_2', 'solvent_3',
                'total_solvent_volume_ml', 'solvent_1_fraction',
                'temperature_k', 'reaction_hours']
display(df_merged.loc[worst_mask, ['experiment_id'] + solvent_cols])


In [ ]:
import re

ROMAN_TO_INT = {'0': 0, 'I': 1, 'II': 2, 'III': 3, 'IV': 4, 'V': 5, 'VI': 6}

def parse_oxidation_state(iupac_name: str) -> float:

    if not isinstance(iupac_name, str):
        return 0.0
    # Match trailing Roman numerals or '0' at end of name
    m = re.search(r'((?:VIII|VII|VI|IV|V|III|II|I|0))$', iupac_name.strip())
    if m:
        return float(ROMAN_TO_INT.get(m.group(1).upper(), 0))
    return 0.0  # default: assume 0 if unparseable

df_merged['precursor_ox_state'] = df_merged['precursor_iupac_standardized'].apply(parse_oxidation_state)
from mendeleev import element as mendeleev_element

from mendeleev import element as mendeleev_element


GROUP11_METALS = {'Cu', 'Ag', 'Au'}

def get_d_electron_count(metal_symbol: str, oxidation_state: float) -> float:
    """
    Returns the d-electron count for a transition metal in a given oxidation state.

    Formula:
      - Group 3-10:  d^n = group_number - oxidation_state
      - Group 11:    d^n = 10 - oxidation_state  (d10s1 neutral; s1 excluded)

    Args:
        metal_symbol:    Element symbol, e.g. 'Pd', 'Cu', 'Rh'
        oxidation_state: Numeric oxidation state, e.g. 0, 1, 2

    Returns:
        d-electron count as float, clamped to [0, 10].
    """
    try:
        el = mendeleev_element(metal_symbol)
        group = el.group_id          # e.g., Pd→10, Rh→9, Fe→8, Cu→11

        if metal_symbol in GROUP11_METALS:
            # d10s1 neutral: effective CBC d-count is 10 minus oxidation state
            d_count = 11 - int(oxidationstate) if oxidationstate >= 1 else 10
        else:
            # Standard CBC formula for groups 3-10 and 12
            d_count = float(group - int(oxidation_state))

        return float(max(0.0, min(10.0, d_count)))   # physically bounded [0,10]

    except Exception:
        return 0.0


df_merged['d_electron_count'] = df_merged.apply(
    lambda r: get_d_electron_count(r['metal_atom'], r['precursor_ox_state']),
    axis=1
)
# CBC lookup: precursor_iupac_standardized → (nL, nX, electron_donation_from_L)
# Electron donation = 2 per L ligand (2e donors)
PRECURSOR_CBC = {
    'tetrakis(triphenylphosphine)palladium(0)':                  (4, 0),
    'tetrakis(triphenylphosphine)platinum(0)':                   (4, 0),
    'Cu4I4(PPh3)4':                                              (1, 1),  # per Cu: 1L, 1X
    'Di-μ-chloro-tetracarbonyldirhodium(I)':                     (2, 1),  # per Rh: 2L(CO), 1X(Cl)
    'carbonylchlorobis(triphenylphosphine)rhodium(I)':            (3, 1),
    'carbonylchlorobis(triphenylphosphine)iridium(I)':            (3, 1),
    'carbonylbromobis(triphenylphosphine)iridium(I)':             (3, 1),
    'bromocarbonylbis(triphenylphosphine)iridium(I)':             (3, 1),
    'carbonylchlorodiiodobis(triphenylphosphine)iridium(I)':       (3, 3),
    'carbonylbis(triphenylphosphine)(maleimidato)iridium(I)':       (3, 1),
    'Chlorotris(triphenylphosphine)rhodium(I)':                   (3, 1),
    'Chlorotris(triphenylphosphine)cobalt(I)':                    (3, 1),
    'chloro(tristriphenylphosphine)cobalt(I)':                    (3, 1),
    'chloro(bistriphenylphosphine)gold(I)':                       (2, 1),
    'acetatochlorobis(triphenylphosphine)rhodium(I)':             (2, 2),
    'carbonylpyridinebis(triphenylphosphine)rhodium(I) tetrafluoroborate':       (4, 1),
    'Rh(py)(CO)(PPh3)2]BF4':                                     (4, 1),
    'Rh(MeCN)(CO)(PPh3)2]BF4':                                   (4, 1),
    'copper(I) iodide':                                           (0, 1),
    'copper(I) bromide':                                          (0, 1),
    'silver(I) iodide':                                           (0, 1),
    'silver(I) bromide':                                          (0, 1),
    'gold(I) iodide':                                             (0, 1),
    'silver(I) bromide cluster':                                  (0, 1),
    'silver(I) bromide cyclohexane':                              (1, 1),  # PCy3
    'CuBr cluster':                                               (1, 1),
    'silver(I) iodide cluster':                                   (1, 1),
    'Cu2I2(P2)3':                                                 (3, 1),  # 3 bidentate P2 = 6e donation
    'digold complex':                                             (2, 0),  # 2 alkynyl-phosphine L
}

def get_cbc(iupac_name):
    """Returns (nL, nX) for a precursor. Defaults to (0, 0) if unknown."""
    return PRECURSOR_CBC.get(str(iupac_name).strip(), (0, 0))

df_merged['precursor_nL'], df_merged['precursor_nX'] = zip(
    *df_merged['precursor_iupac_standardized'].apply(get_cbc)
)


In [ ]:
from rdkit import Chem
import numpy as np
import pandas as pd

# ── Single source of truth: underscores throughout ─────────────────────────
GEOMETRY_LABELS = ['linear', 'trigonal_planar', 'square_planar',
                   'tetrahedral', 'trigonal_bipyramidal', 'octahedral', 'unknown']


def get_precursor_geometry(smiles: str, metal_symbol: str, dcount: float):
    result = {f'precgeom_{g}': 0.0 for g in GEOMETRY_LABELS}  # ← uses GEOMETRY_LABELS
    coordn = 0.0

    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            result['precgeom_unknown'] = 1.0
            return result, coordn

        # ── Step 1: locate metal and check its degree ──────────────────────
        metal_degree = None
        for atom in mol.GetAtoms():
            if atom.GetSymbol() == metal_symbol:
                metal_degree = atom.GetDegree()
                break

        if metal_degree is None:
            result['precgeom_unknown'] = 1.0
            return result, coordn

        # ── Step 2: choose coordination number source ───────────────────────
        if metal_degree > 0:
            coordn = float(metal_degree)
        else:
            frags = Chem.GetMolFrags(mol, asMols=True, sanitizeFrags=False)
            n_ligand_frags = sum(
                1 for frag in frags
                if not any(a.GetSymbol() == metal_symbol for a in frag.GetAtoms())
            )
            coordn = float(n_ligand_frags)

        # ── Step 3: assign geometry ─────────────────────────────────────────
        # FIX 1: strings now use underscores to match GEOMETRY_LABELS keys
        # FIX 2: dcount == 8.0 only (not >= 8.0) — d10 Pd(0)/Ni(0) are tetrahedral
        cn = int(coordn)
        if cn == 2:
            geom = 'linear'
        elif cn == 3:
            geom = 'trigonal_planar'
        elif cn == 4:
            geom = 'square_planar' if dcount == 8.0 else 'tetrahedral'
        elif cn == 5:
            geom = 'trigonal_bipyramidal'
        elif cn == 6:
            geom = 'octahedral'
        else:
            geom = 'unknown'

        result[f'precgeom_{geom}'] = 1.0

    except Exception:
        result['precgeom_unknown'] = 1.0

    return result, coordn


# ── Geometry feature block ──────────────────────────────────────────────────
geom_rows = df_merged.apply(
    lambda r: get_precursor_geometry(
        r.get('smiles_precursor_canon', r.get('smiles_precursor', '')),
        r['metal_atom'],
        r['d_electron_count']
    )[0],
    axis=1
)
df_geom = pd.DataFrame(list(geom_rows))

# Guard: ensure all 7 columns exist in canonical order
for _col in [f'precgeom_{g}' for g in GEOMETRY_LABELS]:
    if _col not in df_geom.columns:
        df_geom[_col] = 0.0
df_geom = df_geom[[f'precgeom_{g}' for g in GEOMETRY_LABELS]]

X_prec_geom = df_geom.values.astype(float)                        # (756, 7)

# ── d-electron count + oxidation state ──────────────────────────────────────
X_d_electron = df_merged[['d_electron_count',
                           'precursor_ox_state']].values.astype(float)  # (756, 2)

# ── CBC ligand counts ────────────────────────────────────────────────────────
Xcbc = df_merged[['precursor_nL', 'precursor_nX']].fillna(0).values.astype(float)  # (756, 2)

# ── Covalent TEC ─────────────────────────────────────────────────────────────
_d_cov = (
    df_merged['d_electron_count'].fillna(0).values
    + df_merged['precursor_ox_state'].fillna(0).values
)
_tec_cov = (
    _d_cov
    + 2.0 * df_merged['precursor_nL'].fillna(0).values
    + 1.0 * df_merged['precursor_nX'].fillna(0).values
)

# CBC validity mask: 0 where lookup defaulted to (nL=0, nX=0) — TEC unreliable
_tec_known = (
    (df_merged['precursor_nL'].fillna(0) + df_merged['precursor_nX'].fillna(0)) > 0
).astype(float).values

Xtec = np.column_stack([
    _tec_cov,                                               # raw covalent TEC
    18.0 - _tec_cov,                                        # Δ from 18e (+ = electron-poor)
    16.0 - _tec_cov,                                        # Δ from 16e
    (_tec_cov == 18.0).astype(float),                       # exactly 18e?
    (_tec_cov == 16.0).astype(float),                       # exactly 16e?
    ((_tec_cov != 18.0) & (_tec_cov != 16.0)).astype(float),  # neither rule?
    _tec_known,                                             # CBC lookup valid?
])  # (756, 7)

# ── Assemble augmented metal block ───────────────────────────────────────────
X_metal_block = np.hstack([X_metal, X_metal_ohe])          # (756, 31)

print(f"\nFinal X_metal_block shape: {X_metal_block.shape}")
print(f"  = {N_METAL_DESCRIPTORS} continuous descriptors + {X_metal_ohe.shape[1]} OHE columns")

X_metal_block_aug = np.hstack([
    X_metal_block,   # 31: continuous metal descriptors + OHE
    X_d_electron,    #  2: d_electron_count, precursor_ox_state
    X_prec_geom,     #  7: geometry OHE
    Xcbc,            #  2: nL, nX
    Xtec,            #  7: tec, Δ18, Δ16, is_18e, is_16e, is_other, cbc_known
])  # → shape (756, 49)

# ── Feature names ─────────────────────────────────────────────────────────────
d_electron_names = ['metal_d_electron_count', 'metal_precursor_ox_state']
prec_geom_names  = [f'precgeom_{g}' for g in GEOMETRY_LABELS]
cbc_names        = ['precursor_nL', 'precursor_nX']
tec_names        = ['tec_covalent', 'tec_delta18', 'tec_delta16',
                    'tec_is_18e', 'tec_is_16e', 'tec_is_other', 'tec_cbc_known']

metal_names_aug = (metal_names + d_electron_names
                   + prec_geom_names + cbc_names + tec_names)
# 31 + 2 + 7 + 2 + 7 = 49

assert len(metal_names_aug) == X_metal_block_aug.shape[1], (
    f"Name/feature mismatch: {len(metal_names_aug)} names "
    f"vs {X_metal_block_aug.shape[1]} columns"
)
print(f"X_metal_block_aug shape: {X_metal_block_aug.shape}")
print(f"Feature names: {len(metal_names_aug)}")


In [ ]:
# ── CBC coverage diagnostic ───────────────────────────────────────────────────
_cbc_missing_mask = (
    df_merged['precursor_nL'].fillna(0) + df_merged['precursor_nX'].fillna(0)
) == 0

_n_missing = _cbc_missing_mask.sum()
_n_total   = len(df_merged)

print(f"\nCBC Coverage: {_n_total - _n_missing}/{_n_total} precursors have known nL/nX")
print(f"  ⚠  {_n_missing} precursors defaulted to (nL=0, nX=0) — TEC unreliable for these\n")

if _n_missing > 0:
    _missing_names = (
        df_merged.loc[_cbc_missing_mask, 'precursor_iupac_standardized']
        .unique()
    )
    print(f"  Unlisted precursors ({len(_missing_names)} unique):")
    for name in sorted(_missing_names):
        print(f"    - {name}")


In [ ]:
import numpy as np
from rdkit.Chem import AllChem

def generate_morgan_fp(smiles, n_bits=2048):
    """
    Generates a Morgan fingerprint (radius=2) for a given SMILES string.
    Returns a numpy array of bits. Returns a zero vector if SMILES is invalid.
    """
    # Handle missing or non-string input
    if pd.isna(smiles) or not isinstance(smiles, str) or smiles.strip() == "":
        return np.zeros(n_bits, dtype=int)

    # Attempt to create molecule object
    mol = Chem.MolFromSmiles(smiles)

    # Handle failed molecule creation
    if mol is None:
        return np.zeros(n_bits, dtype=int)

    # Generate Morgan fingerprint
    try:
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=n_bits)
        # Convert to numpy array
        return np.array(fp)
    except:
        return np.zeros(n_bits, dtype=int)

In [ ]:
import pandas as pd
import numpy as np
from rdkit import Chem, rdBase, DataStructs
from rdkit.Chem import AllChem

# Optional: silence RDKit parsing noise
rdBase.DisableLog("rdApp.error")
rdBase.DisableLog("rdApp.warning")

# ----------------------------
# 1) Helpers: safe SMILES handling
# ----------------------------
def canonicalize_smiles_keep(smiles):
    """
    Return canonical SMILES if RDKit can parse; otherwise return the original string (NOT None).
    """
    if pd.isna(smiles) or not isinstance(smiles, str) or smiles.strip() == "":
        return None
    s = smiles.strip()
    try:
        m = Chem.MolFromSmiles(s)
        if m is None:
            return s  # keep original
        return Chem.MolToSmiles(m, canonical=True)
    except Exception:
        return s  # keep original

ION_MAP = {
    "Cl-": "[Cl-]", "Br-": "[Br-]", "I-": "[I-]", "F-": "[F-]",
    "Cl":  "[Cl-]", "Br":  "[Br-]", "I":  "[I-]", "F":  "[F-]",
}

def normalize_inventory_token(token):
    """
    Inventory column suffix -> either a SMILES (string) or None (meaning: don't fingerprint it).
    """
    if token is None:
        return None
    t = str(token).strip()
    if t in ION_MAP:
        return ION_MAP[t]
    # crude filter: if it contains spaces or clearly isn't SMILES-like, skip fingerprinting
    if " " in t or len(t) == 0:
        return None
    return t

# ----------------------------
# 2) Safe Morgan FP extraction (NumPy)
# ----------------------------
def morgan_fp_numpy(smiles, n_bits=2048, radius=2):
    """
    Returns (fp_array, ok_bool). If parse fails, returns (zeros, False).
    """
    arr = np.zeros((n_bits,), dtype=np.int8)
    if smiles is None or (not isinstance(smiles, str)) or smiles.strip() == "":
        return arr, False
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return arr, False
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=radius, nBits=n_bits)
        DataStructs.ConvertToNumpyArray(fp, arr)
        return arr, True
    except Exception:
        return arr, False

# Keep raw columns and create canonical columns (do NOT overwrite raw)
for col in ["smiles_linker_1", "smiles_linker_2", "smiles_modulator", "smiles_precursor"]:
    raw = f"{col}_raw"
    can = f"{col}_canon"
    if raw not in df_merged.columns:
        df_merged[raw] = df_merged[col]
    df_merged[can] = df_merged[col].apply(canonicalize_smiles_keep)

# Choose which SMILES to FEATURIZE: prefer canonical if it is parseable, else fall back to raw
def pick_featurizable_smiles(row, base_col):
    can = row.get(f"{base_col}_canon", None)
    raw = row.get(f"{base_col}_raw", None)
    # Try canonical first
    fp_can, ok_can = morgan_fp_numpy(can)
    if ok_can:
        return can
    # Fall back to raw
    fp_raw, ok_raw = morgan_fp_numpy(raw)
    return raw if ok_raw else None

df_merged["smiles_linker_1_feat"] = df_merged.apply(lambda r: pick_featurizable_smiles(r, "smiles_linker_1"), axis=1)
df_merged["smiles_modulator_feat"] = df_merged.apply(lambda r: pick_featurizable_smiles(r, "smiles_modulator"), axis=1)

# ----------------------------
# 4) Fixed-component fingerprints
# ----------------------------
X_linker = np.stack([morgan_fp_numpy(s)[0] for s in df_merged["smiles_linker_1_feat"].tolist()])
X_modulator = np.stack([morgan_fp_numpy(s)[0] for s in df_merged["smiles_modulator_feat"].tolist()])

# ── Concentration-scaled fingerprints ─────────────────────────────────────
linker_conc_arr   = df_merged['linker_conc'].fillna(0).to_numpy().reshape(-1, 1)
mod_conc_arr      = df_merged['mod_conc'].fillna(0).to_numpy().reshape(-1, 1)

X_linker_scaled    = X_linker   * linker_conc_arr   # (n, 2048)
X_modulator_scaled = X_modulator * mod_conc_arr      # (n, 2048)

# Also keep modulator equivalents as a scalar feature (prevents “unweighted vs weighted” duplication in bit-space)
if "equivalents" in df_merged.columns:
    mod_eq = pd.to_numeric(df_merged["equivalents"], errors="coerce").fillna(0.0).to_numpy().reshape(-1, 1)
else:
    mod_eq = np.zeros((len(df_merged), 1), dtype=float)

# ============================================================
# 5  Per-Ligand Fingerprints for Precursor Ligands (FIXED)
#    Also retains a "numeric inventory" block for non-SMILES
#    tokens (ions, fragments that fail parsing).
# ============================================================

inventory_cols = [c for c in df_merged.columns if c.startswith('Total_')]
df_merged[inventory_cols] = df_merged[inventory_cols].fillna(0)


precursor_ligand_cols = inventory_cols  # no exclusion needed

# Optional: track which columns overlap with modulator for SHAP annotation
mod_smiles_vals = df_merged['smiles_modulator_feat'].fillna('').astype(str)
_mod_overlap_cols = {
    f'Total_{s}' for s in mod_smiles_vals.unique()
    if s and s.lower() not in ('', 'none', 'nan')
    and f'Total_{s}' in set(inventory_cols)
}


# ----------------------------------------------------------------
# 5a  Decide per column: fingerprint-able vs numeric-only
# ----------------------------------------------------------------
FP_NBITS  = 2048
fp_cols   = []   # cols whose token parses as SMILES → get FP block
num_cols  = []   # cols whose token is an ion / unparseable → count only

fp_cache  = {}   # smiles → np.array(FP_NBITS,)

for col in precursor_ligand_cols:
    token     = col.replace('Total_', '')
    smi_token = normalize_inventory_token(token)      # your existing helper

    if smi_token is None:
        num_cols.append(col)
        continue

    # FIX: Correct keyword argument from nbits to n_bits
    fp_vec, ok = morgan_fp_numpy(smi_token, n_bits=FP_NBITS)
    if not ok:
        num_cols.append(col)   # can't fingerprint → keep as count
        continue

    fp_cache[col] = fp_vec     # cache the vector
    fp_cols.append(col)

print(f"  Per-ligand FP blocks : {len(fp_cols)} ligands × (2048 fp + 1 count) = "
      f"{len(fp_cols) * (FP_NBITS + 1)} features")
print(f"  Numeric-only counts  : {len(num_cols)} tokens")

# ----------------------------------------------------------------
# 5b  Build fixed-width per-ligand FP matrix
#     Shape: (n_samples, n_fp_ligands × (FP_NBITS + 1))
#     Column order: [FP_lig1 (2048) | count_lig1 | FP_lig2 | count_lig2 | ...]
# ----------------------------------------------------------------
n = len(df_merged)
n_fp_lig = len(fp_cols)

if n_fp_lig > 0:
    # Pre-stack all FP vectors → shape (n_fp_lig, FP_NBITS)
    fp_matrix = np.vstack([fp_cache[col] for col in fp_cols])   # (L, 2048)

    # Count matrix → shape (n_samples, n_fp_lig)
    count_matrix = np.column_stack([
        pd.to_numeric(df_merged[col], errors='coerce').fillna(0.0).to_numpy()
        for col in fp_cols
    ])  # (n, L)

    # For each sample build [FP_i * (count_i > 0), count_i] interleaved
    # Shape: (n, L * (FP_NBITS + 1))
    # Strategy: broadcast count as a binary presence mask on FP, keep count scalar
    # This gives the model both "what ligand" and "how many"
    fp_presence = (count_matrix > 0).astype(float)          # (n, L)  1 if present
    fp_expanded = fp_presence[:, :, np.newaxis] * fp_matrix[np.newaxis, :, :]
    # fp_expanded shape: (n, L, FP_NBITS)

    # Interleave: for each ligand → [fp_bits..., count]
    count_expanded = count_matrix[:, :, np.newaxis]          # (n, L, 1)
    per_ligand_block = np.concatenate(
        [fp_expanded, count_expanded], axis=2
    )  # (n, L, FP_NBITS + 1)

    # Flatten last two dims → (n, L * (FP_NBITS + 1))
    X_precursor_perlig = per_ligand_block.reshape(n, n_fp_lig * (FP_NBITS + 1))
else:
    X_precursor_perlig = np.zeros((n, 0), dtype=float)

# ----------------------------------------------------------------
# 5c  Numeric-only block (ions, unparse-able fragments)
# ----------------------------------------------------------------
if len(num_cols) > 0:
    Xinventorynumeric = np.column_stack([
        pd.to_numeric(df_merged[col], errors='coerce').fillna(0.0).to_numpy()
        for col in num_cols
    ])
else:
    Xinventorynumeric = np.zeros((n, 0), dtype=float)

# ----------------------------------------------------------------
# 5d  Sanity checks
# ----------------------------------------------------------------
print(f"\n  X_precursor_perlig   shape: {X_precursor_perlig.shape}")
print(f"  Xinventorynumeric    shape: {Xinventorynumeric.shape}")
print(f"  All-zero perlig rows: "
      f"{int((X_precursor_perlig.sum(axis=1) == 0).sum())} / {n}")

bad = ~np.isfinite(X_precursor_perlig)
if bad.any():
    print(f"  WARNING: {bad.sum()} non-finite values → replacing with 0")
    X_precursor_perlig[bad] = 0.0
else:
    print("  ✓ X_precursor_perlig all finite")

# ----------------------------------------------------------------
# 5e  Feature name tracking (optional but recommended for SHAP)
# ----------------------------------------------------------------
perlig_feature_names = []
for col in fp_cols:
    token = col.replace('Total_', '')
    perlig_feature_names += [f"preclig_{token}_fp{i}" for i in range(FP_NBITS)]
    perlig_feature_names += [f"preclig_{token}_count"]

numeric_feature_names = [f"inv_num_{col.replace('Total_','')}" for col in num_cols]

print(f"\n  Total per-ligand feature names tracked: {len(perlig_feature_names)}")


# ----------------------------
# 6) Add experimental/process variables (use whatever exists)
# ----------------------------
process_cols = [
    'equivalents',
    'total_solvent_volume_ml',
    'solvent_1_fraction',
    'solvent_2_fraction',
    'solvent_3_fraction',
    'Min_Boiling_Point_K',
    'Max_Boiling_Point_K',
    'Weighted_Boiling_Point_K',
    'Weighted_AN_mole',
    'Weighted_DN_mole',
    'Weighted_Dielectric_vol',
    'Weighted_Polarity_vol',
    'Weighted_sig_h_vol',
    'Weighted_sig_d_vol',
    'Weighted_sig_p_vol',
    'Mix_M0_Area',
    'Mix_M2_Polarity',
    'Mix_M3_Asymmetry',
    'Mix_M_HB_Acc',
    'Mix_M_HB_Don',
    'temperature_k',
    'metal_over_linker_ratio',
    'reaction_hours'
]
process_cols_present = [c for c in process_cols if c in df_merged.columns]
X_process = df_merged[process_cols_present].apply(pd.to_numeric, errors="coerce").fillna(0.0).to_numpy()

# ----------------------------
# 7 Assemble final matrix
# ----------------------------
X_features = np.concatenate(
    [X_linker, X_linker_scaled,
     X_modulator, X_modulator_scaled,
     mod_eq,
     X_precursor_perlig,
     Xinventorynumeric,
     X_process],
    axis=1
)

print("X_features shape:", X_features.shape)
print("Blocks:")
print("  - Linker FP              :", X_linker.shape)
print("  - Modulator FP           :", X_modulator.shape)
print("  - Mod equiv              :", mod_eq.shape)
print("  - Precursor per-lig FP   :", X_precursor_perlig.shape,
      f"  ← {len(fp_cols)} ligands × {FP_NBITS+1} (fp+count)")
print("  - Inventory numeric      :", Xinventorynumeric.shape)
print("  - Process vars           :", X_process.shape)
print(f"  - OLD summed FP was      : (761, 2048) ← now gone")

# All-zero sanity checks
print("All-zero linker rows     :", int((X_linker.sum(axis=1) == 0).sum()), "/", n)
print("All-zero modulator rows  :", int((X_modulator.sum(axis=1) == 0).sum()), "/", n)
print("All-zero per-lig rows    :", int((X_precursor_perlig.sum(axis=1) == 0).sum()), "/", n)


display(df_merged.head())

In [ ]:
import numpy as np
import pandas as pd
from rdkit import Chem

if not hasattr(np, 'float'):
    np.float = float
if not hasattr(np, 'product'):
    np.product = np.prod
if not hasattr(np, 'int'):
    np.int = int
if not hasattr(np, 'complex'):
    np.complex = complex
if not hasattr(np, 'bool'):
    np.bool = bool
if not hasattr(np, 'object'):
    np.object = object
if not hasattr(np, 'str'):
    np.str = str

from mordred import Calculator, descriptors

# Patch numpy for mordred compatibility (mordred 1.2.0 uses deprecated np types)
if not hasattr(np, "float"):
    np.float = float
if not hasattr(np, "product"):
    np.product = np.prod

# Initialize Mordred calculator with Autocorrelation descriptors (ATS, MATS, GATS)
calc = Calculator(descriptors.Autocorrelation)
num_descriptors = len(calc.descriptors)
print(f"Initialized Mordred calculator with {num_descriptors} Autocorrelation descriptors.")

# --- Helpers ---
ION_MAP = {
    "Cl-": "[Cl-]", "Br-": "[Br-]", "I-": "[I-]", "F-": "[F-]",
    "Cl":  "[Cl-]", "Br":  "[Br-]", "I":  "[I-]", "F":  "[F-]",
}

def _normalize_inventory_token(token: str):
    """Map obvious ions and return a SMILES-like string (or None if not usable)."""
    if token is None:
        return None
    t = str(token).strip()
    if t == "" or " " in t:
        return None
    return ION_MAP.get(t, t)

def _mordred_result_to_vec_and_stats(results):
    """
    Returns:
      vec_clean: shape (num_descriptors,), non-finite -> 0.0
      any_bad:   1.0 if any non-finite/failed entry occurred else 0.0
      frac_bad:  fraction of entries that were non-finite/failed
    """
    vec = np.zeros(num_descriptors, dtype=float)
    n_bad = 0

    for i, r in enumerate(results):
        try:
            v = float(r)  # float(np.nan) succeeds, so we must check isfinite
        except (ValueError, TypeError):
            v = np.nan

        if not np.isfinite(v):
            n_bad += 1
            v = 0.0

        vec[i] = v

    any_bad = 1.0 if n_bad > 0 else 0.0
    frac_bad = n_bad / float(num_descriptors) if num_descriptors else 0.0
    return vec, any_bad, frac_bad

def get_mordred_racs_smiles_with_stats(smiles):
    """
    Compute Autocorrelation descriptors for a single SMILES.
    Returns (vec_clean, any_bad, frac_bad).
    """
    if pd.isna(smiles) or not isinstance(smiles, str) or smiles.strip() == "":
        return np.zeros(num_descriptors, dtype=float), 1.0, 1.0

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros(num_descriptors, dtype=float), 1.0, 1.0

    try:
        results = calc(mol)
        return _mordred_result_to_vec_and_stats(results)
    except Exception:
        return np.zeros(num_descriptors, dtype=float), 1.0, 1.0

def _pick_first_existing(cols):
    for c in cols:
        if c in df_merged.columns:
            return c
    return None

mod_smiles_col = _pick_first_existing([
    "smiles_modulator_feat", "smiles_modulator_canon", "smiles_modulator"
])
if mod_smiles_col is None:
    raise KeyError("Could not find a modulator SMILES column in df_merged.")

# --- 1) Modulator RACs (one per row) + 2 missingness features ---
print("Calculating RAC descriptors for Modulators...")
mod_out = df_merged[mod_smiles_col].apply(get_mordred_racs_smiles_with_stats).values
X_modulator_rac = np.stack([t[0] for t in mod_out])
mod_rac_any_bad = np.array([t[1] for t in mod_out], dtype=float).reshape(-1, 1)
mod_rac_frac_bad = np.array([t[2] for t in mod_out], dtype=float).reshape(-1, 1)

X_modulator_rac_aug = np.concatenate([X_modulator_rac, mod_rac_any_bad, mod_rac_frac_bad], axis=1)

# --- 2) Precursor-ligand RACs: weighted sum + 2 aggregate missingness features ---
inventory_cols = [c for c in df_merged.columns if c.startswith("Total_")]
df_merged[inventory_cols] = df_merged[inventory_cols].fillna(0.0)

mod_smiles_unique = (
    df_merged[mod_smiles_col].fillna("").astype(str).map(str.strip).unique()
)
mod_total_cols = set(f"Total_{s}" for s in mod_smiles_unique if s and s.lower() != "none")
precursor_ligand_cols = [c for c in inventory_cols if c not in mod_total_cols]

print(f"Inventory columns found: {len(inventory_cols)}")
print(f"Using precursor-ligand columns (excl. modulator totals): {len(precursor_ligand_cols)}")

# Cache per-token vec + stats (computed once)
token_to_vec = {}
token_to_any = {}
token_to_frac = {}

# Build row-wise weighted sum and missingness aggregates
n = len(df_merged)
X_precursor_lig_rac = np.zeros((n, num_descriptors), dtype=float)

prec_rac_missing_any_wsum = np.zeros((n, 1), dtype=float)   # weighted sum of "bad-token" flags
prec_rac_missing_frac_wsum = np.zeros((n, 1), dtype=float)  # weighted sum of frac_bad
prec_total_counts = np.zeros((n, 1), dtype=float)           # sum of counts used in RAC aggregation

print("Calculating RAC descriptors for Precursor ligands (weighted sum)...")
for col in precursor_ligand_cols:
    token = col.replace("Total_", "")
    smi = _normalize_inventory_token(token)
    if smi is None:
        continue

    counts = pd.to_numeric(df_merged[col], errors="coerce").fillna(0.0).to_numpy().reshape(-1, 1)
    if np.allclose(counts, 0.0):
        continue

    if smi not in token_to_vec:
        v, a, f = get_mordred_racs_smiles_with_stats(smi)
        token_to_vec[smi] = v
        token_to_any[smi] = a
        token_to_frac[smi] = f

    vec = token_to_vec[smi]
    X_precursor_lig_rac += counts * vec.reshape(1, -1)

    prec_total_counts += counts
    prec_rac_missing_any_wsum += counts * token_to_any[smi]
    prec_rac_missing_frac_wsum += counts * token_to_frac[smi]

# Convert weighted sums to something scale-stable:
# - any_wsum is still a useful "how much missingness mass" signal
# - frac becomes weighted average frac_bad across present ligands
prec_rac_missing_frac_wavg = prec_rac_missing_frac_wsum / np.clip(prec_total_counts, 1.0, None)

X_precursor_lig_rac_aug = np.concatenate(
    [X_precursor_lig_rac, prec_rac_missing_any_wsum, prec_rac_missing_frac_wavg],
    axis=1
)

# --- 3) Concatenate with existing X_features ---
X_final = np.concatenate([
    X_features,
    X_modulator_rac_aug,
    X_metal_block
], axis=1)

print(f"Updated X_final shape: {X_final.shape}")
print(f"  Added {X_metal_block.shape[1]} metal center features")

# --- 4) Validation ---
print("\nFeature Matrix Assembly Complete.")
print("X_final shape:", X_final.shape)
print("Feature Composition Breakdown:")
print(f"  - Base features (X_features): {X_features.shape[1]}") # Changed from X_features to X_features
print(f"  - Precursor ligand RACs +2 indicators: {X_precursor_lig_rac_aug.shape[1]}")
print(f"  - Modulator RACs +2 indicators: {X_modulator_rac_aug.shape[1]}")
print(f"  - Total: {X_final.shape[1]}")

# Hard check: no NaN/inf
bad = ~np.isfinite(X_final)
print("Non-finite entries in X_final:", int(bad.sum()))


In [ ]:
# ================================================================
# METAL PRECURSOR FEATURIZATION BLOCK
# Strategy: decompose precursor into 3 independent sub-blocks
#   A. Metal center atomic properties (mendeleev)
#   B. Co-ligand inventory (lookup + FP, no metal atoms)
#   C. Complex-level geometry and electronic descriptors
# ================================================================
import re
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import rdmolops
import mendeleev

# -----------------------------------------------------------------
# LOOKUP TABLES
# -----------------------------------------------------------------

# Mendeleev atomic properties per metal symbol
_METAL_PROPS_CACHE = {}

def _get_mendeleev_props(symbol: str) -> dict:
    if symbol in _METAL_PROPS_CACHE:
        return _METAL_PROPS_CACHE[symbol]
    try:
        el = mendeleev.element(symbol)
        def safe(v, default=0.0):
            try:
                f = float(v)
                return f if np.isfinite(f) else default
            except Exception:
                return default

        props = {
            'atomic_number':   safe(el.atomic_number),
            'period':          safe(el.period),
            'group':           safe(el.group_id, 0.0),
            'en_pauling':      safe(el.en_pauling),
            'en_allen':        safe(el.en_allen),
            'atomic_radius':   safe(el.atomic_radius),
            'cov_radius':      safe(el.covalent_radius_pyykko),
            'vdw_radius':      safe(el.vdw_radius),
            'ie1':             safe(el.ionization_energies[0]
                                    if el.ionization_energies else 0),
            'ie2':             safe(el.ionization_energies[1]
                                    if len(el.ionization_energies) > 1 else 0),
            'electron_affinity': safe(el.electron_affinity),
            'dipole_polarizability': safe(el.dipole_polarizability),
            # d/f electron counts
            'd_electrons':     safe(el.d_electrons),
            'f_electrons':     safe(el.f_electrons),
            'valence_electrons': safe(el.valence_electrons),
            # block: s=0, p=1, d=2, f=3
            'block_enc':       {'s': 0.0, 'p': 1.0,
                                'd': 2.0, 'f': 3.0}.get(el.block, 2.0),
        }
    except Exception:
        props = {k: 0.0 for k in [
            'atomic_number','period','group','en_pauling','en_allen',
            'atomic_radius','cov_radius','vdw_radius','ie1','ie2',
            'electron_affinity','dipole_polarizability',
            'd_electrons','f_electrons','valence_electrons','block_enc']}
    _METAL_PROPS_CACHE[symbol] = props
    return props

_MENDELEEV_PROP_KEYS = [
    'atomic_number','period','group','en_pauling','en_allen',
    'atomic_radius','cov_radius','vdw_radius','ie1','ie2',
    'electron_affinity','dipole_polarizability',
    'd_electrons','f_electrons','valence_electrons','block_enc'
]

# Co-ligand property lookup — avoids FP/Mordred on simple ligands
# Values: [TEP_proxy, sigma_donor, pi_acceptor, charge, is_halide, is_carbonyl]
# TEP_proxy: approximate Tolman CO stretch analog (cm⁻¹ normalized to 0–1)
_COLIGAND_LOOKUP = {
    # CO: strong π-acceptor, weak trans influence relative to H,
    #     but highest field strength ligand known
    'CO':    [0.99,  0.90, 1.00,  0.0,  0.0, 1.0],
    '[CO]':  [0.99,  0.90, 1.00,  0.0,  0.0, 1.0],

    # Halides: σ-donors, weak π-donors, negative EL
    # Trans influence order: I > Br > Cl > F (Appleton 1973)
    '[Cl-]': [-0.24, 0.35, 0.47, -1.0,  1.0, 0.0],
    'Cl':    [-0.24, 0.35, 0.47, -1.0,  1.0, 0.0],
    '[Br-]': [-0.22, 0.45, 0.44, -1.0,  1.0, 0.0],
    'Br':    [-0.22, 0.45, 0.44, -1.0,  1.0, 0.0],
    '[I-]':  [-0.24, 0.55, 0.40, -1.0,  1.0, 0.0],
    'I':     [-0.24, 0.55, 0.40, -1.0,  1.0, 0.0],
    '[F-]':  [-0.40, 0.15, 0.52, -1.0,  1.0, 0.0],
    'F':     [-0.40, 0.15, 0.52, -1.0,  1.0, 0.0],
}

# Feature names matching the 6-vector above — use these in SHAP
_COLIGAND_FEATURE_NAMES = [
    'EL_lever_V',          # Lever electrochemical parameter (Lever 1990)
    'trans_influence',     # Appleton 1973 normalized scale
    'field_strength_norm', # Spectrochemical series, CO=1.0
    'formal_charge',       # IUPAC, unambiguous
    'is_halide',           # binary
    'is_carbonyl',         # binary
]
_COLIGAND_DIM = 6


# Known oxidation states for common precursor metals in this dataset
# Parsed as fallback when IUPAC name regex fails
_METAL_OX_STATE_FALLBACK = {
    'Pd': 0, 'Pt': 0,           # M(0) phosphine complexes
    'Rh': 1, 'Ir': 1,           # M(I) carbonyl/phosphine complexes
    'Ag': 1, 'Au': 1, 'Cu': 1,  # M(I) halide complexes
    'Co': 2, 'Ni': 2,
}

def _parse_oxidation_state(iupac_name: str, metal_symbol: str) -> float:
    """Extract Roman numeral oxidation state from IUPAC name."""
    if not isinstance(iupac_name, str):
        return float(_METAL_OX_STATE_FALLBACK.get(metal_symbol, 0))
    roman = {'I': 1, 'II': 2, 'III': 3, 'IV': 4,
             'V': 5, 'VI': 6, '0': 0}
    # Match pattern like "rhodium(I)" or "palladium(0)"
    match = re.search(r'\(([IVX]+|0)\)', iupac_name, re.IGNORECASE)
    if match:
        return float(roman.get(match.group(1).upper(),
                    _METAL_OX_STATE_FALLBACK.get(metal_symbol, 0)))
    return float(_METAL_OX_STATE_FALLBACK.get(metal_symbol, 0))


# -----------------------------------------------------------------
# BLOCK A: Metal center features — 31 dims
# [16 mendeleev props] + [oxidation_state, d_electron_count_in_complex,
#  is_d8, is_d10, is_d6, coord_num_est, is_square_planar, is_tetrahedral,
#  is_octahedral, is_period3, is_period4, is_period5,
#  group_9, group_10, group_11, missing_flag]
# -----------------------------------------------------------------
METAL_BLOCK_DIM = 32

def get_metal_center_block(smiles: str,
                           iupac_name: str = '') -> np.ndarray:
    out = np.zeros(METAL_BLOCK_DIM, dtype=float)
    out[31] = 1.0  # assume missing

    TARGET_METALS = {'Pd','Rh','Pt','Ag','Ir','Au','Cu',
                     'Co','Ni','Fe','Ru','Os','Re','W','Mo'}
    if not isinstance(smiles, str) or not smiles.strip():
        return out

    mol = Chem.MolFromSmiles(smiles, sanitize=False)
    if mol is None:
        return out

    metal_atoms = [a for a in mol.GetAtoms()
                   if a.GetSymbol() in TARGET_METALS]
    if not metal_atoms:
        return out

    # Use the first unique metal found (these are all mononuclear or
    # homo-metallic precursors)
    metal_sym = metal_atoms[0].GetSymbol()
    props = _get_mendeleev_props(metal_sym)

    # Fill mendeleev properties (slots 0–15)
    for i, key in enumerate(_MENDELEEV_PROP_KEYS):
        out[i] = props[key]

    # Oxidation state (slot 16)
    ox = _parse_oxidation_state(iupac_name, metal_sym)
    out[16] = ox

    # d-electron count in complex: d^n = group - oxidation_state
    # e.g. Rh(I) is group 9, d8; Pd(0) is group 10, d10
    group = props['group']
    d_count = group - ox  # approximate, works for groups 8–12
    d_count = max(0.0, min(10.0, d_count))
    out[17] = d_count

    # d-count flags
    out[18] = 1.0 if abs(d_count - 8)  < 0.5 else 0.0  # d8
    out[19] = 1.0 if abs(d_count - 10) < 0.5 else 0.0  # d10
    out[20] = 1.0 if abs(d_count - 6)  < 0.5 else 0.0  # d6

    # Estimated coordination number from fragment count
    # (number of non-metal heavy-atom neighbors across all metal atoms)
    total_bonds = sum(a.GetDegree() for a in metal_atoms)
    coord_num = total_bonds / max(len(metal_atoms), 1)
    out[21] = coord_num

    # Geometry flags from d-count + coord num heuristic
    is_sq = (abs(d_count - 8) < 0.5 and coord_num >= 3.5)
    is_tet = (abs(d_count - 10) < 0.5 and coord_num < 5)
    is_oct = (coord_num >= 5.5)
    out[22] = 1.0 if is_sq  else 0.0  # square planar
    out[23] = 1.0 if is_tet else 0.0  # tetrahedral
    out[24] = 1.0 if is_oct else 0.0  # octahedral

    # Period flags (3d/4d/5d)
    period = int(props['period'])
    out[25] = 1.0 if period == 4 else 0.0  # 3d metals (period 4)
    out[26] = 1.0 if period == 5 else 0.0  # 4d metals (period 5)
    out[27] = 1.0 if period == 6 else 0.0  # 5d metals (period 6)

    # Group flags (relevant for these precursors)
    out[28] = 1.0 if abs(group - 9)  < 0.5 else 0.0   # group 9 (Rh, Ir, Co)
    out[29] = 1.0 if abs(group - 10) < 0.5 else 0.0   # group 10 (Pd, Pt, Ni)
    out[30] = 1.0 if abs(group - 11) < 0.5 else 0.0   # group 11 (Cu, Ag, Au)

    out[31] = 0.0  # success
    bad = ~np.isfinite(out)
    out[bad] = 0.0
    return out


# -----------------------------------------------------------------
# BLOCK B: Co-ligand inventory features — 24 dims
# Encodes the 6 property vector for each of the 4 most common
# simple co-ligand types: [CO, Cl/Br/I-halide, PPh3-type, other]
# Each slot = weighted count × property vector
# Plus: [n_CO, n_halide, n_phosphine, n_other] scalar counts
# -----------------------------------------------------------------
COLIGAND_BLOCK_DIM = 28

def get_coligand_block(smiles: str) -> np.ndarray:
    """
    Characterizes co-ligands attached to the metal in the precursor.
    Separates simple ligands (CO, halide) using lookup and
    complex ligands (phosphines) using known structural flags.
    """
    out = np.zeros(COLIGAND_BLOCK_DIM, dtype=float)
    if not isinstance(smiles, str) or not smiles.strip():
        return out

    # Split on '.' to get individual fragments (SMILES disconnected)
    frags = smiles.split('.')
    TARGET_METALS = {'Pd','Rh','Pt','Ag','Ir','Au','Cu',
                     'Co','Ni','Fe','Ru','Os'}

    co_vec      = np.zeros(_COLIGAND_DIM)
    halide_vec  = np.zeros(_COLIGAND_DIM)
    phos_count  = 0
    other_count = 0

    for frag in frags:
        frag = frag.strip()
        if not frag:
            continue
        # Skip fragments that contain a metal atom
        mol_frag = Chem.MolFromSmiles(frag, sanitize=False)
        if mol_frag and any(a.GetSymbol() in TARGET_METALS
                            for a in mol_frag.GetAtoms()):
            continue

        # Simple lookup
        if frag in _COLIGAND_LOOKUP:
            props = np.array(_COLIGAND_LOOKUP[frag])
            if props[5] > 0:   # is_carbonyl
                co_vec += props
            else:              # is_halide
                halide_vec += props
            continue

        # Phosphine detection: must contain P with aryl substituents
        if 'P' in frag or 'p' in frag:
            if mol_frag is not None:
                has_P = any(a.GetSymbol() == 'P'
                            for a in mol_frag.GetAtoms())
                if has_P:
                    phos_count += 1
                    continue

        other_count += 1

    # Pack: 6 CO props, 6 halide props, phos_count, other_count
    out[0:6]   = co_vec
    out[6:12]  = halide_vec
    out[12]    = float(phos_count)
    out[13]    = float(other_count)

    # Scalar summary counts
    out[14] = float(co_vec[5])       # n_CO (summed carbonyl flag)
    out[15] = float(halide_vec[4])   # n_halide (summed halide flag)
    out[16] = float(phos_count)
    out[17] = float(other_count)

    # Electronic balance: σ-donor sum - π-acceptor sum
    # High = electron-rich metal center, low = electron-poor
    sigma_sum = co_vec[1] + halide_vec[1]
    pi_sum    = co_vec[2] + halide_vec[2]
    out[18]   = sigma_sum
    out[19]   = pi_sum
    out[20]   = sigma_sum - pi_sum  # net donor character

    # Net charge of co-ligands (from lookup charges)
    out[21] = co_vec[3] + halide_vec[3]

    # Flags
    out[22] = 1.0 if co_vec[5] > 0 else 0.0    # has_CO
    out[23] = 1.0 if halide_vec[4] > 0 else 0.0 # has_halide
    out[24] = 1.0 if 'Cl' in smiles else 0.0
    out[25] = 1.0 if 'Br' in smiles else 0.0
    out[26] = 1.0 if 'I'  in smiles and 'Ir' not in smiles else 0.0
    out[27] = 1.0 if phos_count > 0 else 0.0

    bad = ~np.isfinite(out)
    out[bad] = 0.0
    return out


# -----------------------------------------------------------------
# BLOCK C: Complex-level descriptors — 12 dims
# [is_dimer, n_metal_atoms, total_coord_bonds, is_homoleptic,
#  precursor_charge_est, n_unique_ligand_types,
#  has_bridging_halide, has_M_M_bond,
#  is_labile (d10 + no CO), is_robust (d8 + CO),
#  mol_weight_est, missing_flag]
# -----------------------------------------------------------------
COMPLEX_BLOCK_DIM = 12

def get_complex_level_block(smiles: str) -> np.ndarray:
    out = np.zeros(COMPLEX_BLOCK_DIM, dtype=float)
    out[11] = 1.0
    if not isinstance(smiles, str) or not smiles.strip():
        return out

    mol = Chem.MolFromSmiles(smiles, sanitize=False)
    if mol is None:
        return out

    TARGET_METALS = {'Pd','Rh','Pt','Ag','Ir','Au','Cu',
                     'Co','Ni','Fe','Ru','Os'}
    metal_atoms = [a for a in mol.GetAtoms()
                   if a.GetSymbol() in TARGET_METALS]
    n_metals = len(metal_atoms)
    if n_metals == 0:
        return out

    out[0] = 1.0 if n_metals > 1 else 0.0   # is_dimer/cluster
    out[1] = float(n_metals)

    # Total bonds from metal atoms
    total_bonds = sum(a.GetDegree() for a in metal_atoms)
    out[2] = float(total_bonds)

    # Homoleptic: all non-metal fragments are the same SMILES
    frags = smiles.split('.')
    lig_frags = [f for f in frags if not any(
        m in f for m in TARGET_METALS)]
    out[3] = 1.0 if len(set(lig_frags)) <= 1 else 0.0

    # Net charge estimate (count [X-] fragments)
    neg_count = sum(1 for f in frags if f.strip().startswith('[')
                    and f.strip().endswith('-]'))
    out[4] = float(-neg_count)

    # Unique ligand types
    out[5] = float(len(set(f.strip() for f in lig_frags if f.strip())))

    # Bridging halide (present in dimers like [Rh(CO)2Cl]2)
    has_bridge_X = (n_metals > 1 and
                    any(s in smiles for s in ['Cl', 'Br', 'I', 'F'])
                    and 'CO' in smiles)
    out[6] = 1.0 if has_bridge_X else 0.0

    # M-M bond flag (check for direct metal-metal bond in graph)
    has_mm = any(
        n.GetSymbol() in TARGET_METALS
        for a in metal_atoms
        for n in a.GetNeighbors()
        if n.GetIdx() != a.GetIdx()
    )
    out[7] = 1.0 if has_mm else 0.0

    # Lability proxy: d10 metals with no CO are labile (easily displace PPh3)
    d_count = (mendeleev.element(metal_atoms[0].GetSymbol()).group_id or 10) - \
              _METAL_OX_STATE_FALLBACK.get(metal_atoms[0].GetSymbol(), 0)
    has_co = 'CO' in smiles or '[CO]' in smiles or 'OC' in smiles
    out[8] = 1.0 if (d_count >= 9 and not has_co) else 0.0   # labile
    out[9] = 1.0 if (abs(d_count - 8) < 1 and has_co) else 0.0  # robust

    # Approx MW
    try:
        from rdkit.Chem import Descriptors
        mol_san = Chem.MolFromSmiles(smiles)
        if mol_san:
            out[10] = float(Descriptors.MolWt(mol_san))
    except Exception:
        out[10] = 0.0

    out[11] = 0.0
    bad = ~np.isfinite(out)
    out[bad] = 0.0
    return out


# -----------------------------------------------------------------
# ASSEMBLE: Apply all three blocks to df_merged
# -----------------------------------------------------------------
print("Building metal precursor feature blocks...")

# Need both SMILES and IUPAC name columns
precursor_smiles_col = next(
    (c for c in ['smiles_precursor_canon', 'smiles_precursor_raw', 'smiles_precursor']
     if c in df_merged.columns), None)
iupac_col = 'precursor_iupac_standardized'

if precursor_smiles_col is None:
    raise KeyError("No precursor SMILES column found in df_merged.")

iupac_series = df_merged[iupac_col] if iupac_col in df_merged.columns \
               else pd.Series([''] * len(df_merged))

# Block A: Metal center
Xmetal_center = np.stack(
    [get_metal_center_block(
        row[precursor_smiles_col],
        row[iupac_col] if iupac_col in df_merged.columns else '')
     for _, row in df_merged.iterrows()])

# Block B: Co-ligand inventory
Xcoligand = np.stack(
    df_merged[precursor_smiles_col].apply(get_coligand_block).values)

# Block C: Complex-level
Xcomplex_level = np.stack(
    df_merged[precursor_smiles_col].apply(get_complex_level_block).values)

# Sanity checks
for name, arr in [('Metal center', Xmetal_center),
                  ('Co-ligand',    Xcoligand),
                  ('Complex-level', Xcomplex_level)]:
    bad = ~np.isfinite(arr)
    if bad.any():
        arr[bad] = 0.0
    n_missing = int(arr[:, -1].sum())
    print(f"  {name}: {arr.shape}  |  missing: {n_missing}/{len(arr)}")

# Append to X_final — REPLACE the existing Xmetalblock concatenation
Xprecursor_full = np.concatenate(
    [Xmetal_center, Xcoligand, Xcomplex_level], axis=1)

print(f"\nTotal precursor feature block: {Xprecursor_full.shape}")
print(f"  Block A (metal center):    {METAL_BLOCK_DIM}")
print(f"  Block B (co-ligand):       {COLIGAND_BLOCK_DIM}")
print(f"  Block C (complex-level):   {COMPLEX_BLOCK_DIM}")

X_final = np.concatenate([X_final, Xprecursor_full], axis=1)
print(f"X_final shape after precursor block: {X_final.shape}")
print(f"Non-finite in X_final: {int((~np.isfinite(X_final)).sum())}")


In [ ]:


print("Building per-ligand Mordred RAC block...")

n       = len(df_merged)
n_fp_lig = len(fp_cols)            # from Block 5
D        = num_descriptors          # from existing RAC block (606)

# ----------------------------------------------------------
# 1. Compute RAC vector once per unique fp_col ligand (cached)
# ----------------------------------------------------------
rac_cache = {}   # col → (vec: np.array(D,), any_bad: float, frac_bad: float)

for col in fp_cols:
    token = col.replace('Total_', '')
    smi   = normalize_inventory_token(token)

    if smi is None:
        rac_cache[col] = (np.zeros(D, dtype=float), 1.0, 1.0)
        continue

    vec, any_bad, frac_bad = get_mordred_racs_smiles_with_stats(smi)
    rac_cache[col] = (vec, float(any_bad), float(frac_bad))

    status = "OK" if any_bad == 0.0 else f"partial ({frac_bad:.0%} bad)"
    print(f"  {token[:45]:45s}  {status}")

# ----------------------------------------------------------
# 2. Build count matrix (n, n_fp_lig) — same ligand order
#    as fp_cols so columns align with FP and TEP blocks.
# ----------------------------------------------------------
if n_fp_lig > 0:
    count_matrix_rac = np.column_stack([
        pd.to_numeric(df_merged[col], errors='coerce')
          .fillna(0.0).to_numpy()
        for col in fp_cols
    ])  # shape: (n, n_fp_lig)

    # ----------------------------------------------------------
    # 3. Stack cached vectors → (n_fp_lig, D)
    #    and missingness flags → (n_fp_lig,)
    # ----------------------------------------------------------
    rac_matrix  = np.vstack([rac_cache[col][0] for col in fp_cols])   # (L, D)
    any_bad_arr = np.array([rac_cache[col][1] for col in fp_cols], dtype=float)  # (L,)
    frac_bad_arr= np.array([rac_cache[col][2] for col in fp_cols], dtype=float)  # (L,)

    # Presence mask: 1 if ligand is present in this experiment
    presence = (count_matrix_rac > 0).astype(float)    # (n, L)

    # For each sample × ligand:
    #   cols 0..D-1 : RAC vector    (zeros if ligand absent)
    #   col  D      : any_bad flag  (0 if absent; 1 if present+failed)
    #   col  D+1    : frac_bad      (0 if absent)
    #   col  D+2    : count         (raw stoichiometric count)
    rac_expanded     = presence[:, :, np.newaxis] * rac_matrix[np.newaxis, :, :]
    # shape: (n, L, D)

    any_bad_expanded  = presence * any_bad_arr[np.newaxis, :]    # (n, L)
    frac_bad_expanded = presence * frac_bad_arr[np.newaxis, :]   # (n, L)
    count_expanded    = count_matrix_rac                          # (n, L)

    # Interleave: (n, L, D+3)
    per_lig_rac_3d = np.concatenate(
        [rac_expanded,                          # (n, L, D)
         any_bad_expanded[:, :, np.newaxis],    # (n, L, 1)
         frac_bad_expanded[:, :, np.newaxis],   # (n, L, 1)
         count_expanded[:, :, np.newaxis]],     # (n, L, 1)
        axis=2
    )  # (n, L, D+3)

    # Flatten → (n, L*(D+3))
    X_precursor_perlig_rac = per_lig_rac_3d.reshape(n, n_fp_lig * (D + 3))

else:
    X_precursor_perlig_rac = np.zeros((n, 0), dtype=float)

# ----------------------------------------------------------
# 4. Sanity checks
# ----------------------------------------------------------
print(f"\n  X_precursor_perlig_rac shape : {X_precursor_perlig_rac.shape}")
print(f"  = {n_fp_lig} ligands × ({D} RAC + 2 missingness + 1 count) = "
      f"{n_fp_lig * (D + 3)} features")
print(f"  All-zero rows: "
      f"{int((X_precursor_perlig_rac.sum(axis=1) == 0).sum())} / {n}")

bad_rac = ~np.isfinite(X_precursor_perlig_rac)
if bad_rac.any():
    print(f"  WARNING: {bad_rac.sum()} non-finite values → replacing with 0.0")
    X_precursor_perlig_rac[bad_rac] = 0.0
else:
    print("  ✓ X_precursor_perlig_rac all finite")

# ----------------------------------------------------------
# 5. Feature name tracking (for SHAP)
# ----------------------------------------------------------
perlig_rac_feature_names = []
rac_descriptor_names = list(calc.descriptors)   # same calc from RAC block

for col in fp_cols:
    token = col.replace('Total_', '')
    perlig_rac_feature_names += [f"preclig_{token}_rac_{i}" for i in range(D)]
    perlig_rac_feature_names += [
        f"preclig_{token}_rac_anybad",
        f"preclig_{token}_rac_fracbad",
        f"preclig_{token}_rac_count",
    ]

print(f"  Feature names tracked: {len(perlig_rac_feature_names)}")

# ----------------------------------------------------------
# 6. Append to X_final
# ----------------------------------------------------------
X_final = np.concatenate([X_final, X_precursor_perlig_rac], axis=1)

print(f"\nUpdated X_final shape after per-ligand RAC: {X_final.shape}")
print(f"  Added {X_precursor_perlig_rac.shape[1]} per-ligand RAC features")
print(f"  OLD weighted-sum RAC block (608 features) kept in X_final — consider")
print(f"  removing Xprecursorligracaug from the earlier concatenate if redundant")

bad = ~np.isfinite(X_final)
print(f"Non-finite entries in X_final: {int(bad.sum())}")


## Define Physicochemical Calculator and Update Matrix

### Subtask:
Define a function to calculate 9 physicochemical descriptors (including Gasteiger charges), apply it to linker and modulator columns, and concatenate the results with the existing feature matrix.


In [ ]:
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem

def finite(x, default=0.0):
    """Cast to float; replace NaN/inf/errors with default."""
    try:
        v = float(x)
        return v if np.isfinite(v) else default
    except Exception:
        return default

def get_physicochem_10(smiles):
    """
    9 physchem descriptors + 1 missing indicator.
    Patched: throwOnParamFailure=False to handle Si/Sn/Ge gracefully.
    """
    out = np.zeros(10, dtype=float)
    if pd.isna(smiles) or not isinstance(smiles, str) or not smiles.strip():
        out[9] = 1.0
        return out
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        out[9] = 1.0
        return out

    out[0] = finite(Descriptors.MolWt(mol))
    out[1] = finite(Descriptors.MolLogP(mol))
    out[2] = finite(Descriptors.TPSA(mol))
    out[3] = finite(Descriptors.NumRotatableBonds(mol))
    out[4] = finite(Descriptors.NumHAcceptors(mol))
    out[5] = finite(Descriptors.NumHDonors(mol))
    out[7] = finite(Descriptors.HallKierAlpha(mol))
    out[8] = finite(Descriptors.NumAromaticRings(mol))

    missing = 0.0
    try:
        # KEY FIX: throwOnParamFailure=False — Si/Sn/Ge will get charge=0, not crash
        AllChem.ComputeGasteigerCharges(mol, throwOnParamFailure=False)
        v = finite(Descriptors.MaxPartialCharge(mol))
        if not np.isfinite(v):
            missing = 1.0
            v = 0.0
    except Exception:
        missing = 1.0
        v = 0.0

    out[6] = v
    out[9] = missing
    return out

def _pick_first_existing(cols):
    for c in cols:
        if c in df_merged.columns:
            return c
    return None

linker_col = _pick_first_existing(["smiles_linker_1_feat", "smiles_linker_1", "smileslinker1feat", "smileslinker1"])
mod_col    = _pick_first_existing(["smiles_modulator_feat", "smiles_modulator", "smilesmodulatorfeat", "smilesmodulator"])

if linker_col is None or mod_col is None:
    raise KeyError(f"Could not find linker/modulator SMILES columns. linker={linker_col}, mod={mod_col}")

print("Calculating physicochemical descriptors for Linkers...")
X_linker_phys10 = np.stack(df_merged[linker_col].apply(get_physicochem_10).values)

print("Calculating physicochemical descriptors for Modulators...")
X_modulator_phys10 = np.stack(df_merged[mod_col].apply(get_physicochem_10).values)

# Concatenate to your existing X_final
X_final = np.concatenate([X_final, X_linker_phys10, X_modulator_phys10], axis=1)

print("Updated Feature Matrix Shape:", X_final.shape)
print("Added 10 per component:",
      ["MolWt","MolLogP","TPSA","NumRotatableBonds","NumHAcceptors","NumHDonors",
       "MaxPartialCharge_clean","HallKierAlpha","NumAromaticRings","MaxPartialCharge_missing"])

# Hard check: no NaN/inf
bad = ~np.isfinite(X_final)
print("Non-finite entries in X_final:", int(bad.sum()))


#TEP Calculation


#Calculate TEP
https://pubs.acs.org/doi/abs/10.1021/acs.organomet.3c00432

In [ ]:


import joblib
import numpy as np
import pandas as pd
import urllib.request
import lightgbm as lgb
from rdkit import Chem
from rdkit.Chem import Descriptors

url = "https://github.com/DanielEss-lab/TEPid/raw/main/Machine_Learning_Model/LGBMReg_model.pkl"
urllib.request.urlretrieve(url, "LGBMReg_model.pkl")

tepid_model = joblib.load("LGBMReg_model.pkl")
try:
    tep_features = tepid_model.feature_name_
except AttributeError:
    tep_features = tepid_model.booster_.feature_name()

def get_tepid_value(smiles):
    if pd.isna(smiles) or not isinstance(smiles, str) or not smiles.strip():
        return [0.0, 1.0]

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return [0.0, 1.0]

    try:
        all_descs = Descriptors.CalcMolDescriptors(mol)
        x_input = np.array(
            [[all_descs.get(feat, 0.0) for feat in tep_features]],
            dtype=np.float64
        )   # shape (1, 19)

        # Use booster_ directly — bypasses the broken sklearn wrapper
        tep_val = tepid_model.booster_.predict(x_input)[0]
        return [float(tep_val), 0.0]

    except Exception as e:
        print(f" ERROR [{type(e).__name__}]: {e}")
        return [0.0, 1.0]


print("Calculating TEPid for Modulators...")
X_modulator_tep = np.stack(df_merged[mod_col].apply(get_tepid_value).values)

print("Calculating TEPid for Linkers...")
X_linker_tep = np.stack(df_merged[linker_col].apply(get_tepid_value).values)

# ============================================================
# PER-LIGAND TEP BLOCK
# REPLACES: weighted-sum Xprecursorligtep
# Uses fp_cols from the per-ligand FP block (Block 5) so the
# ligand ordering is IDENTICAL — one TEP triple per FP block.
# ============================================================

n = len(df_merged)
print("Calculating per-ligand TEPid for Precursor Ligands...")

# ----------------------------------------------------------
# 1. Compute TEP once per unique fp_col ligand (cached)
# ----------------------------------------------------------

tep_cache = {}   # col → (tep_val: float, missing_flag: float)

for col in fp_cols:                       # fp_cols from Block 5
    token = col.replace('Total_', '')
    smi   = normalize_inventory_token(token)

    if smi is None:
        tep_cache[col] = (0.0, 1.0)      # missing
        continue

    tep_val, miss_flag = get_tepid_value(smi)  # your existing fn

    # get_tepid_value returns (tep, missing_flag) where
    # missing_flag=1.0 means TEP could not be computed.
    tep_cache[col] = (float(tep_val), float(miss_flag))

    if miss_flag == 0.0:
        print(f"  {token[:40]:40s}  TEP = {tep_val:.1f} cm⁻¹")
    else:
        print(f"  {token[:40]:40s}  TEP = MISSING (flag=1)")

# ----------------------------------------------------------
# 2. Build fixed-width per-ligand TEP matrix
#    Columns per ligand: [tep_value, missing_flag, count]
#    Shape: (n, n_fp_ligands * 3)
# ----------------------------------------------------------
n_fp_lig = len(fp_cols)

if n_fp_lig > 0:
    # Count matrix already computed in Block 5, but we rebuild
    # here cleanly to avoid any dependency on variable lifetime.
    count_matrix_tep = np.column_stack([
        pd.to_numeric(df_merged[col], errors='coerce')
          .fillna(0.0).to_numpy()
        for col in fp_cols
    ])  # shape: (n, n_fp_lig)

    # tep_vals and missing_flags as 1-D arrays (one per ligand)
    tep_vals  = np.array([tep_cache[col][0] for col in fp_cols], dtype=float)
    miss_flags = np.array([tep_cache[col][1] for col in fp_cols], dtype=float)

    # Presence mask: 1 where ligand count > 0, else 0
    # Keeps TEP = 0 for truly absent ligands (distinguishable
    # from a ligand with TEP=0 via the missing_flag column)
    presence = (count_matrix_tep > 0).astype(float)   # (n, L)

    # Broadcast: for each sample × ligand:
    #   col 0: tep_value   (0 if absent)
    #   col 1: missing_flag (0 if absent, but 1 if present+failed)
    #   col 2: count        (raw stoichiometric count)
    tep_block    = presence * tep_vals[np.newaxis, :]           # (n, L)
    miss_block   = presence * miss_flags[np.newaxis, :]         # (n, L)
    count_block  = count_matrix_tep                              # (n, L)

    # Interleave: (n, L, 3) → (n, L*3)
    per_lig_tep_3d = np.stack(
        [tep_block, miss_block, count_block], axis=2
    )  # (n, L, 3)
    X_precursor_perlig_tep = per_lig_tep_3d.reshape(n, n_fp_lig * 3)

else:
    X_precursor_perlig_tep = np.zeros((n, 0), dtype=float)

print(f"\n  X_precursor_perlig_tep shape : {X_precursor_perlig_tep.shape}")
print(f"  = {n_fp_lig} ligands × 3 (tep | missing_flag | count)")

# Sanity check
bad_tep = ~np.isfinite(X_precursor_perlig_tep)
if bad_tep.any():
    print(f"  WARNING: {bad_tep.sum()} non-finite → replacing with 0.0")
    X_precursor_perlig_tep[bad_tep] = 0.0
else:
    print("  ✓ X_precursor_perlig_tep all finite")

# ----------------------------------------------------------
# 3. Feature name tracking (for SHAP / importance analysis)
# ----------------------------------------------------------
perlig_tep_feature_names = []
for col in fp_cols:
    token = col.replace('Total_', '')
    perlig_tep_feature_names += [
        f"preclig_{token}_tep",
        f"preclig_{token}_tep_missing",
        f"preclig_{token}_tep_count",
    ]

# ----------------------------------------------------------
# 4. Concatenate — replaces old Xprecursorligtep
# ----------------------------------------------------------
X_final = np.concatenate(
    [X_final, X_modulator_tep, X_linker_tep, X_precursor_perlig_tep],
    axis=1
)

print(f"\nUpdated Feature Matrix Shape with per-ligand TEP: {X_final.shape}")
print(f"  Modulator TEP     : {X_modulator_tep.shape}   (tep, missing_flag)")
print(f"  Linker TEP        : {X_linker_tep.shape}       (tep, missing_flag)")
print(f"  Precursor per-lig : {X_precursor_perlig_tep.shape}  ← NEW")
print(f"  OLD was           : ({n}, 2) weighted sum ← gone")

bad = ~np.isfinite(X_final)
print(f"Non-finite entries in X_final: {int(bad.sum())}")


#Calculate Sterics

In [ ]:
import numpy as np
import pandas as pd
from rdkit import Chem, rdBase
from rdkit.Chem import AllChem
from morfeus import ConeAngle, BuriedVolume
from collections import Counter
from sklearn.impute import SimpleImputer

rdBase.DisableLog("rdApp.error")
rdBase.DisableLog("rdApp.warning")

# Toggle this to see a few failure reasons instead of silently imputing everything
DEBUG_STERICS = True
_debug_errors = []  # collects tuples like (stage, smiles, exception_str)

def process_for_sterics(smiles, extra_remove=None):
    """
    Convert a raw SMILES into a representative ligand fragment for sterics.
    Removes metals (and optionally Sn/Si handles), fragments, and returns the most common P-containing fragment.
    """
    target_metals = {'Pd','Rh','Pt','Ag','Ir','Au','Cu','Co','Ni','Fe','Ru','Os'}
    remove = set(target_metals)
    if extra_remove:
        remove |= set(extra_remove)

    if not isinstance(smiles, str) or smiles.strip() == "":
        return None

    try:
        mol = Chem.MolFromSmiles(smiles, sanitize=False)
        if mol is None:
            return None

        rwmol = Chem.RWMol(mol)
        idxs = [a.GetIdx() for a in rwmol.GetAtoms() if a.GetSymbol() in remove]
        for idx in sorted(idxs, reverse=True):
            rwmol.RemoveAtom(idx)

        frag_mol = rwmol.GetMol()
        frags = Chem.GetMolFrags(frag_mol, asMols=True, sanitizeFrags=False)

        cands = []
        for f in frags:
            try:
                s = Chem.MolToSmiles(f, canonical=True)
                if s:
                    cands.append(s)
            except Exception:
                pass

        if not cands:
            return None

        p_cands = [s for s in cands if ('P' in s or 'p' in s)]
        if p_cands:
            return Counter(p_cands).most_common(1)[0][0]
        return Counter(cands).most_common(1)[0][0]

    except Exception as e:
        if DEBUG_STERICS:
            _debug_errors.append(("process_for_sterics", smiles, repr(e)))
        return None


def _heavy_atom_elements_coords(mol_with_H):
    conf = mol_with_H.GetConformer()
    heavy_idxs = [a.GetIdx() for a in mol_with_H.GetAtoms() if a.GetSymbol() != "H"]
    elements = [mol_with_H.GetAtomWithIdx(i).GetSymbol() for i in heavy_idxs]
    coords = np.array([list(conf.GetAtomPosition(i)) for i in heavy_idxs], dtype=float)
    return heavy_idxs, elements, coords

def get_phosphine_sterics(smiles):
    if pd.isna(smiles) or not isinstance(smiles, str) or smiles.strip() == "":
        return 0.0, 0.0
    if ('P' not in smiles) and ('p' not in smiles):
        return 0.0, 0.0

    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return np.nan, np.nan

        p_atoms = [a for a in mol.GetAtoms() if a.GetSymbol() == "P"]
        if len(p_atoms) == 0:
            return 0.0, 0.0

        molH = Chem.AddHs(mol)

        ps = AllChem.ETKDGv3()
        ps.useRandomCoords = True
        ps.maxIterations = 2000
        if AllChem.EmbedMolecule(molH, ps) == -1:
            return np.nan, np.nan
        try:
            AllChem.UFFOptimizeMolecule(molH, maxIters=200)
        except Exception:
            pass

        conf = molH.GetConformer()

        # find a P atom index in the H-added molecule (same indexing as original heavy atoms)
        p_idx = next(a.GetIdx() for a in molH.GetAtoms() if a.GetSymbol() == "P")
        p_atom = molH.GetAtomWithIdx(p_idx)

        # build lone-pair direction from neighbors
        p_pos = np.array(conf.GetAtomPosition(p_idx))
        neighbors = p_atom.GetNeighbors()
        if not neighbors:
            return np.nan, np.nan

        vec_sum = np.zeros(3)
        for n in neighbors:
            vec_sum += (np.array(conf.GetAtomPosition(n.GetIdx())) - p_pos)

        norm = np.linalg.norm(vec_sum)
        if norm < 1e-8:
            return np.nan, np.nan

        lp_vec = -vec_sum / norm

        # heavy-atom-only set for Morfeus
        heavy_idxs, elements_heavy, coords_heavy = _heavy_atom_elements_coords(molH)

        # map P index -> heavy-atom coordinate index
        try:
            p_heavy_i = heavy_idxs.index(p_idx)
        except ValueError:
            return np.nan, np.nan

        # try increasing dummy distances to avoid vdW collisions
        for dist in [2.28, 2.60, 3.00, 3.50, 4.00]:
            dummy_pos = p_pos + lp_vec * dist

            elements = elements_heavy + ["Pd"]
            coords = np.vstack([coords_heavy, dummy_pos])

            # Morfeus uses 1-indexed central atom index [web:6]
            metal_idx = len(elements)  # 1-indexed central atom (dummy Pd is last)
            try:
              ca = ConeAngle(elements, coords, metal_idx, method="internal")  # or omit method=
              bv = BuriedVolume(elements, coords, metal_idx, radius=3.5)
              return float(ca.cone_angle), float(bv.fraction_buried_volume) * 100.0
            except Exception as e:
              if DEBUG_STERICS:
                _debug_errors.append(("morfeus", smiles, f"dist={dist}", repr(e)))
              continue


        return np.nan, np.nan

    except Exception:
        return np.nan, np.nan



def map_sterics_processed(df, src_col, prefix, processed_col, extra_remove=None):
    if src_col not in df.columns:
        raise KeyError(f"{src_col} not in df.columns")

    df[processed_col] = df[src_col].apply(lambda s: process_for_sterics(s, extra_remove=extra_remove))

    targets = (
        df[processed_col]
        .dropna()
        .astype(str)
        .map(str.strip)
    )
    targets = targets[targets != ""].unique()

    print(f"{src_col}: raw unique={df[src_col].nunique(dropna=False)}; processed unique={len(targets)}")

    cache = {t: get_phosphine_sterics(t) for t in targets}

    df[f"{prefix}_cone_angle"] = df[processed_col].map(lambda x: cache.get(x, (0.0, 0.0))[0])
    df[f"{prefix}_buried_vol"] = df[processed_col].map(lambda x: cache.get(x, (0.0, 0.0))[1])


print("Re-mapping sterics on processed fragments...")
print("About to run modulator…")
map_sterics_processed(df_merged, "smiles_modulator", "modulator", "modulator_sterics_smiles")

print("About to run linker1…")
map_sterics_processed(df_merged, "smiles_linker_1", "linker1", "linker1_sterics_smiles", extra_remove={"Sn", "Si"})

print("About to run precursor…")
map_sterics_processed(df_merged, "smiles_precursor", "precursor", "precursor_sterics_smiles")

steric_cols = [
    "modulator_cone_angle", "modulator_buried_vol",
    "linker1_cone_angle", "linker1_buried_vol",
]

# Imputation (unchanged idea, but now failures are meaningful NaNs)
all_nan_cols = df_merged[steric_cols].columns[df_merged[steric_cols].isna().all()].tolist()
if all_nan_cols:
    print(f"Warning: Entirely-NaN sterics columns filled with 0.0: {all_nan_cols}")
    df_merged[all_nan_cols] = 0.0

cols_to_impute = [c for c in steric_cols if df_merged[c].isna().any()]
if cols_to_impute:
    print(f"Imputing: {cols_to_impute}")
    imputer = SimpleImputer(missing_values=np.nan, strategy="median")
    df_merged[cols_to_impute] = imputer.fit_transform(df_merged[cols_to_impute])

print("NaN counts:")
print(df_merged[steric_cols].isna().sum())

if DEBUG_STERICS and len(_debug_errors) > 0:
    print("\nFirst 10 sterics debug errors:")
    for e in _debug_errors[:10]:
        print(e)


In [ ]:
# ============================================================
# BLOCK: Per-Ligand Sterics for Precursor Ligands
# Uses the EXACT same get_phosphine_sterics() and
# process_for_sterics() functions already defined above.
#
# Pattern mirrors fp_cols / TEP / RAC blocks:
#   For each fp_col ligand → [cone_angle, buried_vol,
#                              steric_missing_flag, count]
#   Non-phosphine ligands → [0.0, 0.0, 0.0, count]
#   Absent ligands        → [0.0, 0.0, 0.0, 0.0]
#
# Shape: (n_samples, n_fp_lig * 4)
# Ligand ordering is IDENTICAL to fp_cols → aligns with
# FP, TEP, and RAC blocks for SHAP grouping.
# ============================================================

print("Building per-ligand sterics block for precursor ligands...")

n        = len(df_merged)
n_fp_lig = len(fp_cols)    # from Block 5

# ----------------------------------------------------------
# 1. Compute sterics once per unique fp_col ligand (cached)
#    Uses process_for_sterics() then get_phosphine_sterics()
#    — exactly how the modulator and linker sterics are done.
# ----------------------------------------------------------
steric_cache = {}   # col → (cone: float, buried: float, missing: float)

for col in fp_cols:
    token = col.replace('Total_', '')
    smi   = normalize_inventory_token(token)

    # Non-SMILES token (shouldn't happen for fp_cols, but guard)
    if smi is None:
        steric_cache[col] = (0.0, 0.0, 0.0)
        continue

    # Step 1: Run process_for_sterics to clean/fragment the SMILES
    # (same function used for modulator/linker sterics)
    processed_smi = process_for_sterics(smi, extra_remove=None)

    # Step 2: Check for phosphorus — non-P ligands get zeros, no flag
    if processed_smi is None or ('P' not in processed_smi and 'p' not in processed_smi):
        steric_cache[col] = (0.0, 0.0, 0.0)   # not a phosphine → no steric
        print(f"  {token[:45]:45s}  non-phosphine → zeros")
        continue

    # Step 3: Call get_phosphine_sterics (Morfeus-based, same as above)
    cone, buried = get_phosphine_sterics(processed_smi)

    if np.isnan(cone) or np.isnan(buried):
        steric_cache[col] = (0.0, 0.0, 1.0)   # phosphine but calculation failed
        print(f"  {token[:45]:45s}  cone=NaN  buried=NaN  → missing_flag=1")
    else:
        steric_cache[col] = (float(cone), float(buried), 0.0)
        print(f"  {token[:45]:45s}  cone={cone:.1f}°  buried={buried:.1f}%")

# ----------------------------------------------------------
# 2. Build count matrix (n, n_fp_lig) — reuses fp_cols order
# ----------------------------------------------------------
if n_fp_lig > 0:
    count_matrix_steric = np.column_stack([
        pd.to_numeric(df_merged[col], errors='coerce')
          .fillna(0.0).to_numpy()
        for col in fp_cols
    ])  # shape: (n, n_fp_lig)

    # ----------------------------------------------------------
    # 3. Unpack cached values into arrays (one entry per ligand)
    # ----------------------------------------------------------
    cone_arr    = np.array([steric_cache[col][0] for col in fp_cols], dtype=float)
    buried_arr  = np.array([steric_cache[col][1] for col in fp_cols], dtype=float)
    missing_arr = np.array([steric_cache[col][2] for col in fp_cols], dtype=float)

    # Presence mask: 1 where count > 0
    presence = (count_matrix_steric > 0).astype(float)   # (n, n_fp_lig)

    # For each sample × ligand (4 features per ligand):
    #   col 0: cone_angle   (0 if absent or non-phosphine)
    #   col 1: buried_vol   (0 if absent or non-phosphine)
    #   col 2: missing_flag (0 if absent; 1 if phosphine but calc failed)
    #   col 3: count        (raw stoichiometric count)
    cone_block    = presence * cone_arr[np.newaxis, :]     # (n, L)
    buried_block  = presence * buried_arr[np.newaxis, :]   # (n, L)
    missing_block = presence * missing_arr[np.newaxis, :]  # (n, L)
    count_block   = count_matrix_steric                    # (n, L)

    # Interleave into (n, L, 4) → flatten to (n, L*4)
    per_lig_steric_3d = np.stack(
        [cone_block, buried_block, missing_block, count_block],
        axis=2
    )  # (n, L, 4)

    X_precursor_perlig_steric = per_lig_steric_3d.reshape(n, n_fp_lig * 4)

else:
    X_precursor_perlig_steric = np.zeros((n, 0), dtype=float)

# ----------------------------------------------------------
# 4. Sanity checks
# ----------------------------------------------------------
print(f"\n  X_precursor_perlig_steric shape : {X_precursor_perlig_steric.shape}")
print(f"  = {n_fp_lig} ligands × 4 (cone | buried | missing_flag | count)")
print(f"  All-zero rows: "
      f"{int((X_precursor_perlig_steric.sum(axis=1) == 0).sum())} / {n}")

# Count how many ligands actually have steric values
n_with_sterics = sum(1 for col in fp_cols
                     if steric_cache[col][2] == 0.0   # not missing
                     and steric_cache[col][0] > 0.0)  # non-zero cone
print(f"  Ligands with valid steric values : {n_with_sterics} / {n_fp_lig}")
print(f"  Ligands non-phosphine (zeros)    : "
      f"{sum(1 for col in fp_cols if steric_cache[col] == (0.0, 0.0, 0.0))}")
print(f"  Ligands phosphine but failed     : "
      f"{sum(1 for col in fp_cols if steric_cache[col][2] == 1.0)}")

bad_steric = ~np.isfinite(X_precursor_perlig_steric)
if bad_steric.any():
    print(f"  WARNING: {bad_steric.sum()} non-finite → replacing with 0.0")
    X_precursor_perlig_steric[bad_steric] = 0.0
else:
    print("  ✓ X_precursor_perlig_steric all finite")

# ----------------------------------------------------------
# 5. Feature name tracking (for SHAP)
# ----------------------------------------------------------
perlig_steric_feature_names = []
for col in fp_cols:
    token = col.replace('Total_', '')
    perlig_steric_feature_names += [
        f"preclig_{token}_cone_angle",
        f"preclig_{token}_buried_vol",
        f"preclig_{token}_steric_missing",
        f"preclig_{token}_steric_count",
    ]

print(f"  Feature names tracked: {len(perlig_steric_feature_names)}")

# ----------------------------------------------------------
# 6. Append to X_final
# ----------------------------------------------------------
X_final = np.concatenate([X_final, X_precursor_perlig_steric], axis=1)

print(f"\nUpdated X_final shape after per-ligand sterics: {X_final.shape}")
print(f"  Added {X_precursor_perlig_steric.shape[1]} per-ligand steric features")
print(f"  (cone_angle + buried_vol + missing_flag + count) × {n_fp_lig} ligands")

bad = ~np.isfinite(X_final)
print(f"Non-finite entries in X_final: {int(bad.sum())}")


In [ ]:
import pandas as pd

def assert_required_columns(df: pd.DataFrame, colmap: dict, required_keys=("id","precursor","linker1","modulator")):
    missing_keys = [k for k in required_keys if k not in colmap]
    if missing_keys:
        raise KeyError(f"COLMAP is missing keys: {missing_keys}")

    missing_cols = [colmap[k] for k in required_keys if colmap[k] not in df.columns]
    if missing_cols:
        raise KeyError(f"DataFrame is missing required columns: {missing_cols}")

    id_col = colmap["id"]
    null_ids = df[id_col].isna().sum()
    dup_ids = df[id_col].duplicated().sum()

    if null_ids > 0:
        raise ValueError(f"{id_col} has {null_ids} null values (merge/key will break).")
    if dup_ids > 0:
        # not always fatal, but usually indicates you need a compound key
        print(f"WARNING: {id_col} has {dup_ids} duplicate values.")

# run it
assert_required_columns(df, COLMAP)


In [ ]:
import numpy as np
from rdkit import Chem, RDLogger

# Optional: silence RDKit parse spam while still tracking failures
RDLogger.DisableLog("rdApp.error")
RDLogger.DisableLog("rdApp.warning")

def clean_smiles(x):
    """Normalize missing/placeholder values but DO NOT rename df columns."""
    if pd.isna(x) or not isinstance(x, str):
        return None
    s = x.strip()
    if not s or s.lower() in {"none", "nan"}:
        return None
    return s

def smiles_parse_ok(smiles):
    s = clean_smiles(smiles)
    if s is None:
        return False
    m = Chem.MolFromSmiles(s)
    if m is not None:
        return True
    # fallback: sometimes sanitization fails for weird coordination/ions
    m2 = Chem.MolFromSmiles(s, sanitize=False)
    return m2 is not None

def add_parse_diagnostics(df, colmap, keys=("precursor","linker1","modulator","linker2")):
    out = df.copy()
    for k in keys:
        if k not in colmap or colmap[k] not in out.columns:
            continue
        col = colmap[k]
        out[f"qa_{k}_smiles_clean"] = out[col].map(clean_smiles)
        out[f"qa_{k}_parse_ok"] = out[col].map(smiles_parse_ok)

        # Print quick stats
        n = len(out)
        ok = int(out[f"qa_{k}_parse_ok"].sum())
        print(f"{k}: parse_ok {ok}/{n} ({ok/n:.1%})")

        # Show top failing raw strings
        fails = out.loc[~out[f"qa_{k}_parse_ok"], col].value_counts(dropna=False).head(15)
        if len(fails) > 0:
            print(f"Top failing {k} strings:")
            print(fails)
            print("-"*60)
    return out

df_qa = add_parse_diagnostics(df, COLMAP)


In [ ]:
from rdkit.Chem import AllChem, DataStructs

def morgan_fp_numpy(smiles, nbits=2048, radius=2):
    arr = np.zeros((nbits,), dtype=np.int8)
    s = clean_smiles(smiles)
    if s is None:
        return arr, False

    mol = Chem.MolFromSmiles(s)
    if mol is None:
        mol = Chem.MolFromSmiles(s, sanitize=False)
        if mol is None:
            return arr, False

    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nbits)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr, True

def fp_zero_report(df, colmap, key="linker1", nbits=2048):
    col = colmap[key]
    fps = []
    ok_flags = []
    for s in df[col].tolist():
        v, ok = morgan_fp_numpy(s, nbits=nbits)
        fps.append(v)
        ok_flags.append(ok)
    X = np.stack(fps)
    ok_flags = np.array(ok_flags, dtype=bool)

    nnz = X.sum(axis=1)
    zero = (nnz == 0)
    print(f"{key}: fp_parse_ok {ok_flags.sum()}/{len(ok_flags)}")
    print(f"{key}: all-zero fingerprints {int(zero.sum())}/{len(zero)} ({zero.mean():.1%})")

    # show a few problematic rows
    bad_idx = np.where(zero)[0][:10]
    if len(bad_idx) > 0:
        print("Example all-zero rows:")
        display(df.iloc[bad_idx][[colmap['id'], col]])

    return X, ok_flags, zero

X_linker, ok_linker, zero_linker = fp_zero_report(df, COLMAP, key="linker1")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
#  BLOCK: ChemBERTa-2 + Extended RDKit Featurization (Linker + Modulator)
# ════════════════════════════════════════════════════════════════════════════

import numpy as np
import torch
from rdkit import Chem
from rdkit.Chem import Descriptors, rdMolDescriptors, AllChem
from transformers import AutoTokenizer, AutoModel

# ─────────────────────────────────────────────────────────────────────────────
# 0.  Helpers
# ─────────────────────────────────────────────────────────────────────────────
def _f(x, default=0.0):
    """Safe float cast; replaces NaN/Inf with default."""
    try:
        v = float(x)
        return v if np.isfinite(v) else default
    except Exception:
        return default


# ─────────────────────────────────────────────────────────────────────────────
# 1.  ChemBERTa-2 MTR  (384-dim CLS token)
#     Pre-trained on 77 M PubChem SMILES with multi-task regression
#     over 200 RDKit properties → rich contextual organic embeddings
# ─────────────────────────────────────────────────────────────────────────────
CHEMBERTA_MODEL = "DeepChem/ChemBERTa-77M-MTR"
BERT_DIM        = 384

print(f"Loading {CHEMBERTA_MODEL} …")
_cb_tok = AutoTokenizer.from_pretrained(CHEMBERTA_MODEL)
_cb_mod = AutoModel.from_pretrained(CHEMBERTA_MODEL)
_cb_mod.eval()


def chemberta_batch(smiles_list, batch_size=32):
    """
    Returns CLS-token embeddings as np.ndarray (n, 384).
    Invalid / missing SMILES → zero vector (NOT flagged separately;
    the zero vector is a natural out-of-distribution signal).
    """
    n   = len(smiles_list)
    out = np.zeros((n, BERT_DIM), dtype=float)

    for i in range(0, n, batch_size):
        raw   = smiles_list[i : i + batch_size]
        clean = [s if (isinstance(s, str) and s.strip()) else "C"
                 for s in raw]                         # fallback = methane

        enc = _cb_tok(
            clean,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt",
        )
        with torch.no_grad():
            h = _cb_mod(**enc).last_hidden_state[:, 0, :].cpu().numpy()
        out[i : i + len(clean)] = h

    return out   # (n, 384)


def chemberta_feature_names(prefix):
    return [f"{prefix}_bert_{i}" for i in range(BERT_DIM)]


# ─────────────────────────────────────────────────────────────────────────────
# 2.  Extended RDKit Descriptor Suite  (50 features per molecule)
#
#     Deliberately avoids duplicating the 9 descriptors already computed
#     in get_physicochem_10 (MolWt, MolLogP, TPSA, NumRotBonds,
#     NumHAcceptors, NumHDonors, MaxPartialCharge, HallKierAlpha,
#     NumAromaticRings, missing_flag).
# ─────────────────────────────────────────────────────────────────────────────

# MOF-relevant SMARTS patterns
#   Coordination sites  → determines denticity / binding mode
#   Functional groups   → electronic effects on metal-linker bond
#   Halogens            → withdrawing/donating effects
_SMARTS_RAW = {
    # ── Coordination / binding sites ────────────────────────────────────────
    "COOH":         "[CX3](=O)[OX2H1]",          # carboxylic acid
    "COO_neg":      "[CX3](=O)[O-]",              # carboxylate (deprotonated)
    "py_N":         "[$([nX2](:*):*),$([nX3](:*)(:*):*)]",  # aromatic N (pyridine/imidazole-type)
    "amine_prim":   "[NH2;!$(N-C=O);!$(N~[#7,#8,F,Cl,Br,I,S])]",
    "amine_sec":    "[NH1;!$(N-C=O);!$(N~[#7,#8,F,Cl,Br,I,S])]",
    "hydroxyl":     "[OX2H;!$(OC=O)]",            # alcohol / phenol
    "ether_O":      "[OX2;!$(O=*);!$([OH])]",     # ether oxygen
    "imidazole":    "c1cnc[nH]1",
    "tetrazole":    "c1nnn[nH]1",
    "catechol":     "c1ccc(O)c(O)c1",
    "phosphonate":  "[PX4](=O)([OX2H])[OX2H]",
    "sulfonate":    "S(=O)(=O)[OX2H1]",
    # ── Reactive / electronic groups ────────────────────────────────────────
    "aldehyde":     "[CX3H1](=O)[#6]",
    "ketone":       "[CX3](=O)([#6])[#6]",
    "nitro":        "[N+](=O)[O-]",
    "cyano":        "[CX2]#[NX1]",
    "thiol":        "[SX2H]",
    # ── Halogens ────────────────────────────────────────────────────────────
    "F":            "[F]",
    "Cl":           "[Cl]",
    "Br":           "[Br]",
    "I":            "[I]",
}

# Pre-compile patterns once
_SMARTS = {k: Chem.MolFromSmarts(v) for k, v in _SMARTS_RAW.items()}
_N_SMARTS = len(_SMARTS)  # 20

# Coordination sites to sum for total denticity
_COORD_KEYS = ["COOH", "COO_neg", "py_N", "amine_prim",
               "amine_sec", "hydroxyl", "imidazole", "tetrazole",
               "phosphonate", "sulfonate"]

#  Layout of the 50-element output vector:
#  [0]   BertzCT            – molecular complexity (topological)
#  [1]   MolMR              – molar refractivity (polarizability proxy)
#  [2]   LabuteASA          – approximate accessible surface area
#  [3]   FractionCSP3       – fraction sp3 carbons (rigidity / flexibility)
#  [4-8] Chi0v … Chi4v      – valence connectivity indices (electronic topology)
#  [9-11] Kappa1 … Kappa3   – shape / branching indices
#  [12]  nAliphaticRings
#  [13]  nSaturatedRings
#  [14]  RingCount           – total ring count (all types)
#  [15]  nBridgeheadAtoms
#  [16]  nSpiroAtoms
#  [17]  MinPartialCharge
#  [18]  MaxAbsPartialCharge
#  [19]  MinAbsPartialCharge
#  [20]  nHeteroatoms
#  [21]  NumRadicalElectrons
#  [22]  NumValenceElectrons
#  [23]  NOCount             – N + O atom count
#  [24]  NHOHCount           – N-H + O-H count
#  [25]  TPSA_per_MW         – normalized polarity
#  [26]  aromatic_frac       – fraction aromatic atoms
#  [27]  nCoordSites         – total putative coordination / binding sites
#  [28-47] smarts_*          – one count per _SMARTS entry (20 patterns)
#  [48]  charge_range        – MaxPartialCharge − MinPartialCharge (asymmetry)
#  [49]  ext_missing         – 1.0 if SMILES was invalid

_N_BASE   = 28          # indices 0–27
_N_TOTAL  = _N_BASE + _N_SMARTS + 2   # 28 + 20 + 2 = 50

def get_ext_rdkit(smiles):
    out = np.zeros(_N_TOTAL, dtype=float)
    missing_idx   = _N_TOTAL - 1          # index 49
    charge_rng_idx = _N_TOTAL - 2         # index 48

    if not isinstance(smiles, str) or not smiles.strip():
        out[missing_idx] = 1.0
        return out

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        out[missing_idx] = 1.0
        return out

    # ── Complexity / surface ────────────────────────────────────────────────
    out[0]  = _f(Descriptors.BertzCT(mol))
    out[1]  = _f(Descriptors.MolMR(mol))
    out[2]  = _f(Descriptors.LabuteASA(mol))
    out[3]  = _f(Descriptors.FractionCSP3(mol))

    # ── Valence connectivity (electronic topology) ─────────────────────────
    out[4]  = _f(Descriptors.Chi0v(mol))
    out[5]  = _f(Descriptors.Chi1v(mol))
    out[6]  = _f(Descriptors.Chi2v(mol))
    out[7]  = _f(Descriptors.Chi3v(mol))
    out[8]  = _f(Descriptors.Chi4v(mol))

    # ── Shape indices ──────────────────────────────────────────────────────
    out[9]  = _f(Descriptors.Kappa1(mol))
    out[10] = _f(Descriptors.Kappa2(mol))
    out[11] = _f(Descriptors.Kappa3(mol))

    # ── Ring system ───────────────────────────────────────────────────────
    out[12] = _f(rdMolDescriptors.CalcNumAliphaticRings(mol))
    out[13] = _f(rdMolDescriptors.CalcNumSaturatedRings(mol))
    out[14] = _f(Descriptors.RingCount(mol))
    out[15] = _f(rdMolDescriptors.CalcNumBridgeheadAtoms(mol))
    out[16] = _f(rdMolDescriptors.CalcNumSpiroAtoms(mol))

    # ── Partial charge distribution ───────────────────────────────────────
    try:
        AllChem.ComputeGasteigerCharges(mol, throwOnParamFailure=True)
        mn  = _f(Descriptors.MinPartialCharge(mol))
        mx  = _f(Descriptors.MaxPartialCharge(mol))
        out[17] = mn
        out[18] = _f(Descriptors.MaxAbsPartialCharge(mol))
        out[19] = _f(Descriptors.MinAbsPartialCharge(mol))
        out[charge_rng_idx] = mx - mn     # charge asymmetry / polarity spread
    except Exception:
        pass   # leave as 0 (no missing flag; Gasteiger failure is common for heteroatom-rich mols)

    # ── Atom composition ──────────────────────────────────────────────────
    out[20] = _f(rdMolDescriptors.CalcNumHeteroatoms(mol))
    out[21] = _f(Descriptors.NumRadicalElectrons(mol))
    out[22] = _f(Descriptors.NumValenceElectrons(mol))
    out[23] = _f(Descriptors.NOCount(mol))
    out[24] = _f(Descriptors.NHOHCount(mol))

    # ── Normalized / derived ─────────────────────────────────────────────
    mw   = _f(Descriptors.MolWt(mol))
    tpsa = _f(Descriptors.TPSA(mol))
    out[25] = tpsa / mw if mw > 0 else 0.0              # polarity per unit mass

    n_atoms  = mol.GetNumAtoms()
    n_arom   = sum(1 for a in mol.GetAtoms() if a.GetIsAromatic())
    out[26]  = n_arom / n_atoms if n_atoms > 0 else 0.0  # aromatic fraction

    # ── Coordination / binding site total (denticity proxy) ──────────────
    coord_total = 0
    for k in _COORD_KEYS:
        patt = _SMARTS.get(k)
        if patt is not None:
            coord_total += len(mol.GetSubstructMatches(patt))
    out[27] = float(coord_total)

    # ── Individual SMARTS counts ──────────────────────────────────────────
    for j, (name, patt) in enumerate(_SMARTS.items()):
        if patt is not None:
            out[_N_BASE + j] = float(len(mol.GetSubstructMatches(patt)))

    return out


# Build matching feature names for SHAP
def ext_rdkit_feature_names(prefix):
    base_names = [
        "bertzCT", "molMR", "labuteASA", "fracCSP3",
        "chi0v", "chi1v", "chi2v", "chi3v", "chi4v",
        "kappa1", "kappa2", "kappa3",
        "nAliphaticRings", "nSaturatedRings", "ringCount",
        "nBridgeheadAtoms", "nSpiroAtoms",
        "minPartialCharge", "maxAbsPartialCharge", "minAbsPartialCharge",
        "nHeteroatoms", "nRadicalElectrons", "nValenceElectrons",
        "NOCount", "NHOHCount",
        "TPSA_per_MW", "aromaticFrac", "nCoordSites",
    ]
    smarts_names = [f"smarts_{k}" for k in _SMARTS.keys()]
    tail = ["chargeRange", "ext_missing"]
    return [f"{prefix}_{n}" for n in base_names + smarts_names + tail]
# Add this right after the ext_rdkit_feature_names() definition
_EXT_RDKIT_BASE_NAMES = [
    n.replace("x_", "") for n in ext_rdkit_feature_names("x")
]
# e.g. ["bertzCT", "molMR", ..., "ext_missing"]


assert len(ext_rdkit_feature_names("x")) == _N_TOTAL, \
    f"Name count mismatch: {len(ext_rdkit_feature_names('x'))} vs {_N_TOTAL}"



In [ ]:
# ── Block A: 3D Shape Descriptors (ETKDGv3 conformer) ────────────────────────
# NPR1/2 encode rod↔disk↔sphere shape WITHOUT size dependency (0–1 range)
# PMIs are size-dependent; Asphericity/Eccentricity/Spherocity are normalized
# conformer_failed=1 acts as its own missingness signal

def get_3d_shape(smiles, n_conf=1):
    """
    3D shape descriptors with multi-strategy embedding for large macrocycles.
    Returns zeros with flag=-1.0 in last slot on failure.
    Output: [NPR1, NPR2, Asphericity, Eccentricity, InertialShapeFactor,
             PMI1, PMI2, PMI3, SpherocityIndex, missing_flag]
    """
    out = np.full(10, 0.0, dtype=float)
    out[9] = -1.0  # assume failure initially

    if pd.isna(smiles) or not isinstance(smiles, str) or not smiles.strip():
        return out
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return out

    molH = Chem.AddHs(mol)

    # Strategy 1: Standard ETKDGv3
    ps = AllChem.ETKDGv3()
    ps.useRandomCoords = True
    ps.randomSeed = 42
    ps.maxIterations = 2000
    ps.numThreads = 1
    result = AllChem.EmbedMolecule(molH, ps)

    # Strategy 2: Disable pruning for macrocycles with large rings
    if result == -1:
        ps2 = AllChem.ETKDGv3()
        ps2.useRandomCoords = True
        ps2.randomSeed = 0
        ps2.maxIterations = 5000
        ps2.pruneRmsThresh = -1.0
        ps2.numThreads = 1
        result = AllChem.EmbedMolecule(molH, ps2)

    # Strategy 3: Distance geometry fallback
    if result == -1:
        try:
            result = AllChem.EmbedMolecule(molH, AllChem.ETKDG())
        except Exception:
            result = -1

    if result == -1:
        return out  # all zeros, missing_flag=-1

    try:
        AllChem.UFFOptimizeMolecule(molH, maxIters=500)
    except Exception:
        pass

    try:
        out[0] = finite(rdMolDescriptors.CalcNPR1(molH))
        out[1] = finite(rdMolDescriptors.CalcNPR2(molH))
        out[2] = finite(rdMolDescriptors.CalcAsphericity(molH))
        out[3] = finite(rdMolDescriptors.CalcEccentricity(molH))
        out[4] = finite(rdMolDescriptors.CalcInertialShapeFactor(molH))
        pmi = rdMolDescriptors.CalcPMI(molH)
        out[5] = finite(pmi[0])
        out[6] = finite(pmi[1])
        out[7] = finite(pmi[2])
        out[8] = finite(rdMolDescriptors.CalcSpherocityIndex(molH))
        out[9] = 0.0  # success
    except Exception:
        out[9] = -1.0

    return out

SHAPE_3D_NAMES = [
    "NPR1", "NPR2",
    "PMI1", "PMI2", "PMI3",
    "Asphericity", "Eccentricity",
    "RadiusOfGyration", "SpherocityIndex",
    "conformer_failed",
]


In [ ]:
# ── Block B: VSA Descriptors ──────────────────────────────────────────────────
# PEOE_VSA: surface area partitioned by Gasteiger partial charge bins  (14 bins)
# SlogP_VSA: surface area partitioned by Crippen logP contribution bins (10 bins)
# SMR_VSA:   surface area partitioned by molar refractivity bins        (10 bins)
# Research shows these are consistently top-retained by tree-based models [web:66]

def get_vsa_descriptors(smiles):
    """34 VSA features: PEOE_VSA1-14, SlogP_VSA1-10, SMR_VSA1-10."""
    out = np.zeros(34, dtype=float)
    if not isinstance(smiles, str) or not smiles.strip():
        return out
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return out

    for i in range(1, 15):
        fn = getattr(Descriptors, f"PEOE_VSA{i}", None)
        if fn: out[i - 1]      = _f(fn(mol))

    for i in range(1, 11):
        fn = getattr(Descriptors, f"SlogP_VSA{i}", None)
        if fn: out[13 + i]     = _f(fn(mol))   # indices 14–23

    for i in range(1, 11):
        fn = getattr(Descriptors, f"SMR_VSA{i}", None)
        if fn: out[23 + i]     = _f(fn(mol))   # indices 24–33

    return out

VSA_NAMES = (
    [f"PEOE_VSA{i}" for i in range(1, 15)] +
    [f"SlogP_VSA{i}" for i in range(1, 11)] +
    [f"SMR_VSA{i}"  for i in range(1, 11)]
)


In [ ]:
# ── Block C: Elemental Composition + Degree of Unsaturation ──────────────────
# DBE (Degree of Bond Equivalents / HDI) = (2C + 2 + N + P - H - halogens) / 2
# N/C and O/C ratios directly separate azolate (high-N) vs. carboxylate (high-O) MOFs
# Average MW per heavy atom normalizes size effects

def get_composition(smiles):
    """8 features: n_heavy, DBE, N/C, O/C, H/C, S_present, halogen_frac, MW_per_heavy."""
    out = np.zeros(8, dtype=float)
    if not isinstance(smiles, str) or not smiles.strip():
        return out
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return out

    counts = {}
    for atom in mol.GetAtoms():
        s = atom.GetSymbol()
        counts[s] = counts.get(s, 0) + 1

    C       = counts.get("C",  0)
    N       = counts.get("N",  0)
    O       = counts.get("O",  0)
    S       = counts.get("S",  0)
    P       = counts.get("P",  0)
    halogens = sum(counts.get(x, 0) for x in ("F", "Cl", "Br", "I"))

    mol_h = Chem.AddHs(mol)
    H = sum(1 for a in mol_h.GetAtoms() if a.GetSymbol() == "H")

    n_heavy = mol.GetNumHeavyAtoms()
    dbe     = (2*C + 2 + N + P - H - halogens) / 2.0

    out[0] = float(n_heavy)
    out[1] = max(0.0, _f(dbe))
    out[2] = N / C          if C > 0 else float(N)
    out[3] = O / C          if C > 0 else float(O)
    out[4] = H / C          if C > 0 else float(H)
    out[5] = float(S > 0)                                   # sulfur present?
    out[6] = halogens / n_heavy if n_heavy > 0 else 0.0
    out[7] = _f(Descriptors.MolWt(mol)) / n_heavy if n_heavy > 0 else 0.0
    return out

COMPOSITION_NAMES = [
    "n_heavy_atoms", "DBE",
    "N_over_C", "O_over_C", "H_over_C",
    "S_present", "halogen_frac", "MW_per_heavy",
]


In [ ]:
# ── Block D: MACCS Keys + RDKit Fragment Counts ───────────────────────────────
# MACCS: 166 fixed public keys — interpretable, ~20 features reach 90% SHAP [web:66]
# fr_*:  15 curated fragment counts with direct MOF-synthesis relevance

from rdkit.Chem import MACCSkeys, Fragments
from rdkit import DataStructs

def get_maccs(smiles):
    """167-element MACCS key bit vector (bit 0 always 0 — placeholder)."""
    out = np.zeros(167, dtype=float)
    if not isinstance(smiles, str) or not smiles.strip():
        return out
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return out
    DataStructs.ConvertToNumpyArray(MACCSkeys.GenMACCSKeys(mol), out)
    return out

MACCS_NAMES = [f"MACCS_{i}" for i in range(167)]


_SELECTED_FRAGS = {
    "fr_COO":             Fragments.fr_COO,
    "fr_pyridine":        Fragments.fr_pyridine,
    "fr_benzene":         Fragments.fr_benzene,
    "fr_ether":           Fragments.fr_ether,
    "fr_ester":           Fragments.fr_ester,
    "fr_amide":           Fragments.fr_amide,
    "fr_NH0":             Fragments.fr_NH0,
    "fr_NH1":             Fragments.fr_NH1,
    "fr_NH2":             Fragments.fr_NH2,
    "fr_Ar_NH":           Fragments.fr_Ar_NH,
    "fr_phenol":          Fragments.fr_phenol,
    "fr_imide":           Fragments.fr_imide,
    "fr_sulfone":         Fragments.fr_sulfone,
    "fr_nitro":           Fragments.fr_nitro,
    "fr_urea":            Fragments.fr_urea,
}

def get_key_fragments(smiles):
    """15 RDKit fr_* fragment counts for MOF-relevant functional groups."""
    out = np.zeros(len(_SELECTED_FRAGS), dtype=float)
    if not isinstance(smiles, str) or not smiles.strip():
        return out
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return out
    for i, fn in enumerate(_SELECTED_FRAGS.values()):
        try:
            out[i] = float(fn(mol))
        except Exception:
            pass
    return out

FRAGMENT_NAMES = list(_SELECTED_FRAGS.keys())


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Compute all ChemBERTa blocks for Linker and Modulator
# Covers: BERT (384) + extRDKit (50) + 3D shape (10) +
#         VSA (34) + Composition (8) + MACCS (167) + Fragments (15) = 668/mol
# ─────────────────────────────────────────────────────────────────────────────

_PER_MOL_DIM  = BERT_DIM + _N_TOTAL + 10 + 34 + 8 + 167 + 15  # = 668
_ROLES        = [("linker", linker_col), ("mod", mod_col)]

# Accumulate feature names across both roles — mirrors pattern used in
# per-ligand RAC / TEP / sterics blocks for SHAP grouping compatibility
chemberta_feature_names_all = []

for role, col in _ROLES:
    smi_list = df_merged[col].tolist()
    n        = len(smi_list)
    print(f"\n{'─'*60}")
    print(f"  {role.upper()}  →  column: {col}  ({n} rows)")
    print(f"{'─'*60}")

    # ── (a) ChemBERTa-2 MTR embeddings ───────────────────────────────────────
    print("  [1/7] ChemBERTa-2 embeddings …")
    bert  = chemberta_batch(smi_list)                    # (n, 384)
    print(f"        shape={bert.shape}  "
          f"non-finite={int((~np.isfinite(bert)).sum())}")

    # ── (b) Extended RDKit (complexity, shape, charge, coord sites, SMARTS) ──
    print("  [2/7] Extended RDKit descriptors …")
    extrd = np.stack([get_ext_rdkit(s) for s in smi_list])  # (n, 50)
    print(f"        shape={extrd.shape}  "
          f"missing_flags={int(extrd[:, -1].sum())}")

    # ── (c) 3D conformer shape (NPR, PMI, Asphericity …) ─────────────────────
    print("  [3/7] 3D shape descriptors (ETKDGv3) …")
    shape = np.stack([get_3d_shape(s) for s in smi_list])   # (n, 10)
    print(f"        shape={shape.shape}  "
          f"conformer_failures={int(shape[:, -1].sum())}/{n}")

    # ── (d) VSA (PEOE / SlogP / SMR surface area partitions) ─────────────────
    print("  [4/7] VSA descriptors …")
    vsa   = np.stack([get_vsa_descriptors(s) for s in smi_list])  # (n, 34)
    print(f"        shape={vsa.shape}")

    # ── (e) Elemental composition (DBE, N/C, O/C …) ──────────────────────────
    print("  [5/7] Elemental composition …")
    comp  = np.stack([get_composition(s) for s in smi_list])   # (n, 8)
    print(f"        shape={comp.shape}")

    # ── (f) MACCS keys ────────────────────────────────────────────────────────
    print("  [6/7] MACCS keys …")
    maccs = np.stack([get_maccs(s) for s in smi_list])         # (n, 167)
    print(f"        shape={maccs.shape}")

    # ── (g) RDKit fragment counts (fr_*) ──────────────────────────────────────
    print("  [7/7] Fragment counts …")
    frags = np.stack([get_key_fragments(s) for s in smi_list]) # (n, 15)
    print(f"        shape={frags.shape}")

    # ── Stack + sanity check ──────────────────────────────────────────────────
    block = np.hstack([bert, extrd, shape, vsa, comp, maccs, frags])
    assert block.shape == (n, _PER_MOL_DIM), \
        f"Shape mismatch for {role}: got {block.shape}, expected ({n}, {_PER_MOL_DIM})"
    print(f"\n  ✓ {role} block: {block.shape}  "
          f"(384+50+10+34+8+167+15 = {_PER_MOL_DIM})")

    # ── Append to X_final ──────────────────────────────────────────────────────
    X_final = np.concatenate([X_final, block], axis=1)

    # ── Feature names (SHAP-compatible, role-prefixed) ────────────────────────
    role_names = (
        [f"{role}_bert_{i}"     for i in range(BERT_DIM)]           # 384
        + [f"{role}_{n}"        for n in _EXT_RDKIT_BASE_NAMES]     # 50
        + [f"{role}_{n}"        for n in SHAPE_3D_NAMES]            # 10
        + [f"{role}_{n}"        for n in VSA_NAMES]                  # 34
        + [f"{role}_{n}"        for n in COMPOSITION_NAMES]         # 8
        + [f"{role}_{n}"        for n in MACCS_NAMES]                # 167
        + [f"{role}_{n}"        for n in FRAGMENT_NAMES]            # 15
    )
    assert len(role_names) == _PER_MOL_DIM, \
        f"Name count mismatch for {role}: {len(role_names)} vs {_PER_MOL_DIM}"
    chemberta_feature_names_all.extend(role_names)

# ─────────────────────────────────────────────────────────────────────────────
# Final finiteness check (outside loop — checks full X_final)
# ─────────────────────────────────────────────────────────────────────────────
bad = ~np.isfinite(X_final)
if bad.any():
    print(f"\nWARNING: {bad.sum()} non-finite entries in X_final → replacing with 0.0")
    X_final[bad] = 0.0
else:
    print("\nX_final all finite ✓")

print(f"\nX_final shape after full ChemBERTa block : {X_final.shape}")
print(f"Total features added                    : {_PER_MOL_DIM * len(_ROLES)}")
print(f"Feature names tracked                   : {len(chemberta_feature_names_all)}")


#Characterize metal center of the linkers

In [ ]:
from rdkit import Chem
from rdkit.Chem import rdmolops
import numpy as np
from collections import deque

# ── Tables ───────────────────────────────────────────────────────────────────
GROUP14_SYMBOLS  = {'Si', 'Ge', 'Sn', 'Pb'}
COORD_SYMBOLS    = {'P', 'N', 'O'}

G14_ENEG    = {'Si': 1.90, 'Ge': 2.01, 'Sn': 1.96, 'Pb': 2.33}   # Pauling
G14_COVRAD  = {'Si': 111,  'Ge': 122,  'Sn': 139,  'Pb': 146}    # pm
G14_PERIOD  = {'Si': 3,    'Ge': 4,    'Sn': 5,    'Pb': 6}


def _arms_from_hub(mol, hub_idx):
    """
    For each direct neighbor of hub_idx, trace a BFS arm until a
    coordination-site atom (P, N, O) is found.
    Returns list of (coord_symbol, bond_path_length, coord_atom_idx).
    One entry per arm that terminates at a coord site.
    """
    hub_atom = mol.GetAtomWithIdx(hub_idx)
    arms = []

    for root_nb in hub_atom.GetNeighbors():
        rn_idx = root_nb.GetIdx()
        rn_sym = root_nb.GetSymbol()

        # Direct neighbor is already a coord site (e.g. Si-P bond)
        if rn_sym in COORD_SYMBOLS:
            arms.append((rn_sym, 1, rn_idx))
            continue

        # BFS through this arm only — don't cross back through hub
        visited = {hub_idx, rn_idx}
        queue   = deque([(rn_idx, 1)])
        found   = None

        while queue:
            idx, depth = queue.popleft()
            sym = mol.GetAtomWithIdx(idx).GetSymbol()
            if sym in COORD_SYMBOLS:
                found = (sym, depth, idx)
                break                       # stop at first coord site in arm
            for nb in mol.GetAtomWithIdx(idx).GetNeighbors():
                nb_idx = nb.GetIdx()
                if nb_idx not in visited:
                    visited.add(nb_idx)
                    queue.append((nb_idx, depth + 1))

        if found:
            arms.append(found)

    return arms


def _path_composition(mol, hub_idx, target_idx):
    """
    Characterise the bond path from hub to target.
    Returns (has_aromatic, has_triple, has_only_sp3_single).
    """
    try:
        path = rdmolops.GetShortestPath(mol, hub_idx, target_idx)
    except Exception:
        return False, False, False

    n_arom, n_triple, n_sp3 = 0, 0, 0
    for i in range(len(path) - 1):
        bond = mol.GetBondBetweenAtoms(path[i], path[i + 1])
        if bond.GetIsAromatic():
            n_arom += 1
        bt = bond.GetBondTypeAsDouble()
        if bt == 3.0:
            n_triple += 1
        if bt == 1.0 and not bond.GetIsAromatic() and not bond.IsInRing():
            n_sp3 += 1

    has_aromatic = n_arom > 0
    has_triple   = n_triple > 0
    # "alkyl" arm: only sp3 single bonds, no aromatic, no triple
    only_sp3     = (n_arom == 0 and n_triple == 0 and n_sp3 > 0)
    return has_aromatic, has_triple, only_sp3


def get_g14_hub_topology(smiles: str) -> np.ndarray:
    """
    25 features encoding the G14 hub and its arm topology.

    idx  name                  meaning
    ---  --------------------  -------------------------------------------------
    0    hub_present           1 if a G14 atom exists in molecule
    1    hub_degree            bonds on hub (degree 4 → tetratopic)
    2    hub_nCoordArms        arms reaching any coord site (topicity)
    3    hub_nP_arms           arms reaching a P atom
    4    hub_nN_arms           arms reaching an N atom
    5    hub_nO_arms           arms reaching an O atom
    6    hub_armLen_min        shortest arm (bonds, hub → coord site)
    7    hub_armLen_max        longest arm
    8    hub_armLen_mean       mean arm length
    9    hub_armLen_std        std of arm lengths (0 = perfectly symmetric)
    10   hub_armFrac_alkynyl   fraction of arms containing a triple bond
    11   hub_armFrac_aromatic  fraction of arms with ≥1 aromatic bond
    12   hub_armFrac_alkyl     fraction of arms with only sp3 single bonds
    13   hub_eccentricity      max graph distance from hub to any atom
    14   hub_centrality        1/eccentricity (high → hub is topological center)
    15   hub_elem_eneg         Pauling electronegativity of hub element
    16   hub_elem_covrad_pm    covalent radius (pm) — sets binding-site spacing
    17   hub_elem_period       row in periodic table (3=Si, 4=Ge, 5=Sn, 6=Pb)
    18   hub_isSi              OHE identity flags
    19   hub_isGe
    20   hub_isSn
    21   hub_isPb
    22   hub_nG14total         total G14 heavy atoms in molecule
    23   hub_fracG14           fraction of heavy atoms that are G14
    24   hub_missing           1 if SMILES invalid or G14 not found
    """
    out = np.zeros(25, dtype=float)

    if not isinstance(smiles, str) or not smiles.strip():
        out[24] = 1.0
        return out

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        out[24] = 1.0
        return out

    g14_atoms = [a for a in mol.GetAtoms() if a.GetSymbol() in GROUP14_SYMBOLS]

    if not g14_atoms:
        return out  # all zeros, no missing flag — molecule simply has no G14

    out[0] = 1.0

    # Total G14 count / fraction
    out[22] = float(len(g14_atoms))
    n_heavy  = mol.GetNumHeavyAtoms()
    out[23]  = len(g14_atoms) / n_heavy if n_heavy > 0 else 0.0

    # ── Hub selection: highest-degree G14 atom (most arms) ─────────────────
    hub      = max(g14_atoms, key=lambda a: a.GetDegree())
    hub_idx  = hub.GetIdx()
    hub_sym  = hub.GetSymbol()

    out[1]  = float(hub.GetDegree())
    out[15] = G14_ENEG.get(hub_sym, 0.0)
    out[16] = float(G14_COVRAD.get(hub_sym, 0))
    out[17] = float(G14_PERIOD.get(hub_sym, 0))
    for i, sym in enumerate(['Si', 'Ge', 'Sn', 'Pb']):
        out[18 + i] = float(hub_sym == sym)

    # ── Graph eccentricity of hub ──────────────────────────────────────────
    try:
        dist_row = rdmolops.GetDistanceMatrix(mol)[hub_idx]
        ecc      = float(np.max(dist_row))
        out[13]  = ecc
        out[14]  = 1.0 / ecc if ecc > 0 else 0.0
    except Exception:
        pass

    # ── Arm topology ──────────────────────────────────────────────────────
    arms = _arms_from_hub(mol, hub_idx)

    if arms:
        out[2] = float(len(arms))
        out[3] = float(sum(1 for sym, _, _ in arms if sym == 'P'))
        out[4] = float(sum(1 for sym, _, _ in arms if sym == 'N'))
        out[5] = float(sum(1 for sym, _, _ in arms if sym == 'O'))

        lengths = [ln for _, ln, _ in arms]
        out[6] = float(min(lengths))
        out[7] = float(max(lengths))
        out[8] = float(np.mean(lengths))
        out[9] = float(np.std(lengths))

        # Arm backbone composition
        n_alkynyl = n_aromatic = n_alkyl = 0
        for _, _, coord_idx in arms:
            has_arom, has_triple, only_sp3 = _path_composition(
                mol, hub_idx, coord_idx
            )
            n_alkynyl  += int(has_triple)
            n_aromatic += int(has_arom)
            n_alkyl    += int(only_sp3)

        n_arms     = len(arms)
        out[10] = n_alkynyl  / n_arms
        out[11] = n_aromatic / n_arms
        out[12] = n_alkyl    / n_arms

    return out


G14_HUB_NAMES = [
    'g14hub_present',
    'g14hub_degree',
    'g14hub_nCoordArms',
    'g14hub_nP_arms',
    'g14hub_nN_arms',
    'g14hub_nO_arms',
    'g14hub_armLen_min',
    'g14hub_armLen_max',
    'g14hub_armLen_mean',
    'g14hub_armLen_std',
    'g14hub_armFrac_alkynyl',
    'g14hub_armFrac_aromatic',
    'g14hub_armFrac_alkyl',
    'g14hub_eccentricity',
    'g14hub_centrality',
    'g14hub_elem_eneg',
    'g14hub_elem_covrad_pm',
    'g14hub_elem_period',
    'g14hub_isSi',
    'g14hub_isGe',
    'g14hub_isSn',
    'g14hub_isPb',
    'g14hub_nG14total',
    'g14hub_fracG14',
    'g14hub_missing',
]
assert len(G14_HUB_NAMES) == 25


# ── Apply to linker and modulator ─────────────────────────────────────────────
X_linker_g14hub = np.stack(
    df_merged[linker_col].apply(get_g14_hub_topology).values
)  # (756, 25)

X_mod_g14hub = np.stack(
    df_merged[mod_col].apply(get_g14_hub_topology).values
)  # (756, 25)

X_final = np.concatenate([X_final, X_linker_g14hub, X_mod_g14hub], axis=1)
g14_feature_names = (
    [f'linker_{n}' for n in G14_HUB_NAMES] +
    [f'mod_{n}'    for n in G14_HUB_NAMES]
)
print(f"G14 hub topology block: {X_linker_g14hub.shape[1]*2} features added")
print(f"X_final shape: {X_final.shape}")


In [ ]:
from rdkit import Chem
import numpy as np

# ── G14 SMARTS patterns ────────────────────────────────────────────────────────
# These slot directly into the same pattern as your existing SMARTS_RAW dict.
# Match COUNT is the feature value (not just presence/absence).

G14_SMARTS_RAW = {

    # ── Hub valence (distinguishes tetrahedral vs. hypervalent center) ────────
    'g14_Si_X4':          '[Si;X4]',            # tetrahedral Si (most linkers)
    'g14_Ge_X4':          '[Ge;X4]',            # tetrahedral Ge
    'g14_Sn_X4':          '[Sn;X4]',            # tetrahedral Sn
    'g14_Sn_X6':          '[Sn;X6]',            # hypervalent Sn (rare, but real)

    # ── G14 → P arm distance ladder ───────────────────────────────────────────
    # Count = number of P atoms at exactly that bond distance from a G14 center.
    # Together these form a "distance fingerprint" of the arm length.
    # A tetratopic Si(CH2CH2P)4 gives dist3=4, all others=0.
    'g14_P_dist1':         '[Si,Ge,Sn]~[P]',
    'g14_P_dist2':         '[Si,Ge,Sn]~*~[P]',
    'g14_P_dist3':         '[Si,Ge,Sn]~*~*~[P]',
    'g14_P_dist4':         '[Si,Ge,Sn]~*~*~*~[P]',
    'g14_P_dist5':         '[Si,Ge,Sn]~*~*~*~*~[P]',
    'g14_P_dist6':         '[Si,Ge,Sn]~*~*~*~*~*~[P]',

    # ── Arm backbone type ─────────────────────────────────────────────────────
    # Encodes rigidity: alkynyl and aryl give rigid rods; alkyl gives flexibility.
    # All of these are critical for MOF pore geometry and crystallization.
    'g14_arm_alkynyl':     '[Si,Ge,Sn]~[#6]#[#6]',   # C≡C directly on G14
    'g14_arm_alkynyl_ext': '[Si,Ge,Sn]~*~[#6]#[#6]', # C≡C one bond away
    'g14_arm_aryl':        '[Si,Ge,Sn]~c1ccccc1',     # phenyl directly on G14
    'g14_arm_vinyl':       '[Si,Ge,Sn]~[#6]=[#6]',    # vinyl arm (flexible-rigid)
    'g14_arm_alkyl_CH2':   '[Si,Ge,Sn]~[CH2]',        # methylene directly on G14

    # ── Full arm: G14 → backbone → P (most informative patterns) ─────────────
    # These directly encode "G14 hub connected to P via a specific backbone".
    # A single count > 0 tells the model exactly what type of linker this is.
    'g14_Si_alkynyl_P':    '[Si]~[#6]#[#6]~[P]',       # Si-C≡C-P (short alkynyl)
    'g14_Si_alkynyl_P2':   '[Si]~*~[#6]#[#6]~[P]',     # Si-?-C≡C-P
    'g14_Si_alkynyl_P3':   '[Si]~[#6]#[#6]~*~[P]',     # Si-C≡C-?-P
    'g14_Si_aryl_P':       '[Si]~c1ccc([P])cc1',        # Si-p-Ph-P (para)
    'g14_Si_CH2CH2_P':     '[Si]~[CH2]~[CH2]~[P]',     # Si-CH2CH2-P (ethyl)
    'g14_Ge_aryl_P':       '[Ge]~c1ccc([P])cc1',        # Ge-p-Ph-P
    'g14_Sn_aryl_P':       '[Sn]~c1ccc([P])cc1',        # Sn-p-Ph-P

    # ── Topicity: how many P arms radiate from ONE G14 center ─────────────────
    # IMPORTANT: these count matches on the hub atom, so count=1 means
    # the pattern is satisfied (regardless of how many total G14 atoms exist).
    # A tetratopic linker matches g14_tetratopic_PP2 exactly once.
    'g14_ditopic_PP':      '[P]~*~[Si,Ge,Sn](~*~[P])',
    'g14_tritopic_PPP':    '[P]~*~[Si,Ge,Sn](~*~[P])(~*~[P])',
    'g14_tetratopic_PPPP': '[P]~*~[Si,Ge,Sn](~*~[P])(~*~[P])~*~[P]',

    # ── N-coordination variants (for pyridyl/amine-armed linkers) ─────────────
    'g14_N_dist2':         '[Si,Ge,Sn]~*~[n,N]',
    'g14_N_dist3':         '[Si,Ge,Sn]~*~*~[n,N]',
    'g14_tetratopic_NNNN': '[n,N]~*~[Si,Ge,Sn](~*~[n,N])(~*~[n,N])~*~[n,N]',
}

# Pre-compile once — same pattern as your existing SMARTS dict
G14_SMARTS = {k: Chem.MolFromSmarts(v) for k, v in G14_SMARTS_RAW.items()}
N_G14_SMARTS = len(G14_SMARTS)
G14_SMARTS_NAMES = list(G14_SMARTS_RAW.keys())


def get_g14_smarts_features(smiles: str) -> np.ndarray:
    """
    Returns match counts for each G14 SMARTS pattern + 1 missing flag.
    Total length: N_G14_SMARTS + 1
    """
    out = np.zeros(N_G14_SMARTS + 1, dtype=float)
    missing_idx = N_G14_SMARTS

    if not isinstance(smiles, str) or not smiles.strip():
        out[missing_idx] = 1.0
        return out

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        out[missing_idx] = 1.0
        return out

    for i, (name, patt) in enumerate(G14_SMARTS.items()):
        if patt is not None:
            out[i] = float(len(mol.GetSubstructMatches(patt)))

    return out


ALL_G14_SMARTS_NAMES = G14_SMARTS_NAMES + ['g14_smarts_missing']


# ── Apply ─────────────────────────────────────────────────────────────────────
X_linker_g14s = np.stack(
    df_merged[linker_col].apply(get_g14_smarts_features).values
)
X_mod_g14s = np.stack(
    df_merged[mod_col].apply(get_g14_smarts_features).values
)

X_final = np.concatenate([X_final, X_linker_g14s, X_mod_g14s], axis=1)

g14_smarts_feature_names = (
    [f'linker_{n}' for n in ALL_G14_SMARTS_NAMES] +
    [f'mod_{n}'    for n in ALL_G14_SMARTS_NAMES]
)
print(f"G14 SMARTS block: {X_linker_g14s.shape[1] * 2} features")
print(f"X_final shape: {X_final.shape}")


In [ ]:
from collections import deque
from rdkit.Chem import rdmolops

# Hub element reference properties
_HUB_PROPS = {
    'Si': {'eneg': 1.90, 'cov_rad': 111, 'period': 3},
    'Ge': {'eneg': 2.01, 'cov_rad': 122, 'period': 4},
    'Sn': {'eneg': 1.96, 'cov_rad': 139, 'period': 5},
    'C':  {'eneg': 2.55, 'cov_rad': 77,  'period': 2},
}
_G14_SYMS = {'Si', 'Ge', 'Sn'}

# Precompiled SMARTS for structural motifs
_MOTIF_SMARTS = {
    'biphenyl':        Chem.MolFromSmarts('c1ccccc1-c1ccccc1'),
    'spirobifluorene': Chem.MolFromSmarts('C12(c3ccccc31)c1ccccc12'),
    'adamantane':      Chem.MolFromSmarts('C1C2CC3CC1CC(C2)C3'),
}

TTP_DIM = 52

TTP_FEATURE_NAMES = [
    # Hub identity
    'ttp_hub_present',   # [0]  1 if G14 or quat-C hub found
    'ttp_hub_isSi',      # [1]
    'ttp_hub_isGe',      # [2]
    'ttp_hub_isSn',      # [3]
    'ttp_hub_isC',       # [4]  quaternary carbon hub (e.g. methane-based linker)
    'ttp_hub_eneg',      # [5]  Pauling electronegativity
    'ttp_hub_covrad_pm', # [6]  covalent radius (pm)
    'ttp_hub_period',    # [7]  row in periodic table
    'ttp_hub_degree',    # [8]  number of bonds on hub
    # Arm topology
    'ttp_nP_arms',       # [9]  number of P-reaching arms
    'ttp_n_unique_arm_types', # [10]
    'ttp_armLen_min',    # [11] shortest hub→P bond path
    'ttp_armLen_max',    # [12]
    'ttp_armLen_mean',   # [13]
    'ttp_armLen_std',    # [14]
    'ttp_frac_aryl',     # [15] fraction of arms that are aryl (hub–Ph–P)
    'ttp_frac_alkynyl',  # [16] fraction alkynyl (hub–C≡C–P)
    'ttp_frac_alkyl',    # [17] fraction alkyl (hub–CH2–P)
    'ttp_frac_vinyl',    # [18] fraction vinyl
    # Topicity
    'ttp_topicity_2',    # [19]
    'ttp_topicity_3',    # [20]
    'ttp_topicity_4',    # [21]
    # Symmetry
    'ttp_is_symmetric',  # [22] 1 if all arms same length
    # Hub graph properties
    'ttp_hub_eccentricity',  # [23]
    'ttp_hub_centrality',    # [24] 1/eccentricity
    # Global molecular descriptors (no Gasteiger, no 3D needed)
    'ttp_n_total_P',     # [25]
    'ttp_n_heavy',       # [26]
    'ttp_MolWt',         # [27]
    'ttp_n_arom_rings',  # [28]
    'ttp_n_rot_bonds',   # [29]
    'ttp_n_rings',       # [30]
    'ttp_frac_arom_heavy', # [31]
    'ttp_TPSA',          # [32]
    'ttp_MolLogP',       # [33]
    'ttp_HallKierAlpha', # [34]
    # Structural motif flags
    'ttp_has_biphenyl',       # [35] extended arm spacer (Si2/Sn2 type)
    'ttp_has_terphenyl',      # [36] proxy: >=6 aromatic rings total
    'ttp_has_spirobifluorene',# [37]
    'ttp_has_adamantane',     # [38]
    # Hub-to-P distance ladder (how many P atoms at each bond distance)
    'ttp_Pdist_1',  # [39]
    'ttp_Pdist_2',  # [40]
    'ttp_Pdist_3',  # [41]
    'ttp_Pdist_4',  # [42] — Si1/Sn1/Ge1 aryl arm: hub-C_ar-C_ar-C_ar-P ≈ 4 bonds
    'ttp_Pdist_5',  # [43]
    'ttp_Pdist_6',  # [44]
    'ttp_Pdist_7',  # [45]
    'ttp_Pdist_8',  # [46] — extended alkynyl arms reach here
    # Diagnostics
    'ttp_missing',    # [47] 1 = parse failure
    'ttp_n_G14',      # [48] total G14 atoms
    'ttp_frac_P_heavy', # [49] P atoms / heavy atoms
    'ttp_arm_PhP_count', # [50] number of Ph2P-aryl arms (diphenylphosphinophenyl)
    'ttp_n_PPh2_groups', # [51] total PPh2 groups in molecule
]
assert len(TTP_FEATURE_NAMES) == TTP_DIM


def _arm_backbone_type(mol, hub_idx, p_idx):
    """Classify hub→P arm as aryl/alkynyl/alkyl/vinyl."""
    try:
        path = list(rdmolops.GetShortestPath(mol, hub_idx, p_idx))
        has_triple = has_arom = False
        all_sp3_single = True
        for i in range(len(path) - 1):
            bond = mol.GetBondBetweenAtoms(path[i], path[i+1])
            if bond.GetBondTypeAsDouble() == 3.0:
                has_triple = True
                all_sp3_single = False
            if bond.GetIsAromatic():
                has_arom = True
                all_sp3_single = False
            if bond.GetBondTypeAsDouble() != 1.0 and not bond.GetIsAromatic():
                all_sp3_single = False
        if has_triple:   return 'alkynyl'
        if has_arom:     return 'aryl'
        if all_sp3_single: return 'alkyl'
        return 'vinyl'
    except Exception:
        return 'unknown'


def _arms_from_hub(mol, hub_idx):
    """BFS from hub: return list of (path_len, P_idx) for each P-containing arm."""
    hub_atom = mol.GetAtomWithIdx(hub_idx)
    arms = []
    for root_nb in hub_atom.GetNeighbors():
        rn_idx = root_nb.GetIdx()
        visited = {hub_idx, rn_idx}
        queue = deque([(rn_idx, 1)])
        found = None
        while queue:
            idx, depth = queue.popleft()
            if mol.GetAtomWithIdx(idx).GetSymbol() == 'P':
                found = (depth, idx)
                break
            for nb in mol.GetAtomWithIdx(idx).GetNeighbors():
                nb_idx = nb.GetIdx()
                if nb_idx not in visited:
                    visited.add(nb_idx)
                    queue.append((nb_idx, depth + 1))
        if found:
            arms.append(found)
    return arms


def _count_PPh2_groups(mol):
    """Count PPh2 groups: P connected to >=2 phenyl rings."""
    patt = Chem.MolFromSmarts('P(c1ccccc1)c1ccccc1')
    if patt is None:
        return 0
    return len(mol.GetSubstructMatches(patt))


def get_ttp_features(smiles: str) -> np.ndarray:
    """
    52-feature vector for tetratopic phosphine linkers.
    Reliable for Si/Ge/Sn/C hubs — uses only 2D descriptors,
    no Gasteiger charges, no 3D embedding.
    """
    out = np.zeros(TTP_DIM, dtype=float)

    if not isinstance(smiles, str) or not smiles.strip():
        out[47] = 1.0
        return out

    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        out[47] = 1.0
        return out

    # --- Identify hub atom ---
    g14_atoms = [a for a in mol.GetAtoms() if a.GetSymbol() in _G14_SYMS]

    if not g14_atoms:
        # Check for quaternary carbon hub (tetrakis-aryl methane type)
        for a in mol.GetAtoms():
            if a.GetSymbol() == 'C' and a.GetDegree() == 4 and not a.GetIsAromatic():
                test_arms = _arms_from_hub(mol, a.GetIdx())
                if len(test_arms) >= 3:
                    g14_atoms = [a]
                    break

    if not g14_atoms:
        return out   # Not this linker type — all zeros, no missing flag

    hub = max(g14_atoms, key=lambda a: a.GetDegree())
    hub_idx = hub.GetIdx()
    hub_sym = hub.GetSymbol()

    out[0] = 1.0
    for oi, sym in [(1,'Si'),(2,'Ge'),(3,'Sn'),(4,'C')]:
        if hub_sym == sym:
            out[oi] = 1.0

    hp = _HUB_PROPS.get(hub_sym, {'eneg':0.,'cov_rad':0,'period':0})
    out[5]  = hp['eneg']
    out[6]  = float(hp['cov_rad'])
    out[7]  = float(hp['period'])
    out[8]  = float(hub.GetDegree())

    # --- Arms ---
    arms = _arms_from_hub(mol, hub_idx)
    n_arms = len(arms)
    out[9] = float(n_arms)

    if arms:
        lengths  = [a[0] for a in arms]
        p_indices = [a[1] for a in arms]
        out[11] = float(min(lengths))
        out[12] = float(max(lengths))
        out[13] = float(np.mean(lengths))
        out[14] = float(np.std(lengths)) if n_arms > 1 else 0.0

        types = [_arm_backbone_type(mol, hub_idx, pi) for pi in p_indices]
        n = len(types)
        out[10] = float(len(set(t for t in types if t != 'unknown')))
        out[15] = sum(1 for t in types if t == 'aryl')    / n
        out[16] = sum(1 for t in types if t == 'alkynyl') / n
        out[17] = sum(1 for t in types if t == 'alkyl')   / n
        out[18] = sum(1 for t in types if t == 'vinyl')   / n

        if n_arms == 2: out[19] = 1.0
        elif n_arms == 3: out[20] = 1.0
        elif n_arms >= 4: out[21] = 1.0

        out[22] = 1.0 if len(set(lengths)) == 1 else 0.0

        for plen, _ in arms:
            if 1 <= plen <= 8:
                out[39 + plen - 1] += 1.0

    # --- Hub graph properties ---
    try:
        dist_row = rdmolops.GetDistanceMatrix(mol)[hub_idx]
        ecc = float(np.max(dist_row))
        out[23] = ecc
        out[24] = 1.0 / ecc if ecc > 0 else 0.0
    except Exception:
        pass

    # --- Global 2D descriptors ---
    n_heavy = mol.GetNumHeavyAtoms()
    out[25] = float(sum(1 for a in mol.GetAtoms() if a.GetSymbol() == 'P'))
    out[26] = float(n_heavy)
    try:
        out[27] = finite(Descriptors.MolWt(mol))
        out[28] = float(rdMolDescriptors.CalcNumAromaticRings(mol))
        out[29] = float(rdMolDescriptors.CalcNumRotatableBonds(mol))
        out[30] = float(rdMolDescriptors.CalcNumRings(mol))
        n_arom  = sum(1 for a in mol.GetAtoms()
                      if a.GetIsAromatic() and a.GetAtomicNum() > 1)
        out[31] = n_arom / n_heavy if n_heavy > 0 else 0.0
        out[32] = finite(Descriptors.TPSA(mol))
        out[33] = finite(Descriptors.MolLogP(mol))
        out[34] = finite(Descriptors.HallKierAlpha(mol))
    except Exception:
        pass

    # --- Structural motifs ---
    try:
        if _MOTIF_SMARTS['biphenyl'] and mol.HasSubstructMatch(_MOTIF_SMARTS['biphenyl']):
            out[35] = 1.0
        if out[28] >= 6:          # rough terphenyl proxy: ≥6 arene rings
            out[36] = 1.0
        if _MOTIF_SMARTS['spirobifluorene'] and mol.HasSubstructMatch(_MOTIF_SMARTS['spirobifluorene']):
            out[37] = 1.0
        if _MOTIF_SMARTS['adamantane'] and mol.HasSubstructMatch(_MOTIF_SMARTS['adamantane']):
            out[38] = 1.0
    except Exception:
        pass

    # --- Diagnostics ---
    out[48] = float(len(g14_atoms))
    out[49] = out[25] / n_heavy if n_heavy > 0 else 0.0
    try:
        n_pph2 = _count_PPh2_groups(mol)
        out[51] = float(n_pph2)
        # Ph2P-aryl arm: PPh2 group directly on an arene that connects to hub
        pph2_aryl_patt = Chem.MolFromSmarts('P(c1ccccc1)(c1ccccc1)c1ccccc1')
        out[50] = float(len(mol.GetSubstructMatches(pph2_aryl_patt))) if pph2_aryl_patt else 0.0
    except Exception:
        pass

    bad = ~np.isfinite(out)
    if bad.any():
        out[bad] = 0.0

    return out


# --- Apply to linker column ---
_linker_col = next(
    (c for c in ['smiles_linker_1_feat', 'smiles_linker_1_canon', 'smiles_linker_1']
     if c in df_merged.columns), None
)
if _linker_col is None:
    raise KeyError("No linker SMILES column found in df_merged.")


Xlinker_ttp = np.stack(df_merged[_linker_col].apply(get_ttp_features).values)


print(f"Tetratopic phosphine features: {Xlinker_ttp.shape}")
print(f"  Hub detected: {int((Xlinker_ttp[:,0]>0).sum())}/{len(df_merged)} linkers")
print(f"  Si hubs: {int(Xlinker_ttp[:,1].sum())}  "
      f"Ge: {int(Xlinker_ttp[:,2].sum())}  "
      f"Sn: {int(Xlinker_ttp[:,3].sum())}  "
      f"C:  {int(Xlinker_ttp[:,4].sum())}")
print(f"  Tetratopic (4-arm): {int(Xlinker_ttp[:,21].sum())}")
print(f"  Missing flag:       {int(Xlinker_ttp[:,47].sum())}")

bad = ~np.isfinite(Xlinker_ttp)
if bad.any():
    print(f"WARNING: {bad.sum()} non-finite → replaced with 0.0")
    Xlinker_ttp[bad] = 0.0
else:
    print("  All finite ✓")

# Append to X_final
X_final = np.concatenate([X_final, Xlinker_ttp], axis=1)
print(f"X_final shape after TTP block: {X_final.shape}")

bad_final = ~np.isfinite(X_final)
print(f"Non-finite entries in X_final: {int(bad_final.sum())}")

In [ ]:
# ── TTP Block Verification ────────────────────────────────────────────────────
import pandas as pd

# Step 1: Confirm the column was found and has real SMILES
print(f"Linker column used: {_linker_col}")
print(f"Sample values:\n{df_merged[_linker_col].dropna().unique()[:3]}\n")

# Step 2: Build a per-unique-linker summary table
unique_linkers = df_merged[[_linker_col]].drop_duplicates().copy()
unique_linkers.columns = ['smiles']
ttp_unique = np.stack(unique_linkers['smiles'].apply(get_ttp_features).values)

summary = pd.DataFrame({
    'smiles':          unique_linkers['smiles'].values,
    'hub_present':     ttp_unique[:, 0].astype(int),
    'hub_type':        [
                           'Si' if r[1] else 'Ge' if r[2] else
                           'Sn' if r[3] else 'C'  if r[4] else 'none'
                           for r in ttp_unique
                       ],
    'n_P_arms':        ttp_unique[:, 9].astype(int),
    'arm_len_min':     ttp_unique[:, 11],
    'arm_len_max':     ttp_unique[:, 12],
    'frac_aryl':       ttp_unique[:, 15].round(2),
    'frac_alkynyl':    ttp_unique[:, 16].round(2),
    'frac_alkyl':      ttp_unique[:, 17].round(2),
    'topicity_2':      ttp_unique[:, 19].astype(int),
    'topicity_3':      ttp_unique[:, 20].astype(int),
    'topicity_4':      ttp_unique[:, 21].astype(int),
    'n_PPh2_groups':   ttp_unique[:, 51].astype(int),
    'missing_flag':    ttp_unique[:, 47].astype(int),
})
print("Per-unique-linker TTP summary:")
display(summary.to_string(index=False))

# Step 3: Chemical sanity checks — things that must be true
print("\n── Sanity checks ──")

# All linkers with hub_present=1 must have n_P_arms >= 1
hub_rows = ttp_unique[ttp_unique[:, 0] > 0]
assert (hub_rows[:, 9] >= 1).all(), \
    "FAIL: hub detected but no P arms found"
print(f"✓ All {len(hub_rows)} hub-containing linkers have ≥1 P arm")

# Fraction columns must sum to ≤1.0
frac_sum = ttp_unique[:, 15] + ttp_unique[:, 16] + ttp_unique[:, 17] + ttp_unique[:, 18]
assert (frac_sum <= 1.001).all(), \
    f"FAIL: arm type fractions sum > 1: {frac_sum.max():.3f}"
print(f"✓ Arm type fractions sum ≤ 1.0 (max={frac_sum.max():.3f})")

# arm_len_min must be <= arm_len_max
armed = ttp_unique[ttp_unique[:, 9] > 0]
assert (armed[:, 11] <= armed[:, 12]).all(), \
    "FAIL: arm_len_min > arm_len_max"
print(f"✓ arm_len_min ≤ arm_len_max for all armed linkers")

# No NaN/inf
assert np.isfinite(Xlinker_ttp).all(), "FAIL: non-finite in Xlinker_ttp"
print(f"✓ All {Xlinker_ttp.shape} values finite")

# Step 4: Known-value spot check for your most common linker
# 1,4-bisdiphenylphosphaneylethynylbenzene = ditopic alkynyl, no G14 hub
# Expected: hub_present=0, missing=0, n_P_arms=0 (not a TTP linker)
test_smiles = df_merged[_linker_col].iloc[0]
test_vec = get_ttp_features(test_smiles)
print(f"\nSpot check — row 0 linker: {test_smiles[:60]}")
print(f"  hub_present : {test_vec[0]:.0f}  (0=not tetratopic, expected for bis-linkers)")
print(f"  missing_flag: {test_vec[47]:.0f}  (0=parsed OK)")
print(f"  n_P_arms    : {test_vec[9]:.0f}")
print(f"  n_PPh2      : {test_vec[51]:.0f}")

# Step 5: Variance check — constant columns will be pruned but warns you
n_const = (Xlinker_ttp.std(axis=0) == 0).sum()
print(f"\nConstant columns (will be pruned by VarianceThreshold): "
      f"{n_const}/{TTP_DIM}")
print(f"Active columns: {TTP_DIM - n_const}/{TTP_DIM}")


In [ ]:
#Linker Helpers
from rdkit.Chem.AtomPairs import Pairs, Torsions
from rdkit.Chem import MACCSkeys
from rdkit.Chem import rdMolDescriptors, Descriptors, GraphDescriptors
import numpy as np

def get_atom_pair_fp(smiles, n_bits=2048):
    """Hashed atom pair fingerprint — captures P···P through-bond distance."""
    out = np.zeros(n_bits, dtype=np.float32)
    if not isinstance(smiles, str) or not smiles.strip():
        return out
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return out
    try:
        fp = Pairs.GetHashedAtomPairFingerprintAsBitVect(mol, nBits=n_bits)
        DataStructs.ConvertToNumpyArray(fp, out)
    except Exception:
        pass
    return out


def get_maccs(smiles):
    out = np.zeros(167, dtype=np.float32)
    if not isinstance(smiles, str) or not smiles.strip():
        return out
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return out
    try:
        fp = MACCSkeys.GenMACCSKeys(mol)
        DataStructs.ConvertToNumpyArray(fp, out)
    except Exception:
        pass
    return out

def get_torsion_fp(smiles, n_bits=1024):
    out = np.zeros(n_bits, dtype=np.float32)
    if not isinstance(smiles, str) or not smiles.strip():
        return out
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return out
    try:
        fp = Torsions.GetHashedTopologicalTorsionFingerprintAsBitVect(
            mol, nBits=n_bits)
        DataStructs.ConvertToNumpyArray(fp, out)
    except Exception:
        pass
    return out


from rdkit.Chem import EState, GraphDescriptors

def get_graph_topo_descriptors(smiles):
    """
    14 descriptors: graph indices + EState summaries + flexibility metrics.
    Output: [Wiener, Ipc, BalabanJ, BertzCT, Chi0, Chi1, Chi2n, Chi3n,
             Kappa1, Kappa2, Kappa3, Fsp3, FractionCSP3, missing_flag]
    """
    out = np.zeros(14, dtype=float)
    if not isinstance(smiles, str) or not smiles.strip():
        out[13] = 1.0
        return out
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        out[13] = 1.0
        return out
    try:
        out[0]  = finite(GraphDescriptors.BalabanJ(mol))
        out[1]  = finite(GraphDescriptors.BertzCT(mol))
        out[2]  = finite(GraphDescriptors.Chi0(mol))
        out[3]  = finite(GraphDescriptors.Chi1(mol))
        out[4]  = finite(GraphDescriptors.Chi2n(mol))
        out[5]  = finite(GraphDescriptors.Chi3n(mol))
        out[6]  = finite(GraphDescriptors.Kappa1(mol))
        out[7]  = finite(GraphDescriptors.Kappa2(mol))
        out[8]  = finite(GraphDescriptors.Kappa3(mol))
        # Wiener index via distance matrix
        dm = rdmolops.GetDistanceMatrix(mol)
        out[9]  = float(dm.sum() / 2.0)
        # Flexibility
        n_heavy = mol.GetNumHeavyAtoms()
        n_sp3_c = sum(
            1 for a in mol.GetAtoms()
            if a.GetSymbol() == 'C'
            and a.GetHybridization().name == 'SP3'
        )
        out[10] = n_sp3_c / n_heavy if n_heavy > 0 else 0.0  # Fsp3
        out[11] = finite(rdMolDescriptors.CalcFractionCSP3(mol))
        # Labute ASA approximation
        out[12] = finite(rdMolDescriptors.CalcLabuteASA(mol))
        out[13] = 0.0
    except Exception:
        out[13] = 1.0
    bad = ~np.isfinite(out)
    out[bad] = 0.0
    return out



from rdkit.Chem.EState.Fingerprinter import FingerprintMol as EStateFP

def get_estate_fp(smiles):
    if not isinstance(smiles, str) or not smiles.strip():
        return np.zeros(79, dtype=float)
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros(79, dtype=float)
    try:
        maxs, sums = EStateFP(mol)
        out = np.array(sums, dtype=float)
        out = np.where(np.isfinite(out), out, 0.0)
        return out  # length 79
    except Exception:
        return np.zeros(79, dtype=float)

Xlinker_topo = np.stack(df_merged[_linker_col].apply(get_graph_topo_descriptors).values)
print(f"Graph topo descriptors: {Xlinker_topo.shape}")

Xlinker_tor = np.stack(df_merged[_linker_col].apply(get_torsion_fp).values)
print(f"Topological torsion FP: {Xlinker_tor.shape}")

Xlinker_ap = np.stack(df_merged[_linker_col].apply(get_atom_pair_fp).values)
print(f"Atom pair FP: {Xlinker_ap.shape}")

Xlinker_maccs = np.stack(df_merged[_linker_col].apply(get_maccs).values)
print(f"MACCS keys: {Xlinker_maccs.shape}")

Xlinker_estate = np.stack(df_merged[_linker_col].apply(get_estate_fp).values)
X_final = np.concatenate([X_final, Xlinker_estate, Xlinker_topo, Xlinker_tor, Xlinker_ap], axis=1)
print(f"EState FP: {Xlinker_estate.shape}")



In [ ]:
# HALIDE BLOCK v4 — reuses halide cols already in df_inventory

HALIDE_FEAT_COLS = ['halide_type', 'halide_count', 'halide_present',
                    'halide_I_count', 'halide_Br_count', 'halide_Cl_count']

# Re-compute cleanly from the raw Total cols
I_cols  = ['Total_I', 'Total_[I]',  'Total_[I-]']
Br_cols = ['Total_Br', 'Total_[Br-]']
Cl_cols = ['Total_Cl', 'Total_[Cl-]']

I_cols  = [c for c in I_cols  if c in df_inventory.columns]
Br_cols = [c for c in Br_cols if c in df_inventory.columns]
Cl_cols = [c for c in Cl_cols if c in df_inventory.columns]

print(f"I cols:  {I_cols}")
print(f"Br cols: {Br_cols}")
print(f"Cl cols: {Cl_cols}")

df_inventory['halide_I_count']  = df_inventory[I_cols].sum(axis=1)  if I_cols  else 0.0
df_inventory['halide_Br_count'] = df_inventory[Br_cols].sum(axis=1) if Br_cols else 0.0
df_inventory['halide_Cl_count'] = df_inventory[Cl_cols].sum(axis=1) if Cl_cols else 0.0
df_inventory['halide_count']    = (df_inventory['halide_I_count'] +
                                   df_inventory['halide_Br_count'] +
                                   df_inventory['halide_Cl_count'])
df_inventory['halide_present']  = (df_inventory['halide_count'] > 0).astype(float)

def _halide_type(row):
    if row['halide_I_count']  > 0: return 1.0
    if row['halide_Br_count'] > 0: return 2.0
    if row['halide_Cl_count'] > 0: return 3.0
    return 0.0

df_inventory['halide_type'] = df_inventory.apply(_halide_type, axis=1)

# Align to df_merged row order — NO [mask] here, X_final is still full 756 rows
Xhalide_full = (
    pd.DataFrame({'experiment_id': df_merged['experiment_id'].values})
    .merge(df_inventory[['experiment_id'] + HALIDE_FEAT_COLS],
           on='experiment_id', how='left')
    .fillna(0.0)[HALIDE_FEAT_COLS]
    .values.astype(float)
)

assert Xhalide_full.shape[0] == X_final.shape[0], \
    f"Row mismatch: Xhalide {Xhalide_full.shape[0]} vs X_final {X_final.shape[0]}"

X_final = np.concatenate([X_final, Xhalide_full], axis=1)

lbl = {0.0: 'none', 1.0: 'iodide', 2.0: 'bromide', 3.0: 'chloride'}
print(f"Halide block added {Xhalide_full.shape} → X_final now {X_final.shape}")
for val, name in lbl.items():
    print(f"  {name}: {int((Xhalide_full[:, 0] == val).sum())}")


In [ ]:
# ── Block 8: DRFP Reaction Fingerprint ────────────────────────────────────────
# Encodes cross-terms between precursor + linker + modulator SMILES.
# Uses RAW SMILES only — augmented features (cone angles, metal props)
# are NOT fed in here; they remain in their own blocks.

from drfp import DrfpEncoder

def make_rxn_smiles(row):
    """
    Reactant side: precursor + linker + modulator (whatever is present).
    Product side: empty — we are predicting the outcome, not encoding it.
    Prefers canonical SMILES; falls back to raw if canonical is unavailable.
    """
    parts = []
    for col in ['smiles_precursor', 'smiles_linker1', 'smiles_modulator']:
        s_can = row.get(f'{col}_canon')
        s_raw = row.get(col)
        val = s_can if pd.notna(s_can) and str(s_can).strip() else s_raw
        if pd.notna(val) and str(val).strip():
            parts.append(str(val).strip())
    return '.'.join(parts) + '>>' if parts else '>>'

rxn_smiles_list = df_merged.apply(make_rxn_smiles, axis=1).tolist()

n_empty = sum(1 for s in rxn_smiles_list if s == '>>')
print(f"DRFP: {len(rxn_smiles_list) - n_empty} valid reactions, "
      f"{n_empty} empty → zero vectors")

# Encode at radius=3 (DRFP default/validated) with ring membership
X_drfp = np.array(DrfpEncoder.encode(
    rxn_smiles_list,
    n_folded_length=2048,
    radius=3,
    rings=True
), dtype=np.float32)

# ── Sanity checks ──────────────────────────────────────────────────────────────
assert X_drfp.shape == (len(df_merged), 2048), \
    f"Unexpected DRFP shape: {X_drfp.shape}"
assert np.isfinite(X_drfp).all(), "Non-finite values in X_drfp"

# Diagnostic: check whether metal complex SMILES are contributing
# Low bit counts for a metal = precursor SMILES failing to parse in DRFP
bit_counts = X_drfp.sum(axis=1)
print(f"\nBit count stats — mean: {bit_counts.mean():.1f}, "
      f"min: {bit_counts.min():.0f}, max: {bit_counts.max():.0f}")
print("\nMean DRFP bits set per metal (low → precursor parsing failing):")
df_merged['drfp_bits_tmp'] = bit_counts
print(df_merged.groupby('metal_atom')['drfp_bits_tmp']
      .mean().sort_values().to_string())
df_merged.drop(columns=['drfp_bits_tmp'], inplace=True)

# ── Append to X_final ──────────────────────────────────────────────────────────
X_final = np.concatenate([X_final, X_drfp], axis=1)
drfp_feature_names = [f'drfp_{i}' for i in range(2048)]

print(f"\nX_final shape after DRFP block: {X_final.shape}")
print(f"Non-finite entries in X_final: {int(~np.isfinite(X_final).sum())}")


In [ ]:
!pip install ase
!pip install sparse

In [ ]:
# ── SOAP Block: 3D Descriptors for Precursor + Linker ─────────────────────────
# Replaces SLATM (qml not installable on Python 3.12).
# SOAP = Smooth Overlap of Atomic Positions — captures full 3D density
# around each atom, averaged to one vector per molecule.
# Applied to:
#   (A) precursor_sterics_smiles  (~6 unique)  — phosphine fragment
#   (B) linkercol                 (~22 unique) — linker fragment

from dscribe.descriptors import SOAP
from ase import Atoms
from rdkit import Chem
from rdkit.Chem import AllChem

# ── Shared element lookup ─────────────────────────────────────────────────────
ELEMENT_TO_Z = {
    'H':1,'C':6,'N':7,'O':8,'F':9,'P':15,'S':16,'Cl':17,
    'Br':35,'I':53,'Si':14,'Ge':32,'Sn':50,'B':5
}
Z_TO_ELEMENT = {v: k for k, v in ELEMENT_TO_Z.items()}

SOAP_SPECIES = list(ELEMENT_TO_Z.keys())  # all 14 elements
soap_species_set = set(SOAP_SPECIES)

# ── Build SOAP descriptor (shared between precursor and linker) ───────────────
soap = SOAP(
    species=SOAP_SPECIES,
    r_cut=6.0,
    n_max=8,
    l_max=6,
    average='outer',   # one vector per molecule
    sparse=False
)
SOAP_DIM = soap.get_number_of_features()
print(f"SOAP vector dimension: {SOAP_DIM:,}")

# ── 3D conformer generation (metal-free organic SMILES only) ──────────────────
def embed_organic_3d(smiles):
    """
    Multi-strategy ETKDGv3 + UFF embedding for organic (metal-free) SMILES.
    Returns (heavy_atom_coords, nuclear_charges) or (None, None) on failure.
    """
    if not isinstance(smiles, str) or not smiles.strip():
        return None, None
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None, None

        molH = Chem.AddHs(mol)

        # Strategy 1: Standard ETKDGv3
        ps = AllChem.ETKDGv3()
        ps.useRandomCoords = True
        ps.randomSeed = 42
        ps.maxIterations = 2000
        ps.numThreads = 1
        result = AllChem.EmbedMolecule(molH, ps)

        # Strategy 2: Disable RMSD pruning for large/macrocyclic molecules
        if result == -1:
            ps2 = AllChem.ETKDGv3()
            ps2.useRandomCoords = True
            ps2.randomSeed = 0
            ps2.maxIterations = 5000
            ps2.pruneRmsThresh = -1.0
            ps2.numThreads = 1
            result = AllChem.EmbedMolecule(molH, ps2)

        # Strategy 3: Classic ETKDG fallback
        if result == -1:
            result = AllChem.EmbedMolecule(molH, AllChem.ETKDG())

        if result == -1:
            return None, None

        try:
            AllChem.UFFOptimizeMolecule(molH, maxIters=500)
        except Exception:
            pass

        conf = molH.GetConformer()
        coords, charges = [], []
        for atom in molH.GetAtoms():
            if atom.GetSymbol() == 'H':
                continue  # heavy atoms only
            sym = atom.GetSymbol()
            charges.append(ELEMENT_TO_Z.get(sym, atom.GetAtomicNum()))
            coords.append(list(conf.GetAtomPosition(atom.GetIdx())))

        if not coords:
            return None, None

        return np.array(coords, dtype=float), np.array(charges, dtype=int)

    except Exception:
        return None, None


def run_soap_block(label, smiles_series):
    """
    Given a label (for printing) and a pandas Series of SMILES,
    returns (X_soap, feature_names) ready to append to X_final.
    """
    unique_smiles = [
        s for s in smiles_series.dropna().astype(str).map(str.strip).unique()
        if s
    ]
    print(f"\n── {label}: {len(unique_smiles)} unique structures ──")

    # Step 1: Generate conformers
    conform_cache = {}
    for smi in unique_smiles:
        coords, charges = embed_organic_3d(smi)
        if coords is not None and len(coords) >= 2:
            # Validate all elements are in SOAP_SPECIES
            unknown = {Z_TO_ELEMENT.get(z, f'Z={z}') for z in charges
                       if Z_TO_ELEMENT.get(z, f'Z={z}') not in soap_species_set}
            if unknown:
                print(f"  WARN {smi[:45]}: unknown elements {unknown} → zero vector")
                conform_cache[smi] = None
            else:
                conform_cache[smi] = (coords, charges)
                print(f"  OK   {smi[:55]} ({len(charges)} heavy atoms)")
        else:
            conform_cache[smi] = None
            print(f"  FAIL {smi[:55]}")

    n_ok = sum(1 for v in conform_cache.values() if v is not None)
    print(f"  Conformers: {n_ok}/{len(unique_smiles)} succeeded")

    # Step 2: Generate SOAP vectors
    vec_cache = {}
    for smi in unique_smiles:
        entry = conform_cache[smi]
        if entry is None:
            vec_cache[smi] = np.zeros(SOAP_DIM, dtype=np.float32)
            continue
        try:
            coords, charges = entry
            symbols = [Z_TO_ELEMENT[z] for z in charges]
            atoms = Atoms(symbols=symbols, positions=coords)
            vec = soap.create(atoms).astype(np.float32)
            vec = np.where(np.isfinite(vec), vec, 0.0)
            vec_cache[smi] = vec
        except Exception as e:
            print(f"  SOAP FAILED {smi[:45]}: {e}")
            vec_cache[smi] = np.zeros(SOAP_DIM, dtype=np.float32)

    # Step 3: Map to full dataset
    X = np.stack(smiles_series.apply(
        lambda s: vec_cache.get(
            str(s).strip() if pd.notna(s) else '',
            np.zeros(SOAP_DIM, dtype=np.float32)
        )
    ).values)

    n_zero = (X.sum(axis=1) == 0).sum()
    bad = ~np.isfinite(X)
    if bad.any():
        print(f"  WARNING: {bad.sum()} non-finite values → replacing with 0.0")
        X[bad] = 0.0
    print(f"  Output shape: {X.shape}  |  zero-vector rows: {n_zero}/{len(X)}")

    names = [f'soap_{label.lower()}_{i}' for i in range(SOAP_DIM)]
    return X, names


# ── (A) Precursor SOAP ────────────────────────────────────────────────────────
slatm_src_col = 'precursor_sterics_smiles'
X_soap_precursor, soap_precursor_names = run_soap_block(
    'precursor', df_merged[slatm_src_col]
)

X_final = np.concatenate([X_final, X_soap_precursor], axis=1)
print(f"X_final after precursor SOAP: {X_final.shape}")

# ── (B) Linker SOAP ───────────────────────────────────────────────────────────
X_soap_linker, soap_linker_names = run_soap_block(
    'linker', df_merged[linker_col]
)

X_final = np.concatenate([X_final, X_soap_linker], axis=1)
print(f"X_final after linker SOAP:    {X_final.shape}")
print(f"Non-finite in X_final: {int((~np.isfinite(X_final)).sum())}")

# ── Combined feature names for SHAP ──────────────────────────────────────────
soap_all_names = soap_precursor_names + soap_linker_names
# Append to your correct_feature_names list in the name-assembly cell:
# correct_feature_names += soap_all_names


#Dimensionality Control

In [ ]:
import numpy as np
import pandas as pd

X = np.asarray(X_final, dtype=float)
y = pd.to_numeric(df_merged["pxrd_score"], errors="coerce").to_numpy()

mask = np.isfinite(y)
X = X[mask]
y = y[mask]

y_high = (y >= 7).astype(int)

print("High (>=7) positives:", int(y_high.sum()), "/", len(y_high), "rate=", y_high.mean())
print("X:", X.shape, "y:", y.shape, "unique y:", np.unique(y))

print("Any non-finite in X?", (~np.isfinite(X)).any())


In [ ]:
from sklearn.feature_selection import VarianceThreshold
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import numpy as np

# ── Diagnostic KMeans + VT preview (does NOT modify X used downstream) ──
_N = 10  # match N_CLUSTERS used in the Ordinal cell

_sc  = StandardScaler()
_km  = KMeans(n_clusters=_N, random_state=42, n_init=10)
_ohe = OneHotEncoder(sparse_output=False, dtype=float)

_X_preview = np.hstack([
    X,
    _ohe.fit_transform(
        _km.fit_predict(_sc.fit_transform(X)).reshape(-1, 1)
    )
])  # diagnostic only — shape (n, 50684 + _N)

vt = VarianceThreshold(threshold=0.0)
X_vt = vt.fit_transform(_X_preview)

print("Before (with KMeans OHE):", _X_preview.shape[1],
      "After VarianceThreshold:", X_vt.shape[1],
      "Removed:", _X_preview.shape[1] - X_vt.shape[1])


#Ordinal

In [ ]:
import warnings
import numpy as np
import pandas as pd
import optuna
import matplotlib.pyplot as plt
import umap
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted
from sklearn.metrics import make_scorer
from sklearn.model_selection import (StratifiedGroupKFold, cross_validate,
                                     cross_val_score, cross_val_predict)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import (VarianceThreshold, SelectFromModel,
                                        SelectKBest, mutual_info_classif)
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
import colorlog

# Silence noisy but harmless warnings from silhouette_score + KMeans
warnings.filterwarnings(
    "ignore",
    message="Clustering metrics expects discrete values",
    category=UserWarning,
)


# ─────────────────────────────────────────────────────────────
# 1.  Ordinal Classifier
# ─────────────────────────────────────────────────────────────
class FrankHallOrdinalClassifier(BaseEstimator, ClassifierMixin):
    _estimator_type = "classifier"

    def __init__(self, base_estimator=None):
        self.base_estimator = base_estimator

    def fit(self, X, y):
        X, y = check_X_y(X, y)
        self.classes_     = np.sort(np.unique(y))
        self.classifiers_ = {}
        base = self.base_estimator if self.base_estimator is not None \
               else XGBClassifier()
        for k in self.classes_[:-1]:
            y_binary = (y > k).astype(int)
            sample_weights = compute_sample_weight(class_weight="balanced",
                                                   y=y_binary)
            clf = clone(base)
            clf.fit(X, y_binary, sample_weight=sample_weights)
            self.classifiers_[k] = clf
        return self

    def predict_proba(self, X):
        check_is_fitted(self, ["classifiers_", "classes_"])
        X = check_array(X)
        probas_gt = {-1: np.ones(X.shape[0])}
        for k in self.classifiers_:
            probas_gt[k] = self.classifiers_[k].predict_proba(X)[:, 1]
        probas_gt[self.classes_[-1]] = np.zeros(X.shape[0])
        probs = []
        for i, k in enumerate(self.classes_):
            prev_prob = probas_gt[self.classes_[i - 1] if i > 0 else -1]
            p = np.maximum(0, prev_prob - probas_gt[k])
            probs.append(p)
        probs_matrix = np.column_stack(probs)
        return probs_matrix / probs_matrix.sum(axis=1, keepdims=True)

    def predict(self, X):
        check_is_fitted(self, ["classifiers_", "classes_"])
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]


# ─────────────────────────────────────────────────────────────
# 1b.  Custom Stacking Classifier
# ─────────────────────────────────────────────────────────────
class OrdinalStackingClassifier(BaseEstimator, ClassifierMixin):
    _estimator_type = "classifier"

    def __init__(self, base_estimators, meta_learner, inner_cv=5):
        self.base_estimators = base_estimators
        self.meta_learner    = meta_learner
        self.inner_cv        = inner_cv

    def fit(self, X, y):
        X, y = check_X_y(X, y)
        self.classes_ = np.sort(np.unique(y))

        oof_preds = []
        for name, est in self.base_estimators:
            oof = cross_val_predict(
                est, X, y,
                cv=self.inner_cv,
                method="predict_proba",
                n_jobs=1,
            )
            oof_preds.append(oof)

        X_meta = np.hstack(oof_preds)
        self.meta_learner_ = clone(self.meta_learner)
        self.meta_learner_.fit(X_meta, y)

        self.fitted_bases_ = []
        for name, est in self.base_estimators:
            fitted = clone(est)
            fitted.fit(X, y)
            self.fitted_bases_.append((name, fitted))

        return self

    def predict_proba(self, X):
        check_is_fitted(self, ["meta_learner_", "fitted_bases_", "classes_"])
        X = check_array(X)
        test_preds = [est.predict_proba(X) for _, est in self.fitted_bases_]
        return self.meta_learner_.predict_proba(np.hstack(test_preds))

    def predict(self, X):
        check_is_fitted(self, ["classes_"])
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]


# ─────────────────────────────────────────────────────────────
# 2.  Scoring Metrics  (3-CLASS)
# ─────────────────────────────────────────────────────────────
def qwk_0_9(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    if len(np.unique(y_true)) < 2:
        return 0.0
    n = 3
    O = np.zeros((n, n), dtype=float)
    for a, b in zip(y_true, y_pred):
        O[a, b] += 1.0
    act  = O.sum(axis=1)
    pred = O.sum(axis=0)
    E    = np.outer(act, pred) / max(O.sum(), 1.0)
    W    = np.array([[(i - j) ** 2 / (n - 1) ** 2
                      for j in range(n)] for i in range(n)])
    num  = (W * O).sum()
    den  = (W * E).sum()
    return 1.0 - (num / den if den > 0 else 0.0)

def mae_0_9(y_true, y_pred):
    return np.mean(np.abs(np.asarray(y_true, float) - np.asarray(y_pred, float)))

def within1(y_true, y_pred):
    return np.mean(np.abs(np.asarray(y_true, float) - np.asarray(y_pred, float)) <= 1.0)

def exact_acc(y_true, y_pred):
    return np.mean(np.asarray(y_true, int) == np.asarray(y_pred, int))

scoring_ordinal = {
    "qwk":       make_scorer(qwk_0_9,   greater_is_better=True),
    "mae":       make_scorer(mae_0_9,   greater_is_better=False),
    "within1":   make_scorer(within1,   greater_is_better=True),
    "exact_acc": make_scorer(exact_acc, greater_is_better=True),
}



In [ ]:
import warnings
import numpy as np
import pandas as pd
import optuna
import matplotlib.pyplot as plt
import umap
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.utils.validation import check_X_y, check_array, check_is_fitted
from sklearn.metrics import make_scorer
from sklearn.model_selection import (StratifiedGroupKFold, cross_validate,
                                     cross_val_score, cross_val_predict)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import (VarianceThreshold, SelectFromModel,
                                        SelectKBest, mutual_info_classif)
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from xgboost import XGBClassifier
from sklearn.utils.class_weight import compute_sample_weight
import colorlog

# Silence noisy but harmless warnings from silhouette_score + KMeans
warnings.filterwarnings(
    "ignore",
    message="Clustering metrics expects discrete values",
    category=UserWarning,
)


# ─────────────────────────────────────────────────────────────
# 1.  Ordinal Classifier
# ─────────────────────────────────────────────────────────────
class FrankHallOrdinalClassifier(BaseEstimator, ClassifierMixin):
    _estimator_type = "classifier"

    def __init__(self, base_estimator=None):
        self.base_estimator = base_estimator

    def fit(self, X, y):
        X, y = check_X_y(X, y)
        self.classes_     = np.sort(np.unique(y))
        self.classifiers_ = {}
        base = self.base_estimator if self.base_estimator is not None \
               else XGBClassifier()
        for k in self.classes_[:-1]:
            y_binary = (y > k).astype(int)
            sample_weights = compute_sample_weight(class_weight="balanced",
                                                   y=y_binary)
            clf = clone(base)
            clf.fit(X, y_binary, sample_weight=sample_weights)
            self.classifiers_[k] = clf
        return self

    def predict_proba(self, X):
        check_is_fitted(self, ["classifiers_", "classes_"])
        X = check_array(X)
        probas_gt = {-1: np.ones(X.shape[0])}
        for k in self.classifiers_:
            probas_gt[k] = self.classifiers_[k].predict_proba(X)[:, 1]
        probas_gt[self.classes_[-1]] = np.zeros(X.shape[0])
        probs = []
        for i, k in enumerate(self.classes_):
            prev_prob = probas_gt[self.classes_[i - 1] if i > 0 else -1]
            p = np.maximum(0, prev_prob - probas_gt[k])
            probs.append(p)
        probs_matrix = np.column_stack(probs)
        return probs_matrix / probs_matrix.sum(axis=1, keepdims=True)

    def predict(self, X):
        check_is_fitted(self, ["classifiers_", "classes_"])
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]


# ─────────────────────────────────────────────────────────────
# 1b.  Custom Stacking Classifier
# ─────────────────────────────────────────────────────────────
class OrdinalStackingClassifier(BaseEstimator, ClassifierMixin):
    _estimator_type = "classifier"

    def __init__(self, base_estimators, meta_learner, inner_cv=5):
        self.base_estimators = base_estimators
        self.meta_learner    = meta_learner
        self.inner_cv        = inner_cv

    def fit(self, X, y):
        X, y = check_X_y(X, y)
        self.classes_ = np.sort(np.unique(y))

        oof_preds = []
        for name, est in self.base_estimators:
            oof = cross_val_predict(
                est, X, y,
                cv=self.inner_cv,
                method="predict_proba",
                n_jobs=1,
            )
            oof_preds.append(oof)

        X_meta = np.hstack(oof_preds)
        self.meta_learner_ = clone(self.meta_learner)
        self.meta_learner_.fit(X_meta, y)

        self.fitted_bases_ = []
        for name, est in self.base_estimators:
            fitted = clone(est)
            fitted.fit(X, y)
            self.fitted_bases_.append((name, fitted))

        return self

    def predict_proba(self, X):
        check_is_fitted(self, ["meta_learner_", "fitted_bases_", "classes_"])
        X = check_array(X)
        test_preds = [est.predict_proba(X) for _, est in self.fitted_bases_]
        return self.meta_learner_.predict_proba(np.hstack(test_preds))

    def predict(self, X):
        check_is_fitted(self, ["classes_"])
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]


# ─────────────────────────────────────────────────────────────
# 2.  Scoring Metrics  (3-CLASS)
# ─────────────────────────────────────────────────────────────
def qwk_0_9(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)
    if len(np.unique(y_true)) < 2:
        return 0.0
    n = 3
    O = np.zeros((n, n), dtype=float)
    for a, b in zip(y_true, y_pred):
        O[a, b] += 1.0
    act  = O.sum(axis=1)
    pred = O.sum(axis=0)
    E    = np.outer(act, pred) / max(O.sum(), 1.0)
    W    = np.array([[(i - j) ** 2 / (n - 1) ** 2
                      for j in range(n)] for i in range(n)])
    num  = (W * O).sum()
    den  = (W * E).sum()
    return 1.0 - (num / den if den > 0 else 0.0)

def mae_0_9(y_true, y_pred):
    return np.mean(np.abs(np.asarray(y_true, float) - np.asarray(y_pred, float)))

def within1(y_true, y_pred):
    return np.mean(np.abs(np.asarray(y_true, float) - np.asarray(y_pred, float)) <= 1.0)

def exact_acc(y_true, y_pred):
    return np.mean(np.asarray(y_true, int) == np.asarray(y_pred, int))

scoring_ordinal = {
    "qwk":       make_scorer(qwk_0_9,   greater_is_better=True),
    "mae":       make_scorer(mae_0_9,   greater_is_better=False),
    "within1":   make_scorer(within1,   greater_is_better=True),
    "exact_acc": make_scorer(exact_acc, greater_is_better=True),
}



In [ ]:

# ─────────────────────────────────────────────────────────────
# 3.  Data Prep
# ─────────────────────────────────────────────────────────────
X = np.asarray(X_final, dtype=float)
y = pd.to_numeric(df_merged["pxrd_score"], errors="coerce").to_numpy()

mask = np.isfinite(y)
X, y = X[mask], y[mask].astype(int)

def remap_score(s):
    if s <= 2: return 0
    if s <= 5: return 1
    return 2

y = np.array([remap_score(s) for s in y])

print("3-class distribution:")
class_labels = {0: "Amorphous  (0–2)", 1: "Partial    (3–5)", 2: "Crystalline(6–9)"}
for c in range(3):
    n = (y == c).sum()
    print(f"  Class {c} {class_labels[c]}: {n:4d}  ({100*n/len(y):.1f}%)")

# 3b. Pre-filtering — MUST run BEFORE UMAP

# ── 3b-0. KMeans clustering (runs BEFORE VT and MI) ───────────────────────
# Purpose: Discover sample-level clusters in the raw high-dim space,
# encode them as binary features, and let MI decide if they're informative.
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, OneHotEncoder

N_CLUSTERS = 10  # tune: try 5, 8, 10, 15, 20

print(f"[KMeans] Scaling X ({X.shape}) before clustering…")
_scaler_km = StandardScaler()
_X_scaled   = _scaler_km.fit_transform(X)

print(f"[KMeans] Fitting k={N_CLUSTERS} clusters…")
_km_pre          = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
cluster_labels_raw = _km_pre.fit_predict(_X_scaled)          # (n,) int labels

# One-hot encode → binary columns so VT/MI treat them fairly
_ohe_km    = OneHotEncoder(sparse_output=False, dtype=float)
_cluster_ohe = _ohe_km.fit_transform(cluster_labels_raw.reshape(-1, 1))
# shape: (n, N_CLUSTERS)

X = np.hstack([X, _cluster_ohe])   # (n, 50684 + N_CLUSTERS)
print(f"[KMeans] X shape after appending cluster OHE features: {X.shape}")
print(f"[KMeans] Cluster distribution: "
      f"{ {int(k): int(v) for k, v in zip(*np.unique(cluster_labels_raw, return_counts=True))} }")

# ── 3b-1. VarianceThreshold (unchanged) ───────────────────────────────────
vt_pre = VarianceThreshold(threshold=0.0)
X = vt_pre.fit_transform(X)
print(f"After VarianceThreshold: {X.shape}")

# ── 3b-2: MI DIAGNOSTIC ONLY — no longer applied to X here ──────────
# X is currently (756, 9631) after VT. Save it before any MI leakage.
X_vt = X.copy()   # ← the clean, leak-free version

# Run MI once globally — ONLY for the cliff diagnostic plot.
# DO NOT reassign X here. mi_pre is kept so the cliff plot cell still works.
mi_pre = SelectKBest(
    score_func=lambda X, y: mutual_info_classif(
        X, y, discrete_features=True, random_state=42
    ),
    k=min(5500, X_vt.shape[1]),
)
mi_pre.fit(X_vt, y)   # fit only — transform NOT called

print(f"After VarianceThreshold   : {X_vt.shape}")
print(f"MI will select top-1600 INSIDE each CV fold (no leakage)")
print(f"Diagnostic MI fit complete — cliff plot cell still works unchanged")



# ── 3b-ii: Force-append process vars + interactions ───────────────────
# (unchanged logic — just applied to X_vt instead of MI-filtered X)
from sklearn.preprocessing import MinMaxScaler

Xprocessraw = df_merged[process_cols_present].apply(
    pd.to_numeric, errors='coerce'
).fillna(0.0).to_numpy()[mask]

Xprocnorm = MinMaxScaler().fit_transform(Xprocessraw)

ti  = list(process_cols_present).index('temperature_k')
rhi = list(process_cols_present).index('reaction_hours')
mri = list(process_cols_present).index('metal_over_linker_ratio')

interactions = np.column_stack([
    Xprocnorm[:, ti]  * Xprocnorm[:, mri],   # temp × metal_ratio
    Xprocnorm[:, ti]  * Xprocnorm[:, rhi],   # temp × rxn_hours
    Xprocnorm[:, mri] * Xprocnorm[:, rhi],   # metal_ratio × rxn_hours
    Xprocnorm[:, ti]  ** 2,
    Xprocnorm[:, mri] ** 2,
    (Xprocnorm[:, ti] > 0.85).astype(float), # high-temp flag ≥ 350 K
])

# ── BUILD X FOR CV (no MI leakage) ───────────────────────────────────
# X = VT-filtered features + process vars + interactions
# MI selection happens INSIDE the pipeline per fold
X = np.hstack([X_vt, Xprocnorm, interactions])
print(f"X for CV (VT-only + proc/interactions, no MI): {X.shape}")
# Expected: (756, 9631 + 3 + 6) = (756, 9640)

# ── 3b-iii: PCA for group assignment (cross-platform reproducible) ───
# Replaces UMAP: UMAP's approximate nearest-neighbor graph gives
# different embeddings on Windows vs Linux even with the same seed,
# shifting CV groups and QWK by ±10-15 points.
# PCA(50) captures more variance than 2D UMAP for KMeans clustering
# and is fully deterministic across all platforms.
from sklearn.decomposition import PCA

X_for_pca = np.hstack([
    mi_pre.transform(X_vt),   # MI-selected features
    Xprocnorm,
    interactions
])
n_comp = min(50, X_for_pca.shape[0] - 1, X_for_pca.shape[1])
print(f"PCA input: {X_for_pca.shape}  →  {n_comp} components")
reducer = PCA(n_components=n_comp, random_state=42)
X_2d = reducer.fit_transform(X_for_pca)
var_exp = reducer.explained_variance_ratio_.sum()
print(f"PCA done: {X_2d.shape}  (cumulative variance: {var_exp:.3f})")


In [ ]:
# ── MI score cliff check ──────────────────────────────────────
mi_scores_sorted = np.sort(mi_pre.scores_)[::-1]

print(f"MI at rank  300: {mi_scores_sorted[299]:.5f}")
print(f"MI at rank  400: {mi_scores_sorted[399]:.5f}")
print(f"MI at rank  500: {mi_scores_sorted[499]:.5f}")
print(f"MI at rank  750: {mi_scores_sorted[749]:.5f}")
print(f"MI at rank 1000: {mi_scores_sorted[999]:.5f}")
print(f"MI at rank 1500: {mi_scores_sorted[1499]:.5f}")
print(f"MI at rank 2000: {mi_scores_sorted[1999]:.5f}")
print(f"MI at rank  2500: {mi_scores_sorted[2499]:.5f}")
print(f"MI at rank  3000: {mi_scores_sorted[2999]:.5f}")
print(f"MI at rank  3500: {mi_scores_sorted[3499]:.5f}")
print(f"MI at rank  4000: {mi_scores_sorted[3999]:.5f}")
print(f"MI at rank 4500: {mi_scores_sorted[4499]:.5f}")
print(f"MI at rank 5000: {mi_scores_sorted[4999]:.5f}")
print(f"MI at rank  5500: {mi_scores_sorted[5499]:.5f}")
print(f"MI at rank  6000: {mi_scores_sorted[5999]:.5f}")
print(f"MI at rank  6500: {mi_scores_sorted[6499]:.5f}")
print(f"MI at rank  7000: {mi_scores_sorted[6999]:.5f}")
print(f"MI at rank 7500: {mi_scores_sorted[7499]:.5f}")
print(f"MI at rank 8000: {mi_scores_sorted[7999]:.5f}")

import matplotlib.pyplot as plt
plt.figure(figsize=(10, 4))
plt.plot(mi_scores_sorted[:10000])
plt.axvline(300, color='red',    linestyle='--', label='k=300')
plt.axvline(750, color='orange', linestyle='--', label='k=750')
plt.axvline(5500, color='green',  linestyle='--', label='k=5500')
plt.xlabel("Feature rank")
plt.ylabel("MI score")
plt.title("MI score cliff")
plt.legend(); plt.tight_layout()
plt.savefig("mi_cliff.png", dpi=150); plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────
# DIAGNOSTIC: Process Variable Signal Check
#
# Uses Xprocess (already computed in Block 6 of feature assembly)
# and processcols_present — no feature name reconstruction needed.
# ─────────────────────────────────────────────────────────────

# ── Step 1: Recover process variable matrix (apply y-finite mask) ──
# Xprocess was built before masking; apply the same mask used on y
Xprocess_raw = df_merged[process_cols_present].apply(
    pd.to_numeric, errors="coerce"
).fillna(0.0).to_numpy()[mask]    # (756, n_process_cols)

print(f"Process variable matrix: {Xprocess_raw.shape}")
print(f"\n{'Feature':<35} {'Unique':>7} {'Min':>10} {'Max':>10} {'Mean':>10}")
print("─" * 75)
for i, col in enumerate(process_cols_present):
    col_data = Xprocess_raw[:, i]
    print(f"{col:<35} {len(np.unique(col_data)):>7d} "
          f"{col_data.min():>10.3f} {col_data.max():>10.3f} "
          f"{col_data.mean():>10.3f}")


# ── Step 2: Check if any process vars survived VT + MI ────────
# Find the column indices of process vars in the FULL X_final
# by locating them at the END of the Xfeatures block
# (process cols are the last columns of Xfeatures before RAC blocks)
n_proc = len(process_cols_present)

# Total columns in Xfeatures (first block):
# Xlinker(2048) + Xmodulator(2048) + modeq(1) +
# Xprecursorperlig + Xinventorynumeric + Xprocess
# → process vars occupy the LAST n_proc columns of Xfeatures
n_xfeatures = (X_linker.shape[1] + X_linker_scaled.shape[1] +
               X_modulator.shape[1] + X_modulator_scaled.shape[1] +
               mod_eq.shape[1] +
               X_precursor_perlig.shape[1] + Xinventorynumeric.shape[1] +
               X_process.shape[1])

proc_start_in_X_final = n_xfeatures - n_proc   # column index in X_final

# Check which of those survived VT
vt_support = vt_pre.get_support()              # (n_X_final_pre_vt,) bool
proc_survived_vt = [
    (col, vt_support[proc_start_in_X_final + i])
    for i, col in enumerate(process_cols_present)
]

# Among VT survivors: find their column index after VT, then check MI
names_vt = np.where(vt_support)[0]            # original indices that survived VT
proc_in_vt_space = {}
for i, col in enumerate(process_cols_present):
    orig_idx = proc_start_in_X_final + i
    if vt_support[orig_idx]:
        vt_col = np.where(names_vt == orig_idx)[0]
        if len(vt_col):
            proc_in_vt_space[col] = int(vt_col[0])

mi_support = mi_pre.get_support()             # (n_after_vt,) bool
print(f"\n─── Process variable survival through VT + MI ─────────")
print(f"{'Variable':<35} {'Survived VT':>12} {'Survived MI':>12}")
print("─" * 62)
for col in process_cols_present:
    sv = col in proc_in_vt_space
    sm = sv and mi_support[proc_in_vt_space[col]]
    print(f"{col:<35} {'✅' if sv else '❌':>12}   {'✅' if sm else '❌':>10}")

n_proc_in_X = sum(
    mi_support[proc_in_vt_space[c]]
    for c in process_cols_present if c in proc_in_vt_space
)
print(f"\n{'─'*62}")
print(f"Process variables in final X (2000 features): {n_proc_in_X} / {n_proc}")
if n_proc_in_X == 0:
    print("⚠️  NO process variables reached Optuna — forcing inclusion below.")


# ── Step 3: MI scores for process variables (standalone) ──────
# Compute MI of raw process vars vs y independently of the 50k pool
from sklearn.feature_selection import mutual_info_classif as mic

mi_proc = mic(
    Xprocess_raw, y,
    discrete_features=False,   # process vars are continuous
    random_state=42
)
print(f"\n─── Process variable MI scores (vs 3-class y) ─────────")
print(f"{'Variable':<35} {'MI Score':>10}  {'Signal?':>10}")
print("─" * 60)
for col, score in sorted(zip(process_cols_present, mi_proc),
                          key=lambda x: -x[1]):
    signal = "✅ strong" if score > 0.05 else ("⚠️ weak" if score > 0.01 else "❌ noise")
    print(f"{col:<35} {score:>10.4f}  {signal}")


# ── Step 4: Process variable UMAP ─────────────────────────────
from sklearn.preprocessing import StandardScaler as _SS

Xproc_scaled = _SS().fit_transform(Xprocess_raw)

fig, axes = plt.subplots(1, 3, figsize=(22, 6))

# Panel 1 — full feature UMAP (placeholder; will be filled after UMAP runs)
# We re-use X_2d which is computed in the UMAP cell below — skip if not yet run
try:
    sc1 = axes[0].scatter(X_2d[:, 0], X_2d[:, 1],
                           c=y, cmap="RdYlGn", s=15, alpha=0.7, vmin=0, vmax=2)
    plt.colorbar(sc1, ax=axes[0], label="3-Class Score")
    axes[0].set_title("All-Feature UMAP (2000 MI features)\nFingerprint-dominated")
    axes[0].set_xlabel("UMAP 1"); axes[0].set_ylabel("UMAP 2")
except NameError:
    axes[0].text(0.5, 0.5, "Run UMAP cell first\n(X_2d not yet defined)",
                 ha="center", va="center", transform=axes[0].transAxes, fontsize=12)

# Panel 2 — process variable UMAP
print("\nFitting process-variable UMAP...")
reducer_proc = umap.UMAP(n_components=2, random_state=42,
                          n_neighbors=min(15, len(y) - 1), min_dist=0.1)
X_proc_2d = reducer_proc.fit_transform(Xproc_scaled)

sc2 = axes[1].scatter(X_proc_2d[:, 0], X_proc_2d[:, 1],
                       c=y, cmap="RdYlGn", s=15, alpha=0.7, vmin=0, vmax=2)
plt.colorbar(sc2, ax=axes[1], label="3-Class Score")
axes[1].set_title(f"Process Variable UMAP\n({len(process_cols_present)} variables only)")
axes[1].set_xlabel("UMAP 1"); axes[1].set_ylabel("UMAP 2")

# Panel 3 — top-2 process vars by MI score as a scatter
top2 = sorted(zip(process_cols_present, mi_proc, range(len(process_cols_present))),
              key=lambda x: -x[1])[:2]
(c1, s1, i1), (c2, s2, i2) = top2[0], top2[1]
sc3 = axes[2].scatter(Xprocess_raw[:, i1], Xprocess_raw[:, i2],
                       c=y, cmap="RdYlGn", s=20, alpha=0.7, vmin=0, vmax=2)
plt.colorbar(sc3, ax=axes[2], label="3-Class Score")
axes[2].set_xlabel(f"{c1}\n(MI={s1:.4f})")
axes[2].set_ylabel(f"{c2}\n(MI={s2:.4f})")
axes[2].set_title("Top-2 Process Variables\nvs. Crystallinity")

plt.tight_layout()
plt.savefig("process_diagnostic.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: process_diagnostic.png")


# ── Step 5: Force-include process vars if they were filtered out ──
if n_proc_in_X == 0:
    print("\n─── Forcing process variables into X ──────────────────")
    X = np.hstack([X, Xproc_scaled])
    print(f"X shape after forced process inclusion: {X.shape}")
    print("Note: re-run UMAP + KMeans cells after this change.")





In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# DIAGNOSTIC: UMAP after Contrastive Learning
# Shows whether CL embedding separates the 3 crystallinity classes
# better than the raw feature UMAP (X_2d).
# ═══════════════════════════════════════════════════════════════════════════

class ContrastiveMITransformer(BaseEstimator, TransformerMixin):
    """
    sklearn-compatible contrastive feature augmenter.

    Input to fit/transform:
        X = pipeline features AFTER impute -> vt

    Output from transform:
        [X, embedding] if concat_original=True
        embedding only if concat_original=False

    Modes:
        - loss_mode="supcon"  (recommended default)
        - loss_mode="triplet" (exercise-style)
    """

    def __init__(
        self,
        embedding_dim=128,
        hidden_dim=256,
        loss_mode="supcon",              # "supcon" or "triplet"
        negative_class="partial",       # for triplet mode: "partial", "amorphous", or 1/0
        temperature=0.07,               # supcon
        margin=0.5,                     # triplet
        epochs=15,
        batch_size=128,
        lr=1e-3,
        weight_decay=1e-4,
        n_triplets=4000,
        balanced_batches=True,
        scale_for_cl=True,
        concat_original=True,
        device=None,
        random_state=42,
        verbose=False,
    ):
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.loss_mode = loss_mode
        self.negative_class = negative_class
        self.temperature = temperature
        self.margin = margin
        self.epochs = epochs
        self.batch_size = batch_size
        self.lr = lr
        self.weight_decay = weight_decay
        self.n_triplets = n_triplets
        self.balanced_batches = balanced_batches
        self.scale_for_cl = scale_for_cl
        self.concat_original = concat_original
        self.device = device
        self.random_state = random_state
        self.verbose = verbose

    def _get_device(self):
        if self.device is not None:
            return torch.device(self.device)
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def _set_seed(self):
        np.random.seed(self.random_state)
        torch.manual_seed(self.random_state)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(self.random_state)

    def _build_encoder(self, input_dim):
        return _MLPEncoder(
            in_dim=input_dim,
            hidden_dim=self.hidden_dim,
            embedding_dim=self.embedding_dim,
            dropout=0.2,
        )

    def _supcon_loss(self, z, y):
        z = F.normalize(z, dim=1)
        logits = torch.matmul(z, z.T) / self.temperature

        n = z.shape[0]
        eye = torch.eye(n, dtype=torch.bool, device=z.device)

        logits = logits.masked_fill(eye, -1e9)
        pos_mask = (y.view(-1, 1) == y.view(1, -1)) & (~eye)

        pos_counts = pos_mask.sum(dim=1)
        valid = pos_counts > 0
        if valid.sum() == 0:
            return None

        log_prob = F.log_softmax(logits, dim=1)
        mean_log_prob_pos = (pos_mask * log_prob).sum(dim=1) / pos_counts.clamp(min=1)
        loss = -mean_log_prob_pos[valid].mean()
        return loss

    def _make_supcon_loader(self, X, y):
      X_t = torch.tensor(X, dtype=torch.float32)
      y_t = torch.tensor(y, dtype=torch.long)
      ds = TensorDataset(X_t, y_t)

      # Seeded generator for reproducibility
      generator = torch.Generator()
      generator.manual_seed(self.random_state)

      if not self.balanced_batches:
          return DataLoader(
              ds,
              batch_size=self.batch_size,
              shuffle=True,
              drop_last=False,
              generator=generator,   # ← add here too
          )

      classes, counts = np.unique(y, return_counts=True)
      class_w = {c: 1.0 / cnt for c, cnt in zip(classes, counts)}
      sample_w = np.array([class_w[yi] for yi in y], dtype=np.float32)
      sampler = WeightedRandomSampler(
          weights=torch.tensor(sample_w, dtype=torch.float32),
          num_samples=len(sample_w),
          replacement=True,
          generator=generator,       # ← seeds the sampler itself
      )
      return DataLoader(
          ds,
          batch_size=self.batch_size,
          sampler=sampler,
          drop_last=False,
          generator=generator,       # ← seeds the DataLoader worker
      )


    def _build_triplets(self, y):
        y = np.asarray(y).astype(int)

        idx_pos = np.where(y == 2)[0]      # crystalline anchors/positives

        if len(idx_pos) < 2:
            raise ValueError("Triplet mode needs at least 2 class-2 samples.")

        if self.negative_class in ("partial", 1):
            idx_neg = np.where(y == 1)[0]
        elif self.negative_class in ("amorphous", 0):
            idx_neg = np.where(y == 0)[0]
        elif self.negative_class in ("rest", "non_crystalline"):
            idx_neg = np.where(y != 2)[0]
        else:
            raise ValueError(
                "negative_class must be one of: 'partial', 'amorphous', 'rest', 0, 1."
            )

        if len(idx_neg) == 0:
            raise ValueError(f"No negatives available for negative_class={self.negative_class!r}.")

        rng = np.random.default_rng(self.random_state)
        triplets = []
        for _ in range(self.n_triplets):
            a_idx, p_idx = rng.choice(idx_pos, size=2, replace=False)
            n_idx = rng.choice(idx_neg)
            triplets.append((int(a_idx), int(p_idx), int(n_idx)))
        return triplets

    def fit(self, X, y):
        self._set_seed()

        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y)
        self.n_features_in_ = X.shape[1]

        if self.scale_for_cl:
            self.scaler_ = StandardScaler()
            X_cl = self.scaler_.fit_transform(X).astype(np.float32)
        else:
            self.scaler_ = None
            X_cl = X

        self.encoder_ = self._build_encoder(X_cl.shape[1])
        device = self._get_device()
        self.encoder_.to(device)

        opt = optim.Adam(self.encoder_.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=self.epochs)

        self.loss_history_ = []

        if self.loss_mode == "supcon":
            loader = self._make_supcon_loader(X_cl, y)

            for epoch in range(self.epochs):
                self.encoder_.train()
                losses = []

                for xb, yb in loader:
                    xb = xb.to(device)
                    yb = yb.to(device)

                    z = self.encoder_(xb)
                    loss = self._supcon_loss(z, yb)
                    if loss is None or not torch.isfinite(loss):
                        continue

                    opt.zero_grad()
                    loss.backward()
                    opt.step()
                    losses.append(float(loss.item()))

                sched.step()
                epoch_loss = float(np.mean(losses)) if losses else np.nan
                self.loss_history_.append(epoch_loss)

                if self.verbose:
                    print(f"[SupCon] epoch {epoch+1:02d}/{self.epochs}  loss={epoch_loss:.4f}")

        elif self.loss_mode == "triplet":
          self.triplets_ = self._build_triplets(y)

          generator = torch.Generator()
          generator.manual_seed(self.random_state)
          loader = DataLoader(
            TripletDataset(X_cl, self.triplets_),
            batch_size=self.batch_size,
            shuffle=True,
            drop_last=True,
            generator=generator,
          )

          loss_fn = nn.TripletMarginLoss(margin=self.margin, p=2)

          for epoch in range(self.epochs):
              self.encoder_.train()
              losses = []

              for a, p, n in loader:
                  a = a.to(device)
                  p = p.to(device)
                  n = n.to(device)

                  za = F.normalize(self.encoder_(a), dim=1)
                  zp = F.normalize(self.encoder_(p), dim=1)
                  zn = F.normalize(self.encoder_(n), dim=1)

                  loss = loss_fn(za, zp, zn)
                  if not torch.isfinite(loss):
                      continue

                  opt.zero_grad()
                  loss.backward()
                  opt.step()
                  losses.append(float(loss.item()))

              sched.step()
              epoch_loss = float(np.mean(losses)) if losses else np.nan
              self.loss_history_.append(epoch_loss)

              if self.verbose:
                  print(f"[Triplet] epoch {epoch+1:02d}/{self.epochs}  loss={epoch_loss:.4f}")


        else:
            raise ValueError("loss_mode must be 'supcon' or 'triplet'.")

        self.encoder_.cpu()
        self.encoder_.eval()
        return self

    def _embed(self, X):
        X = np.asarray(X, dtype=np.float32)

        if self.scaler_ is not None:
            X_cl = self.scaler_.transform(X).astype(np.float32)
        else:
            X_cl = X

        self.encoder_.eval()
        Xt = torch.tensor(X_cl, dtype=torch.float32)

        out = []
        with torch.no_grad():
            for i in range(0, len(Xt), 256):
                z = self.encoder_(Xt[i:i+256])
                z = F.normalize(z, dim=1)
                out.append(z.cpu().numpy())

        return np.vstack(out)

    def transform(self, X):
        check_is_fitted(self, ["encoder_", "n_features_in_"])
        X = np.asarray(X, dtype=np.float32)
        emb = self._embed(X)

        # IMPORTANT:
        # return the ORIGINAL incoming pipeline X plus learned embedding,
        # not the internally scaled CL matrix.
        if self.concat_original:
            return np.hstack([X, emb])
        return emb

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# SECTIONS 3c – 9  ·  Unified Benchmark: MI-only vs CL+MI  (RF & XGB)
# Replaces all duplicate code.  Pipeline order everywhere:
#   impute → vt → [cl] → mi → smote → ordinal model
# ═══════════════════════════════════════════════════════════════════════════

# ── Imports ─────────────────────────────────────────────────────────────────
from imblearn.pipeline  import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

from sklearn.feature_selection import SelectKBest, mutual_info_classif
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from torch.utils.data import Dataset, DataLoader, TensorDataset, WeightedRandomSampler
import numpy as np
# ── Shared constants ─────────────────────────────────────────────────────────
SMOTE_STRATEGY = {1: 180, 2: 250}
XGB_FIXED      = {"tree_method": "hist", "eval_metric": "logloss", "random_state": 42}
XGB_TUNED_KEYS = {
    "n_estimators", "max_depth", "learning_rate", "subsample",
    "colsample_bytree", "min_child_weight", "gamma", "reg_alpha", "reg_lambda",
}
MI_K       = 5500
CL_EMB_DIM = 128
RANDOM_STATE    = 42
CV_NJOBS        = -1

# Optional:
# If you know which ORIGINAL pre-CL features are continuous, pass a boolean mask here.
# Length must equal the number of features entering the MI step when with_cl=False.
# True  = discrete
# False = continuous
#
# Leave as None to preserve your current notebook behavior:
# all original features treated as discrete, appended CL embedding treated as continuous.
ORIGINAL_DISCRETE_MASK = None
def suggest_rf_params(trial):
    return {
        "n_estimators": trial.suggest_int("n_estimators", 100, 600),
        "max_depth": trial.suggest_int("max_depth", 5, 30),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 15),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2"]),
    }


def suggest_xgb_params(trial):
    return {
        "n_estimators": trial.suggest_int("n_estimators", 100, 800),
        "max_depth": trial.suggest_int("max_depth", 3, 6),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 3, 10),
        "gamma": trial.suggest_float("gamma", 0.0, 2.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-5, 15.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-5, 15.0, log=True),
    }


# -----------------------------
# Objective factory
# -----------------------------
def make_objective(model_type, with_cl=False):
    def objective(trial):
        if model_type == "rf":
            params = suggest_rf_params(trial)
            pipe = make_rf_pipe(params, with_cl=with_cl)
        elif model_type == "xgb":
            params = suggest_xgb_params(trial)
            pipe = make_xgb_pipe(params, with_cl=with_cl)
        else:
            raise ValueError(f"Unknown model_type={model_type}")

        out = cross_validate(
            pipe,
            X, y,
            cv=cv,
            groups=groups,
            scoring=scoring_ordinal,
            n_jobs=CV_NJOBS,
            return_train_score=False,
        )

        qwk = np.mean(out["test_qwk"])
        mae = np.mean(out["test_mae"])
        acc = np.mean(out["test_exact_acc"])

        trial.set_user_attr("mean_qwk", float(qwk))
        trial.set_user_attr("mean_mae", float(mae))
        trial.set_user_attr("mean_exact_acc", float(acc))
        return qwk

    return objective


# -----------------------------
# Optuna helpers
# -----------------------------
def make_progress_callback(label):
    def progress_callback(study, trial):
        marker = "  ◄ NEW BEST" if trial.number == study.best_trial.number else ""
        print(
            f"[{label}] Trial {trial.number:3d} | "
            f"QWK={trial.value:.4f} | Best={study.best_value:.4f}{marker}"
        )
    return progress_callback


def run_study(label, model_type, with_cl, stage1_trials=15, stage2_trials=35):
    print(f"\n{'='*80}")
    print(f"TUNING: {label}")
    print(f"{'='*80}")

    sampler = optuna.samplers.TPESampler(seed=42)
    study = optuna.create_study(direction="maximize", sampler=sampler, study_name=label)

    objective = make_objective(model_type=model_type, with_cl=with_cl)
    callback = make_progress_callback(label)

    if stage1_trials > 0:
        print(f"\nStage 1: {stage1_trials} trials")
        study.optimize(objective, n_trials=stage1_trials, callbacks=[callback])
        print(f"Stage 1 best QWK: {study.best_value:.4f}")

    if stage2_trials > 0:
        print(f"\nStage 2: {stage2_trials} trials")
        study.optimize(objective, n_trials=stage2_trials, callbacks=[callback])

    print(f"\nBest QWK for {label}: {study.best_value:.4f}")
    print(f"Best params for {label}: {study.best_params}")
    return study

class AdaptiveSelectKBest(BaseEstimator, TransformerMixin):
    """
    SelectKBest that:
      - caps k at the number of available columns
      - optionally appends continuous CL embedding columns to the mask
      - can accept a mixed discrete/continuous mask for the original features
    """

    def __init__(
        self,
        k=5500,
        with_cl=False,
        embedding_dim=128,
        base_discrete_mask=None,
        random_state=42,
    ):
        self.k = k
        self.with_cl = with_cl
        self.embedding_dim = embedding_dim
        self.base_discrete_mask = base_discrete_mask
        self.random_state = random_state

    def _disc_mask(self, n_features):
        if not self.with_cl:
            if self.base_discrete_mask is None:
                return True
            mask = np.asarray(self.base_discrete_mask, dtype=bool)
            if len(mask) != n_features:
                raise ValueError(
                    f"base_discrete_mask has length {len(mask)} but X has {n_features} features."
                )
            return mask

        n_orig = n_features - self.embedding_dim
        if n_orig <= 0:
            raise ValueError(
                f"Expected at least {self.embedding_dim + 1} features, got {n_features}."
            )

        if self.base_discrete_mask is None:
            base_mask = np.ones(n_orig, dtype=bool)
        else:
            base_mask = np.asarray(self.base_discrete_mask, dtype=bool)
            if len(base_mask) != n_orig:
                raise ValueError(
                    f"base_discrete_mask has length {len(base_mask)} but original block has {n_orig}."
                )

        return np.r_[base_mask, np.zeros(self.embedding_dim, dtype=bool)]

    def fit(self, X, y):
        X = np.asarray(X)
        k_use = min(self.k, X.shape[1])
        disc = self._disc_mask(X.shape[1])

        self.selector_ = SelectKBest(
            score_func=lambda Xin, yin: mutual_info_classif(
                Xin,
                yin,
                discrete_features=disc,
                random_state=self.random_state,
            ),
            k=k_use,
        )
        self.selector_.fit(X, y)
        return self

    def transform(self, X):
        check_is_fitted(self, "selector_")
        return self.selector_.transform(X)

    def get_support(self, indices=False):
        check_is_fitted(self, "selector_")
        return self.selector_.get_support(indices=indices)

class SafeMISelectKBest(BaseEstimator, TransformerMixin):
    def __init__(self, k=5500, with_cl=False, embedding_dim=128, random_state=42):
        self.k = k
        self.with_cl = with_cl
        self.embedding_dim = embedding_dim
        self.random_state = random_state

    def _make_discrete_mask(self, n_features):
      if not self.with_cl:
          return True

      n_original = n_features - self.embedding_dim
      if n_original <= 0:
          raise ValueError(
              f"Expected appended embedding_dim={self.embedding_dim}, got n_features={n_features}."
          )

      return np.r_[
          np.ones(n_original, dtype=bool),      # original VT features
          np.zeros(self.embedding_dim, dtype=bool)  # CL embedding
      ]


    def fit(self, X, y):
        n_features = X.shape[1]
        discrete_mask = self._make_discrete_mask(n_features)

        self.scores_ = mutual_info_classif(
            X, y,
            discrete_features=discrete_mask,
            random_state=self.random_state
        )

        if self.k == "all":
            self.k_ = n_features
        else:
            self.k_ = min(int(self.k), n_features)

        ranked = np.argsort(self.scores_)[::-1]
        keep = ranked[:self.k_]
        self.support_idx_ = np.sort(keep)
        self.n_features_in_ = n_features
        return self

    def transform(self, X):
        check_is_fitted(self, "support_idx_")
        return X[:, self.support_idx_]

    def get_support(self, indices=False):
        check_is_fitted(self, "support_idx_")
        if indices:
            return self.support_idx_
        mask = np.zeros(self.n_features_in_, dtype=bool)
        mask[self.support_idx_] = True
        return mask

class _TripletDataset(Dataset):
    def __init__(self, X, triplets):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.triplets = triplets

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, i):
        a, p, n = self.triplets[i]
        return self.X[a], self.X[p], self.X[n]


class _MLPEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim=256, embedding_dim=128, dropout=0.2):
        super().__init__()
        h1 = hidden_dim
        h2 = max(hidden_dim // 2, embedding_dim * 2)
        self.net = nn.Sequential(
            nn.Linear(in_dim, h1),
            nn.LayerNorm(h1),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(h1, h2),
            nn.LayerNorm(h2),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(h2, embedding_dim),
        )

    def forward(self, x):
        return self.net(x)











def make_rf_pipe(rf_params, with_cl=False):
    return ImbPipeline(
        steps=_base_steps(with_cl=with_cl) + [
            (
                "ordinal_rf",
                FrankHallOrdinalClassifier(
                    base_estimator=RandomForestClassifier(
                        **rf_params,
                        bootstrap=True,
                        n_jobs=-1,
                        random_state=RANDOM_STATE,
                    )
                ),
            )
        ]
    )


def make_xgb_pipe(xgb_params, with_cl=False):
    return ImbPipeline(
        steps=_base_steps(with_cl=with_cl) + [
            (
                "ordinal_xgb",
                FrankHallOrdinalClassifier(
                    base_estimator=XGBClassifier(
                        **xgb_params,
                        **XGB_FIXED,
                    )
                ),
            )
        ]
    )
def make_cl_transformer():
    return ContrastiveMITransformer(
        embedding_dim=128,
        hidden_dim=512,
        epochs=15,
        lr=1e-4,
        weight_decay=1e-4,
        batch_size=128,
        margin=0.5,
        n_triplets=4000,
        negative_class="partial",   # hard negatives
        concat_original=True,
        scale_for_cl=True,
        random_state=42,
        verbose=False,
    )



# -----------------------------
# Shared feature pipeline
# Order:
# impute -> vt -> [cl] -> mi -> smote -> ordinal model
# -----------------------------
def make_feature_steps(with_cl=False):
    steps = [
        ("impute", SimpleImputer(strategy="median")),
        ("vt", VarianceThreshold(threshold=0.0)),
    ]

    if with_cl:
        steps.append((
            "cl",
            ContrastiveMITransformer(
                embedding_dim=128,
                hidden_dim=512,
                epochs=15,
                lr=1e-4,
                weight_decay=1e-4,
                batch_size=128,
                margin=0.5,
                n_triplets=4000,
                negative_class="partial",   # hard negatives
                concat_original=True,
                scale_for_cl=True,
                random_state=42,
                verbose=False,
            )
        ))

    steps.append((
        "mi",
        SafeMISelectKBest(
            k=5500,
            with_cl=with_cl,
            embedding_dim=128,
            random_state=42,
        )
    ))

    steps.append((
        "smote",
        SMOTE(sampling_strategy={1: 180, 2: 250}, k_neighbors=5, random_state=42)
    ))

    return steps


# ═══════════════════════════════════════════════════════════════════════════
# C.  Sample-weight diagnostics  (informational only; weights are computed
#     inside FrankHallOrdinalClassifier via compute_sample_weight)
# ═══════════════════════════════════════════════════════════════════════════
def balanced_sample_weights(y):
    classes, counts = np.unique(y, return_counts=True)
    w = {c: len(y) / (len(classes) * n) for c, n in zip(classes, counts)}
    return np.array([w[yi] for yi in y])

sample_weights = balanced_sample_weights(y)
print("Sample weights per class:")
for c in [0, 1, 2]:
    print(f"  Class {c}: weight = {sample_weights[y == c][0]:.3f}  "
          f"(n={(y == c).sum()})")


# ═══════════════════════════════════════════════════════════════════════════
# 3c.  UMAP + KMeans grouping  (runs once; groups reused for all CV below)
# ═══════════════════════════════════════════════════════════════════════════
print("\n─── KMeans cluster sweep ──────────────────────────────")
print(f"{'k':>5} {'Silhouette':>12} {'Min class-2 in val':>20} {'Status':>12}")

best_k, best_score = None, -1

for k in range(8, 30, 2):
    km       = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_k = km.fit_predict(X_2d).astype(int)
    sil      = silhouette_score(X_2d, labels_k)

    cv_tmp = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)
    worst  = 999
    try:
        for _, val_idx in cv_tmp.split(X, y, labels_k):
            worst = min(worst, (y[val_idx] == 2).sum())
    except ValueError:
        worst = 0

    status = "✅ good" if worst >= 5 else "✗ too few crystalline"
    print(f"{k:>5d} {sil:>12.4f} {worst:>20d} {status}")
    if worst >= 5 and sil > best_score:
        best_score, best_k = sil, k

if best_k is None:
    print("\nNo k satisfied ≥5 crystalline per fold — falling back to n_splits=3 …")
    cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)
    for k in range(6, 20, 2):
        km       = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels_k = km.fit_predict(X_2d).astype(int)
        sil      = silhouette_score(X_2d, labels_k)
        worst    = 999
        try:
            for _, val_idx in cv.split(X, y, labels_k):
                worst = min(worst, (y[val_idx] == 2).sum())
        except ValueError:
            worst = 0
        status = "✅" if worst >= 5 else "✗"
        print(f"  k={k:2d}  sil={sil:.4f}  min_crystalline={worst}  {status}")
        if worst >= 5 and sil > best_score:
            best_score, best_k = sil, k
else:
    cv = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=42)

print(f"\nSelected k={best_k} (silhouette={best_score:.4f})")
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
groups   = km_final.fit_predict(X_2d).astype(int)

n_groups = len(np.unique(groups))
print(f"Groups: {n_groups} | Avg per group: {len(y)/n_groups:.1f}")

# ── Fold balance check ────────────────────────────────────────────────────
print("─── 3-fold balance ────────────────────────────────────")
for fold, (_, val_idx) in enumerate(cv.split(X, y, groups)):
    c0, c1, c2 = (y[val_idx] == 0).sum(), (y[val_idx] == 1).sum(), (y[val_idx] == 2).sum()
    flag = " ⚠️  low Partial" if c1 < 15 else ""
    print(f"  Fold {fold+1}: n={len(val_idx):3d} | "
          f"Amorphous={c0:3d}  Partial={c1:3d}  Crystalline={c2:3d}{flag}")

# ── Fold 1 chemistry diagnosis ────────────────────────────────────────────
fold1_val_idx = list(cv.split(X, y, groups))[0][1]
df_masked     = df_merged[mask].reset_index(drop=True)
fold1_df      = df_masked.iloc[fold1_val_idx]

print(f"\nFold 1: {len(fold1_df)} samples\n")
fold1_inv   = df_inventory[df_inventory["experiment_id"].isin(fold1_df["experiment_id"].values)]
overall_inv = df_inventory[df_inventory["experiment_id"].isin(df_masked["experiment_id"].values)]

print("── Fold 1 metal atoms ─────────────────────────────────")
print(fold1_inv["metal_atom"].value_counts())
print("\n── Overall metal atoms ────────────────────────────────")
print(overall_inv["metal_atom"].value_counts())
print("\n── Fold 1 precursors (top 10) ─────────────────────────")
print(fold1_df["precursor_iupac_standardized"].value_counts().head(10))
print("\n── Temperature ────────────────────────────────────────")
print(fold1_df["temperature_k"].describe().round(1))
print("\n── Metal:linker ratio ─────────────────────────────────")
print(fold1_df["metal_over_linker_ratio"].describe().round(3))
print("\n── Class distribution ─────────────────────────────────")
for cls, name in {0: "Amorphous", 1: "Partial", 2: "Crystalline"}.items():
    print(f"  {name}: {(y[fold1_val_idx] == cls).sum()}")

# ── UMAP visualisation ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sc1 = axes[0].scatter(X_2d[:, 0], X_2d[:, 1], c=y, cmap="RdYlGn",
                       s=15, alpha=0.7, vmin=0, vmax=2)
plt.colorbar(sc1, ax=axes[0], label="3-Class Score")
axes[0].set_title("Crystallinity Score (3-class) — PCA")
axes[0].set_xlabel("PC 1"); axes[0].set_ylabel("PC 2")

sc2 = axes[1].scatter(X_2d[:, 0], X_2d[:, 1], c=groups, cmap="tab20", s=15, alpha=0.7)
axes[1].set_title(f"KMeans Groups (k={best_k})")
axes[1].set_xlabel("PC 1"); axes[1].set_ylabel("PC 2")
plt.tight_layout()
plt.savefig("groups_umap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: groups_umap.png")
print(f"\nFeature shape entering Optuna: {X.shape}")


# ═══════════════════════════════════════════════════════════════════════════
# 4.  Pipeline factory
#     All four final pipelines share this builder so structure never drifts.
# ═══════════════════════════════════════════════════════════════════════════
def _base_steps(with_cl: bool) -> list:
    """
    Common prefix:
        impute -> vt -> [cl] -> mi -> smote

    Default CL = supervised contrastive over all 3 classes.
    To match the exercise exactly, switch to:
        loss_mode="triplet", negative_class="amorphous"  # easy
    or:
        loss_mode="triplet", negative_class="partial"    # hard
    """
    steps = [
        ("impute", SimpleImputer(strategy="median")),
        ("vt", VarianceThreshold(threshold=0.0)),
    ]

    if with_cl:
        steps.append((
            "cl",
            ContrastiveMITransformer(
                embedding_dim=CL_EMB_DIM,
                hidden_dim=256,
                loss_mode="supcon",          # recommended default
                negative_class="partial",    # used only in triplet mode
                temperature=0.07,
                margin=0.5,
                epochs=15,                   # keep modest: this runs inside CV
                batch_size=128,
                lr=1e-3,
                weight_decay=1e-4,
                n_triplets=4000,
                balanced_batches=True,
                scale_for_cl=True,
                concat_original=True,
                device=None,
                random_state=42,
                verbose=False,
            )
        ))

    steps += [
        (
            "mi",
            AdaptiveSelectKBest(
                k=MI_K,
                with_cl=with_cl,
                embedding_dim=CL_EMB_DIM,
                base_discrete_mask=ORIGINAL_DISCRETE_MASK,
                random_state=42,
            ),
        ),
        (
            "smote",
            SMOTE(
                sampling_strategy=SMOTE_STRATEGY,
                k_neighbors=5,
                random_state=42,
            ),
        ),
    ]
    return steps


# ═══════════════════════════════════════════════════════════════════════════
# 5.  Optuna — XGBoost
#     Tuned on the MI-only pipeline (no CL) for speed.
#     The same best params are later reused for the CL+MI variant.
# ═══════════════════════════════════════════════════════════════════════════
def objective_xgb(trial):
    param = {
        "n_estimators":     trial.suggest_int("n_estimators", 100, 800),
        "max_depth":        trial.suggest_int("max_depth", 3, 6),
        "learning_rate":    trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
        "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 3, 10),
        "gamma":            trial.suggest_float("gamma", 0.0, 2.0),
        "reg_alpha":        trial.suggest_float("reg_alpha", 1e-5, 15.0, log=True),
        "reg_lambda":       trial.suggest_float("reg_lambda", 1e-5, 15.0, log=True),
    }
    pipe   = make_xgb_pipe(param, with_cl=False)          # ← same structure as final pipe
    cv_res = cross_validate(
        pipe, X, y, cv=cv, groups=groups,
        scoring=scoring_ordinal, n_jobs=-1, return_train_score=False,
    )
    fold_qwk = cv_res["test_qwk"]
    fold_mae = cv_res["test_mae"]
    fold_acc = cv_res["test_exact_acc"]
    mean_qwk = np.mean(fold_qwk)

    try:
        is_best = mean_qwk > trial.study.best_value
    except ValueError:
        is_best = True

    if is_best:
        print(f"\n  ↳ Trial {trial.number} per-fold:")
        print(f"    {'Fold':>6} {'QWK':>8} {'MAE':>8} {'ExactAcc':>10} {'Partial_n':>10}")
        for i, (qwk, mae, acc) in enumerate(zip(fold_qwk, fold_mae, fold_acc)):
            val_idx   = list(cv.split(X, y, groups))[i][1]
            n_partial = (y[val_idx] == 1).sum()
            print(f"    {i+1:>6d} {qwk:>8.4f} {mae:>8.4f} {acc:>10.4f} {n_partial:>10d}")
        print(f"    {'Mean':>6} {mean_qwk:>8.4f} {np.mean(fold_mae):>8.4f} "
              f"{np.mean(fold_acc):>10.4f}")
    return mean_qwk


def progress_callback(study, trial):
    marker = " ◄ NEW BEST" if trial.value == study.best_value else ""
    print(f"  Trial {trial.number:3d} | QWK={trial.value:.4f} "
          f"| Best={study.best_value:.4f}{marker}")


study_xgb = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    study_name="xgb_mi_only"
)

# Stage 1 — sanity check (~15 min). Stop if QWK < 0.45.
study_xgb.optimize(objective_xgb, n_trials=15, callbacks=[progress_callback])
print(f"\nStage 1 done. Best XGB QWK: {study_xgb.best_value:.4f}")
print(">>> If QWK < 0.45, stop and check features/labels <<<")

# Stage 2 — full search
study_xgb.optimize(objective_xgb, n_trials=85, callbacks=[progress_callback])
print("Best XGB QWK:", study_xgb.best_value)
print("Best XGB Params:", study_xgb.best_params)


# ═══════════════════════════════════════════════════════════════════════════
# 6.  Optuna — Random Forest
#     Tuned on the MI-only pipeline; same params reused for CL+MI variant.
# ═══════════════════════════════════════════════════════════════════════════
def objective_rf(trial):
    param = {
        "n_estimators":      trial.suggest_int("n_estimators", 100, 600),
        "max_depth":         trial.suggest_int("max_depth", 5, 30),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 15),
        "min_samples_leaf":  trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features":      trial.suggest_categorical("max_features",
                                                        ["sqrt", "log2"]),
    }
    pipe   = make_rf_pipe(param, with_cl=False)            # ← same structure as final pipe
    scores = cross_val_score(
        pipe, X, y, cv=cv, groups=groups,
        scoring=scoring_ordinal["qwk"], n_jobs=-1,
    )
    return np.mean(scores)


study_rf = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    study_name="rf_mi_only"
)
study_rf.optimize(objective_rf, n_trials=50, callbacks=[progress_callback])
print("Best RF QWK:", study_rf.best_value)
print("Best RF Params:", study_rf.best_params)


# ═══════════════════════════════════════════════════════════════════════════
# 7.  Build the four final pipelines
# ═══════════════════════════════════════════════════════════════════════════
best_xgb_params = {k: v for k, v in study_xgb.best_params.items() if k in XGB_TUNED_KEYS}
best_rf_params  = study_rf.best_params          # RF has no fixed-key separation needed

pipe_rf_mi      = make_rf_pipe( best_rf_params,  with_cl=False)   # baseline
pipe_rf_cl_mi   = make_rf_pipe( best_rf_params,  with_cl=True)    # + contrastive
pipe_xgb_mi     = make_xgb_pipe(best_xgb_params, with_cl=False)   # baseline
pipe_xgb_cl_mi  = make_xgb_pipe(best_xgb_params, with_cl=True)   # + contrastive


# ═══════════════════════════════════════════════════════════════════════════
# 8.  Evaluate all four pipelines
# ═══════════════════════════════════════════════════════════════════════════
def eval_pipe(name: str, pipe, n_jobs: int = 1):
    out = cross_validate(
        pipe, X, y, cv=cv, groups=groups,
        scoring=scoring_ordinal, n_jobs=n_jobs,
    )
    print(f"\n{name}")
    print(f"  QWK      {np.mean(out['test_qwk']):.4f}  ±{np.std(out['test_qwk']):.4f}")
    print(f"  MAE      {-np.mean(out['test_mae']):.4f}  ±{np.std(out['test_mae']):.4f}")
    print(f"  Within-1 {np.mean(out['test_within1']):.4f}  ±{np.std(out['test_within1']):.4f}")
    print(f"  Exact    {np.mean(out['test_exact_acc']):.4f}  ±{np.std(out['test_exact_acc']):.4f}")





In [ ]:
import importlib, sys

packages = [
    "mordred", "scipy", "sklearn", "imblearn", "umap",
    "rdkit", "lightgbm", "mendeleev", "morfeus", "networkx",
    "optuna", "colorlog", "drfp", "dscribe", "numpy",
    "torch", "xgboost", "pandas",
]

print(f"{'Package':<20} {'Version':<20} {'Location'}")
print("─" * 80)
for pkg in packages:
    try:
        mod = importlib.import_module(pkg)
        version = getattr(mod, "__version__", "no __version__")
        location = getattr(mod, "__file__", "unknown")
    except ImportError:
        version = "NOT INSTALLED"
        location = ""
    print(f"{pkg:<20} {version:<20} {location}")

print(f"\nPython: {sys.version}")
print(f"Python executable: {sys.executable}")

In [ ]:

# ── Step 1: Fit CL transformer on the full masked X ───────────────────────
print("Fitting ContrastiveMITransformer on full X (this may take ~1-2 min)...")

cl_diag = ContrastiveMITransformer(
    embedding_dim=CL_EMB_DIM,
    hidden_dim=256,
    loss_mode="supcon",
    negative_class="partial",
    temperature=0.07,
    margin=0.5,
    epochs=15,
    batch_size=128,
    lr=1e-3,
    weight_decay=1e-4,
    n_triplets=4000,
    balanced_batches=True,
    scale_for_cl=True,
    concat_original=False,   # embedding only — we only want the 128-dim CL space
    device=None,
    random_state=42,
    verbose=True,
)

# Impute + VT first (same as pipeline does before CL)
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold

_imp = SimpleImputer(strategy="median").fit(X)
X_imp = _imp.transform(X)
_vt  = VarianceThreshold(threshold=0.0).fit(X_imp)
X_vt = _vt.transform(X_imp)

cl_diag.fit(X_vt, y)
X_cl_emb = cl_diag.transform(X_vt)   # (n, 128)
print(f"CL embedding shape: {X_cl_emb.shape}")

# ── Step 2: UMAP on CL embedding ──────────────────────────────────────────
print("Fitting UMAP on CL embedding...")
reducer_cl = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    random_state=42,
)
X_cl_2d = reducer_cl.fit_transform(X_cl_emb)

# ── Step 3: Plot side-by-side comparison ──────────────────────────────────
class_names = {0: "Amorphous", 1: "Partial", 2: "Crystalline"}
colors      = {0: "#d73027", 1: "#fee08b", 2: "#1a9850"}
markers     = {0: "o", 1: "s", 2: "^"}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, X_2d_plot, title in zip(
    axes,
    [X_2d,    X_cl_2d],
    ["Raw Features (pre-CL UMAP)", f"CL Embedding (128-dim → UMAP)"],
):
    for cls in [0, 1, 2]:
        idx = y == cls
        ax.scatter(
            X_2d_plot[idx, 0], X_2d_plot[idx, 1],
            c=colors[cls], marker=markers[cls],
            s=20, alpha=0.7, label=f"{class_names[cls]} (n={idx.sum()})",
            edgecolors="none",
        )
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.set_xlabel("UMAP 1"); ax.set_ylabel("UMAP 2")
    ax.legend(framealpha=0.8, fontsize=9)

plt.suptitle("UMAP: Raw Features vs. Contrastive Learning Embedding", fontsize=14)
plt.tight_layout()
plt.savefig("umap_cl_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: umap_cl_comparison.png")

# ── Step 4: Quantify separation (silhouette score) ────────────────────────
from sklearn.metrics import silhouette_score

sil_raw = silhouette_score(X_2d,    y, random_state=42)
sil_cl  = silhouette_score(X_cl_2d, y, random_state=42)

print(f"\n─── Class separation (silhouette score, higher = better) ──")
print(f"  Raw feature UMAP : {sil_raw:.4f}")
print(f"  CL embedding UMAP: {sil_cl:.4f}")
delta = sil_cl - sil_raw
direction = "✅ CL improved separation" if delta > 0 else "⚠️  CL did not improve separation"
print(f"  Δ = {delta:+.4f}  →  {direction}")

In [ ]:
print("\n─── FINAL COMPARISON ──────────────────────────────────")
# n_jobs=-1 is safe for pure sklearn/XGB pipelines.
# CL pipelines use PyTorch internally; n_jobs=1 avoids fork/CUDA conflicts.
eval_pipe("RF  | MI only",  pipe_rf_mi,     n_jobs=-1)
eval_pipe("RF  | CL + MI",  pipe_rf_cl_mi,  n_jobs=1)
eval_pipe("XGB | MI only",  pipe_xgb_mi,    n_jobs=-1)
eval_pipe("XGB | CL + MI",  pipe_xgb_cl_mi, n_jobs=1)

  1. UMAP is platform-non-deterministic (biggest factor)

  Even with random_state=RANDOM_STATE, UMAP's approximate nearest-neighbor algorithm gives different 2D embeddings on Windows vs. Linux (Colab). This directly changes the KMeans groups in select_kmeans_groups, which changes what samples land in each of the 3 CV folds. With only 3 folds   and a small/imbalanced dataset, which ~33% of samples end up in the validation set swings QWK by ±10-15 points on its own.

  2. XGBoost 3.2 vs. ~1.7 on Colab

  Major behavioral changes across these versions:
  - Default regularization changed (especially reg_lambda default dropped from 1.0 → 0.0 in 2.x)
  - QuantileDMatrix is now the default in hist mode (3.x) — more aggressive approximation, different splits
  - The crash you just saw on trial 3 was the QuantileDMatrix path failing under Windows multiprocessing; that NaN trial wasted search budget in early Optuna trials

  3. apply_variance_threshold adds KMeans OHE features locally

  dimensionality.py:84-94 fits KMeans (k=10) on all data and appends OHE cluster columns to X_cv before cross-validation. This is not a pipeline step — it runs globally. When MI then selects the top-5500 features inside each fold, these globally-learned cluster columns compete with    
  actual molecular features and may or may not be selected, depending on the fold's training data. This is a structural difference from the Colab notebook where VT and MI are purely inside the CV pipeline.

  4. Only 15 trials on noisy 3-fold CV

  With 3 folds and the crash on trial 3, the Optuna search is fitting to noise from the specific group split rather than true generalization. The Colab result of 0.40 at trial 15 was likely on a favorable group split.

  ---
  What to do:

  The most impactful fix is to stabilize the group split by saving and reusing the groups array from the Colab run. Short of that, your best levers are:

  - Run more Optuna trials (100+) — the difference between 0.27 and 0.40 early in the search is mostly variance
  - Check that X_cv shape matches what you had on Colab (print X_cv.shape and compare)
  - Pin XGBoost to a Colab-compatible version (xgboost==1.7.6) to rule out model behavior changes

  The features and their informativeness haven't changed — the platform, library versions, and non-deterministic UMAP grouping are the culprits.

In [ ]:
# =============================================================================
# SHAP Analysis -- Featurized, Chemist-Readable Labels
# =============================================================================
# Produces per XGB pipeline (MI-only and CL+MI):
#   1) Feature-GROUP bar chart  -- category-level importance (most interpretable)
#      e.g. "Process Variables", "Linker ChemBERT", "Metal Center", ...
#   2) Top-N individual feature bar chart -- real names, colour-coded by category
#   3) SHAP beeswarm -- signed impact of top features on crystallinity prediction
#   4) Side-by-side group comparison across both pipelines
#
# CSV files are saved for every plot.
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from sklearn.base import clone
from matplotlib.patches import Patch

# -----------------------------------------------------------------------------
# SECTION A  --  Reconstruct a feature-name + group-label map for X
#
# X (from cell 51) = np.hstack([X_vt,  Xprocnorm,  interactions])
#   X_vt        : vt_pre.transform(np.hstack([X_final, KMeans_OHE_10]))
#   Xprocnorm   : MinMaxScaled process variables (len = N_PROC)
#   interactions: 6 engineered interaction terms
# -----------------------------------------------------------------------------


# A1. Suffix columns -- always the last N_PROC + 6 columns of X
_INT_NAMES = [
    "proc:temp x metal_ratio",
    "proc:temp x rxn_hours",
    "proc:metal_ratio x rxn_hours",
    "proc:temp^2",
    "proc:metal_ratio^2",
    "proc:hightemp_flag",
]
_N_INT  = len(_INT_NAMES)
_N_PROC = len(process_cols_present)
_N_XVT  = X.shape[1] - _N_PROC - _N_INT
assert _N_XVT >= 0, (
    f"X.shape[1]={X.shape[1]} is smaller than N_PROC+N_INT={_N_PROC + _N_INT}. "
    "Check that process_cols_present and _INT_NAMES are correct.")

# A2. Propagate through vt_pre to name the X_vt columns
_vt_mask  = vt_pre.get_support()          # bool, length = X_pre_vt width
_n_pre_vt = len(_vt_mask)
_N_KMEANS = 10                            # N_CLUSTERS used in cell 51
_n_xfinal = _n_pre_vt - _N_KMEANS

assert _vt_mask.sum() == _N_XVT, (
    f"vt_pre kept {_vt_mask.sum()} cols but X_vt block has {_N_XVT} cols. "
    "Re-check N_CLUSTERS or the assembly of X in cell 51.")

# A3. Helpers
def _scope_list(varname, n_expected=None):
    v = globals().get(varname, [])
    v = list(v) if v else []
    if n_expected is None or len(v) == n_expected:
        return v
    return []                              # mismatch -> caller uses generic names

def _ncols(varname):
    v = globals().get(varname)
    if v is not None and hasattr(v, "shape") and v.ndim >= 2:
        return v.shape[1]
    return None

# A4. Build X_final feature names in the exact assembly order (cells 14-44)
_xf_names  = []
_xf_groups = []

def _push(names_iter, group):
    names = list(names_iter)
    _xf_names.extend(names)
    _xf_groups.extend([group] * len(names))

def _push_block(varname, group, name_list=None, prefix=None):
    n = _ncols(varname)
    if n is None:
        return
    if name_list and len(name_list) == n:
        names = name_list
    else:
        pfx = prefix or varname[:14]
        names = [f"{pfx}_{i:04d}" for i in range(n)]
    _push(names, group)

# -- X_features block (cell 14) -----------------------------------------------
# X_features = [X_linker | X_linker_scaled | X_modulator | X_modulator_scaled
#              | mod_eq | X_precursor_perlig | Xinventorynumeric | X_process]
_push_block("X_linker",            "Linker Morgan FP",    prefix="linker_fp")
_push_block("X_linker_scaled",     "Linker Morgan FP",    prefix="linker_fp_sc")
_push_block("X_modulator",         "Modulator Morgan FP", prefix="mod_fp")
_push_block("X_modulator_scaled",  "Modulator Morgan FP", prefix="mod_fp_sc")
_push_block("mod_eq",              "Modulator Equiv.",    prefix="mod_eq")
_push_block("X_precursor_perlig",  "Precursor Ligand FP", prefix="perlig_fp")
_push_block("Xinventorynumeric",   "Inventory Numeric",   prefix="inv_num")

# X_process -- use exact process variable names
_np_raw = _ncols("X_process")
if _np_raw:
    _push([f"proc_raw:{c}" for c in list(process_cols_present)[:_np_raw]],
          "Process Variables (raw)")

# -- Cell 15: modulator RAC + metal block -------------------------------------
_push_block("X_modulator_rac_aug", "Modulator RAC",            prefix="mod_rac")
_push_block("X_metal_block",       "Metal Center (mendeleev)",
            name_list=_scope_list("METAL_DESCRIPTOR_NAMES",
                                  _ncols("X_metal_block")))

# -- Cell 16: metal precursor complex -----------------------------------------
_push_block("Xprecursor_full",     "Metal Precursor Complex",  prefix="prec_cplx")

# -- Cell 17: per-ligand RAC --------------------------------------------------
_push_block("X_precursor_perlig_rac", "Precursor Ligand RAC", prefix="perlig_rac")

# -- Cell 19: physchem --------------------------------------------------------
_push_block("X_linker_phys10",    "Linker Physchem",           prefix="linker_phys")
_push_block("X_modulator_phys10", "Modulator Physchem",        prefix="mod_phys")

# -- Cell 22: per-ligand TEP --------------------------------------------------
_tep = _scope_list("perlig_tep_feature_names")
if _tep:
    _push(_tep, "Ligand TEP (Electronic)")

# -- Cell 25: per-ligand sterics ----------------------------------------------
_ster = _scope_list("perlig_steric_feature_names")
if _ster:
    _push(_ster, "Ligand Sterics")

# -- Cell 34: ChemBERT linker + mod (668 each = 1336 total) ------------------
# Per-molecule block: 384 BERT + 50 extRDKit + 10 shape + 34 VSA + 8 comp
#                   + 167 MACCS + 15 frags = 668
_cbn = _scope_list("chemberta_feature_names_all", 1336)
if _cbn:
    _push(_cbn[:384],      "Linker ChemBERT")
    _push(_cbn[384:668],   "Linker Physchem/FP")
    _push(_cbn[668:1052],  "Mod ChemBERT")
    _push(_cbn[1052:1336], "Mod Physchem/FP")

# -- Cell 36: G14 hub topology (linker + mod) ---------------------------------
_g14hub = _scope_list("g14_feature_names")
if _g14hub:
    _push(_g14hub, "G14 Hub Topology")

# -- Cell 37: G14 SMARTS (linker + mod) ---------------------------------------
_g14s = _scope_list("g14_smarts_feature_names")
if _g14s:
    _push(_g14s, "G14 Hub SMARTS")

# -- Cell 38: TTP linker -------------------------------------------------------
_ttp = _scope_list("TTP_FEATURE_NAMES", 52)
if _ttp:
    _push(_ttp, "Linker TTP")

# -- Cell 40: linker graph / topo FPs -----------------------------------------
for _vn, _pfx, _grp in [
    ("Xlinker_estate",  "linker_estate",    "Linker EState"),
    ("Xlinker_topo",    "linker_topo",      "Linker Topological"),
    ("Xlinker_tor",     "linker_torsion",   "Linker Torsion FP"),
    ("Xlinker_ap",      "linker_atompair",  "Linker Atom-Pair FP"),
]:
    _push_block(_vn, _grp, prefix=_pfx)

# -- Cell 41: halide -----------------------------------------------------------
_hal_names = globals().get("HALIDE_FEAT_COLS",
    ["halide_type","halide_count","halide_present",
     "halide_I_count","halide_Br_count","halide_Cl_count"])
_push(list(_hal_names), "Halide Features")

# -- Cell 42: DRFP (2048) -----------------------------------------------------
_drfp_n = _ncols("X_drfp") or 2048
_push([f"drfp_{i}" for i in range(_drfp_n)], "Reaction FP (DRFP)")

# -- Cell 44: SOAP ------------------------------------------------------------
_soap_all = _scope_list("soap_all_names")
if _soap_all:
    _push(_soap_all, "3D SOAP Descriptor")
else:
    for _vn, _pfx, _grp in [
        ("X_soap_precursor", "soap_prec", "3D SOAP (Precursor)"),
        ("X_soap_linker",    "soap_link", "3D SOAP (Linker)"),
    ]:
        _push_block(_vn, _grp, prefix=_pfx)

# A5. Pad / trim to match _n_xfinal
if len(_xf_names) < _n_xfinal:
    _pad = _n_xfinal - len(_xf_names)
    print(f"  Padding {_pad} unnamed X_final columns as 'Other Structural'")
    _push([f"unnamed_{i:05d}" for i in range(_pad)], "Other Structural")
elif len(_xf_names) > _n_xfinal:
    print(f"  Trimming {len(_xf_names) - _n_xfinal} excess names to match X_final")
    _xf_names  = _xf_names[:_n_xfinal]
    _xf_groups = _xf_groups[:_n_xfinal]

# A6. Append KMeans OHE labels and apply vt_pre mask
_pre_vt_names  = _xf_names  + [f"kmeans_cluster_{i}" for i in range(_N_KMEANS)]
_pre_vt_groups = _xf_groups + ["KMeans Cluster OHE"] * _N_KMEANS

assert len(_pre_vt_names) == _n_pre_vt, (
    f"Pre-VT name count {len(_pre_vt_names)} != {_n_pre_vt}. "
    "Check N_CLUSTERS or block sizes.")

_vt_idx     = np.where(_vt_mask)[0]
_xvt_names  = [_pre_vt_names[i]  for i in _vt_idx]
_xvt_groups = [_pre_vt_groups[i] for i in _vt_idx]

# A7. Assemble full X name map
_X_NAMES  = (  _xvt_names
             + [f"process:{c}" for c in process_cols_present]
             + _INT_NAMES)
_X_GROUPS = (  _xvt_groups
             + ["Process Variables"] * _N_PROC
             + ["Process Interactions"] * _N_INT)

assert len(_X_NAMES) == X.shape[1], (
    f"Feature name count {len(_X_NAMES)} != X.shape[1]={X.shape[1]}. "
    "Re-check block sizes vs actual array shapes.")

_X_NAMES_a  = np.array(_X_NAMES,  dtype=object)
_X_GROUPS_a = np.array(_X_GROUPS, dtype=object)

print(f"Feature map built: {len(_X_NAMES)} features")
for _g, _c in pd.Series(_X_GROUPS).value_counts().items():
    print(f"    {_g:<42} {_c:>5}")


# -----------------------------------------------------------------------------
# SECTION B  --  Propagate names through pipeline VT + [CL] + MI
# -----------------------------------------------------------------------------

def transform_with_names(fitted_pipe, X_in, names_in, groups_in, label=""):
    # Run X through impute -> vt -> [cl] -> mi, tracking which features survive
    # each selection step. Stops before 'smote' and the final ordinal model.
    # Returns (X_model, feat_names_arr, feat_groups_arr).
    Xt    = X_in.copy()
    names = np.array(names_in, dtype=object)
    grps  = np.array(groups_in, dtype=object)

    for step_name, step in fitted_pipe.steps:
        if step_name in ("smote", "ordinal_xgb", "ordinal_rf", "ordinalxgb", "ordinalrf"):
            break

        if step_name == "impute":
            Xt = step.transform(Xt)

        elif step_name == "vt":
            Xt   = step.transform(Xt)
            mask = step.get_support()
            names = names[mask]
            grps  = grps[mask]

        elif step_name == "cl":
            Xt    = step.transform(Xt)
            n_out = Xt.shape[1]
            emb   = getattr(step, "embeddingdim", None)
            if emb is None:
                emb = getattr(step, "embedding_dim", n_out)

            cl_n = [f"cl_emb_{i:03d}" for i in range(min(emb, n_out))]
            cl_g = ["CL Embedding"] * len(cl_n)

            if n_out > emb:
                n_orig = n_out - emb
                orig_n = (
                    list(names[:n_orig]) if len(names) >= n_orig
                    else [f"orig_{i}" for i in range(n_orig)]
                )
                orig_g = (
                    list(grps[:n_orig]) if len(grps) >= n_orig
                    else ["CL Original"] * n_orig
                )
                names = np.array(cl_n + orig_n, dtype=object)
                grps  = np.array(cl_g + orig_g, dtype=object)
            else:
                names = np.array(cl_n, dtype=object)
                grps  = np.array(cl_g, dtype=object)

        elif step_name == "mi":
            Xt   = step.transform(Xt)
            mask = step.get_support()
            names = names[mask]
            grps  = grps[mask]

        else:
            try:
                Xt_new = step.transform(Xt)
                if Xt_new.shape[1] == Xt.shape[1]:
                    Xt = Xt_new
                else:
                    print(f"  [warn] '{step_name}' changed shape "
                          f"{Xt.shape[1]}->{Xt_new.shape[1]}; names reset.")
                    Xt    = Xt_new
                    names = np.array([f"{step_name}_{i}" for i in range(Xt.shape[1])],
                                     dtype=object)
                    grps  = np.array(["Unknown"] * Xt.shape[1], dtype=object)
            except Exception as err:
                print(f"  [warn] Could not transform '{step_name}': {err}")

    if label:
        print(f"  [{label}] features entering model: {Xt.shape[1]} "
              f"(names tracked: {len(names)})")
    return Xt, names, grps

# -----------------------------------------------------------------------------
# SECTION C  --  Colour palette (one colour per feature category)
# -----------------------------------------------------------------------------

_GRP_PAL = {
    "Process Variables":          "#e63946",
    "Process Interactions":       "#f4722b",
    "Metal Center (mendeleev)":   "#2a9d8f",
    "Metal Precursor Complex":    "#48cae4",
    "Linker ChemBERT":            "#6a4c93",
    "Mod ChemBERT":               "#9b5de5",
    "Linker Physchem/FP":         "#b5a0d8",
    "Mod Physchem/FP":            "#c8b6e2",
    "Linker Morgan FP":           "#457b9d",
    "Modulator Morgan FP":        "#1d3557",
    "Modulator Equiv.":           "#4e9af1",
    "Precursor Ligand FP":        "#235789",
    "Precursor Ligand RAC":       "#52b788",
    "Modulator RAC":              "#74c69d",
    "Ligand TEP (Electronic)":    "#f4a261",
    "Ligand Sterics":             "#e76f51",
    "Reaction FP (DRFP)":         "#264653",
    "3D SOAP Descriptor":         "#2b9348",
    "3D SOAP (Precursor)":        "#2b9348",
    "3D SOAP (Linker)":           "#55a630",
    "G14 Hub Topology":           "#e9c46a",
    "G14 Hub SMARTS":             "#f4d58d",
    "Linker TTP":                 "#f9c74f",
    "Linker EState":              "#90e0ef",
    "Linker Topological":         "#00b4d8",
    "Linker Torsion FP":          "#0077b6",
    "Linker Atom-Pair FP":        "#023e8a",
    "Halide Features":            "#d4a5a5",
    "Inventory Numeric":          "#a8dadc",
    "Process Variables (raw)":    "#f1faee",
    "KMeans Cluster OHE":         "#adb5bd",
    "CL Embedding":               "#ff006e",
    "Other Structural":           "#888888",
    "Unknown":                    "#cccccc",
}

def _pal(group):
    return _GRP_PAL.get(group, "#888888")


# -----------------------------------------------------------------------------
# SECTION D  --  Main SHAP analysis + three-plot output per pipeline
# -----------------------------------------------------------------------------


def _get_ordinal_step(fitted_pipe):
    for key in ("ordinal_xgb", "ordinalxgb", "ordinal_rf", "ordinalrf"):
        if key in fitted_pipe.named_steps:
            return key, fitted_pipe.named_steps[key]
    raise KeyError(
        f"No ordinal step found. Available steps: {list(fitted_pipe.named_steps.keys())}"
    )


def _normalize_shap_values(shap_values):
    if isinstance(shap_values, list):
        if len(shap_values) == 2:
            return np.asarray(shap_values[1])   # positive class
        return np.asarray(shap_values[-1])

    sv = np.asarray(shap_values)
    if sv.ndim == 3:
        return sv[..., -1]
    return sv


def run_shap_featurized(pipe_label, pipe, X, y, X_names, X_groups, top_n=15):
    print(f"\n{'='*70}")
    print(f"  SHAP  >>  {pipe_label}")
    print(f"{'='*70}")

    fitted = clone(pipe)
    fitted.fit(X, y)

    X_model, feat_names, feat_grps = transform_with_names(
        fitted, X, X_names, X_groups, label=pipe_label
    )

    ordinal_key, ordinal_step = _get_ordinal_step(fitted)
    print(f"  Using ordinal step: {ordinal_key}")

    classifiers = ordinal_step.classifiers_

    thresh_abs = {}
    shap_stack = []

    for thresh, model in classifiers.items():
        explainer = shap.TreeExplainer(model)
        sv = _normalize_shap_values(explainer.shap_values(X_model))

        if sv.shape[1] != len(feat_names):
            raise ValueError(
                f"SHAP width mismatch for {pipe_label} threshold {thresh}: "
                f"{sv.shape[1]} vs {len(feat_names)} tracked names"
            )

        thresh_abs[thresh] = np.abs(sv).mean(axis=0)
        shap_stack.append(sv)

    mean_abs_shap = np.stack(list(thresh_abs.values()), axis=0).mean(axis=0)
    shap_avg      = np.stack(shap_stack, axis=0).mean(axis=0)

    df_imp = pd.DataFrame({
        "feature":       feat_names,
        "group":         feat_grps,
        "mean_abs_shap": mean_abs_shap,
    }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

    df_imp.to_csv(f"shap_named_{pipe_label}.csv", index=False)
    print(f"  Saved: shap_named_{pipe_label}.csv")

    # ── Plot 1 -- Feature-group bar chart (SUM) ───────────────────────────
    df_grp = (
        df_imp.groupby("group")["mean_abs_shap"]
        .sum()
        .sort_values(ascending=True)
    )
    df_grp_cnt = df_imp.groupby("group")["mean_abs_shap"].count()

    fig1, ax1 = plt.subplots(figsize=(11, max(5, 0.42 * len(df_grp))))
    cols1 = [_pal(g) for g in df_grp.index]
    bars1 = ax1.barh(
        df_grp.index, df_grp.values,
        color=cols1, edgecolor="white", linewidth=0.6
    )
    xmax = df_grp.max() if len(df_grp) else 1.0
    for bar, val, grp in zip(bars1, df_grp.values, df_grp.index):
        n_feat = df_grp_cnt[grp]
        ax1.text(
            val + xmax * 0.01,
            bar.get_y() + bar.get_height() / 2,
            f"{val:.4f}  (n={n_feat})",
            va="center", ha="left", fontsize=7.5
        )
    ax1.set_xlabel(
        "Sum of Mean |SHAP value|  (summed across Frank-Hall thresholds)",
        fontsize=11
    )
    ax1.set_title(
        f"{pipe_label}\nSHAP Feature-Category Importance (total)  --  LVMOF Crystallinity",
        fontsize=12, fontweight="bold"
    )
    ax1.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    fig1.savefig(f"shap_group_{pipe_label}.png", dpi=180, bbox_inches="tight")
    plt.show()
    print(f"  Saved: shap_group_{pipe_label}.png")

    (
        df_grp.sort_values(ascending=False)
        .reset_index()
        .rename(columns={"mean_abs_shap": "sum_mean_abs_shap"})
        .assign(n_features=lambda d: d["group"].map(df_grp_cnt))
        .to_csv(f"shap_group_{pipe_label}.csv", index=False)
    )
    print(f"  Saved: shap_group_{pipe_label}.csv")

    # ── Plot 1b -- Feature-group bar chart (MEAN, size-normalised) ────────
    df_grp_avg = (
        df_imp.groupby("group")["mean_abs_shap"]
        .mean()
        .sort_values(ascending=True)
    )

    fig1b, ax1b = plt.subplots(figsize=(11, max(5, 0.42 * len(df_grp_avg))))
    cols1b = [_pal(g) for g in df_grp_avg.index]
    bars1b = ax1b.barh(
        df_grp_avg.index, df_grp_avg.values,
        color=cols1b, edgecolor="white", linewidth=0.6
    )
    xmax1b = df_grp_avg.max() if len(df_grp_avg) else 1.0
    for bar, val, grp in zip(bars1b, df_grp_avg.values, df_grp_avg.index):
        n_feat = df_grp_cnt[grp]
        ax1b.text(
            val + xmax1b * 0.01,
            bar.get_y() + bar.get_height() / 2,
            f"{val:.5f}  (n={n_feat})",
            va="center", ha="left", fontsize=7.5
        )
    ax1b.set_xlabel(
        "Mean |SHAP value| per feature  (averaged across Frank-Hall thresholds)",
        fontsize=11
    )
    ax1b.set_title(
        f"{pipe_label}\nSHAP Feature-Category Importance (size-normalised)  --  LVMOF Crystallinity",
        fontsize=12, fontweight="bold"
    )
    ax1b.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    fig1b.savefig(f"shap_group_avg_{pipe_label}.png", dpi=180, bbox_inches="tight")
    plt.show()
    print(f"  Saved: shap_group_avg_{pipe_label}.png")

    (
        df_grp_avg.sort_values(ascending=False)
        .reset_index()
        .rename(columns={"mean_abs_shap": "mean_per_feature_shap"})
        .assign(n_features=lambda d: d["group"].map(df_grp_cnt))
        .to_csv(f"shap_group_avg_{pipe_label}.csv", index=False)
    )
    print(f"  Saved: shap_group_avg_{pipe_label}.csv")

    # ── Plot 2 -- Top-N individual features (bar, colour = category) ──────
    top_df = df_imp.head(top_n).iloc[::-1]
    cols2  = [_pal(g) for g in top_df["group"]]

    fig2, ax2 = plt.subplots(figsize=(11, max(6, 0.38 * top_n)))
    ax2.barh(
        top_df["feature"], top_df["mean_abs_shap"],
        color=cols2, edgecolor="white", linewidth=0.4
    )
    legend_h = [Patch(facecolor=_pal(g), label=g) for g in pd.unique(top_df["group"])]
    ax2.legend(
        handles=legend_h, loc="lower right", fontsize=7.5,
        framealpha=0.8, title="Feature Category", title_fontsize=8
    )
    ax2.set_xlabel("Mean |SHAP value|", fontsize=11)
    ax2.set_title(
        f"{pipe_label}  --  Top {top_n} Individual Features\n"
        "(colour = feature category)",
        fontsize=12, fontweight="bold"
    )
    ax2.spines[["top", "right"]].set_visible(False)
    plt.tight_layout()
    fig2.savefig(f"shap_top{top_n}_{pipe_label}.png", dpi=180, bbox_inches="tight")
    plt.show()
    print(f"  Saved: shap_top{top_n}_{pipe_label}.png")

    # ── Plot 3 -- SHAP beeswarm (signed, top-N, avg over thresholds) ──────
    name_to_idx = {nm: i for i, nm in enumerate(feat_names)}
    top_feat_ord = [nm for nm in df_imp["feature"].iloc[:top_n] if nm in name_to_idx]
    top_cols = [name_to_idx[nm] for nm in top_feat_ord]

    try:
        shap.summary_plot(
            shap_avg[:, top_cols],
            X_model[:, top_cols],
            feature_names=top_feat_ord,
            max_display=len(top_cols),
            show=False,
            plot_type="dot",
            color_bar_label="Feature value (normalised)",
        )
        plt.title(
            f"{pipe_label}  --  SHAP Beeswarm  (avg over ordinal thresholds)\n"
            f"Top {len(top_cols)} features  |  positive SHAP => pushes toward Crystalline",
            fontsize=9, fontweight="bold"
        )
        plt.tight_layout()
        plt.savefig(f"shap_beeswarm_{pipe_label}.png", dpi=180, bbox_inches="tight")
        plt.show()
        print(f"  Saved: shap_beeswarm_{pipe_label}.png")
    except Exception as _e:
        plt.close("all")
        print(f"  Beeswarm skipped: {_e}")

    print(f"\n  Top 15 features -- {pipe_label}")
    print(f"  {'Feature':<45} {'Category':<30} {'Mean|SHAP|':>10}")
    print(f"  {'-'*88}")
    for _, row in df_imp.head(15).iterrows():
        print(
            f"  {str(row['feature']):<45} {str(row['group']):<30} "
            f"{row['mean_abs_shap']:>10.5f}"
        )

    return df_imp


# -----------------------------------------------------------------------------
# SECTION E  --  Run for both XGB pipelines
# -----------------------------------------------------------------------------

ALL_SHAP_PIPES = [
    ("XGB_MI_only",    pipe_xgb_mi),
    #("XGB_CL_plus_MI", pipe_xgb_cl_mi),
    ("RF_MI_only",     pipe_rf_mi),
    #("RF_CL_plus_MI",  pipe_rf_cl_mi),
]

_all_shap = {}
for _label, _pipe in ALL_SHAP_PIPES:
    _all_shap[_label] = run_shap_featurized(
        pipe_label=_label,
        pipe=_pipe,
        X=X, y=y,
        X_names=_X_NAMES,
        X_groups=_X_GROUPS,
        top_n=25,
    )


# -----------------------------------------------------------------------------
# SECTION F  --  Side-by-side group comparison (MI-only vs CL+MI)
# -----------------------------------------------------------------------------
print("\n--- Comparing feature categories across all pipelines ---")

_all_grps = sorted({
    g for df in _all_shap.values()
    for g in df["group"].unique()
})

_grp_df = pd.DataFrame(
    {
        lbl: df.groupby("group")["mean_abs_shap"].sum().reindex(_all_grps, fill_value=0.0)
        for lbl, df in _all_shap.items()
    },
    index=_all_grps,
)
_grp_df = _grp_df.loc[_grp_df.max(axis=1).sort_values().index]

fig4, ax4 = plt.subplots(figsize=(13, max(8, 0.44 * len(_grp_df))))
_y = np.arange(len(_grp_df))

_pcs = ["#457b9d", "#e63946", "#2a9d8f", "#9b5de5"]
_n_cols = len(_grp_df.columns)
_w = 0.8 / _n_cols

for _k, _col in enumerate(_grp_df.columns):
    _offset = (_k - (_n_cols - 1) / 2) * _w
    ax4.barh(
        _y + _offset, _grp_df[_col],
        height=_w, label=_col, color=_pcs[_k % len(_pcs)], alpha=0.85
    )

ax4.set_yticks(_y)
ax4.set_yticklabels(_grp_df.index, fontsize=8.5)
ax4.set_xlabel("Sum of Mean |SHAP value|  (avg over ordinal thresholds)", fontsize=11)
ax4.set_title(
    "RF and XGB Pipelines\n"
    "Feature-Category SHAP Importance -- LVMOF Crystallinity Prediction",
    fontsize=13, fontweight="bold"
)
ax4.legend(fontsize=9, loc="lower right")
ax4.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
fig4.savefig("shap_group_comparison.png", dpi=180, bbox_inches="tight")
plt.show()
print("Saved: shap_group_comparison.png")

(
    _grp_df.reset_index()
    .rename(columns={"index": "feature_group"})
    .to_csv("shap_group_comparison.csv", index=False)
)
print("Saved: shap_group_comparison.csv")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# ROC + PRC comparison block
# Uses out-of-fold predicted probabilities from cross_val_predict
# Default: one-vs-rest for class 2 (Crystalline)
# Change POSITIVE_CLASSES to [1, 2] or [0, 1, 2] if you want more panels
# ═══════════════════════════════════════════════════════════════════════════

from sklearn.model_selection import cross_val_predict
from sklearn.metrics import (
    roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score
)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ── Settings ───────────────────────────────────────────────────────────────
POSITIVE_CLASSES = [2]   # default: crystalline vs rest
CLASS_LABELS = {0: "Amorphous", 1: "Partial", 2: "Crystalline"}

# Use n_jobs=1 for CL pipelines because they train PyTorch inside CV
PIPELINES_TO_COMPARE = [
    ("RF | MI only",  pipe_rf_mi,     -1),
    ("RF | CL + MI",  pipe_rf_cl_mi,   1),
    ("XGB | MI only", pipe_xgb_mi,    -1),
    ("XGB | CL + MI", pipe_xgb_cl_mi,  1),
]

# ── Step 1: get out-of-fold probabilities for each pipeline ───────────────
oof_pred_proba = {}
summary_rows = []

print("\n─── Generating out-of-fold probabilities ─────────────────────────")
for name, pipe, n_jobs_cvpred in PIPELINES_TO_COMPARE:
    print(f"Running: {name}")
    proba = cross_val_predict(
        pipe,
        X, y,
        cv=cv,
        groups=groups,
        method="predict_proba",
        n_jobs=n_jobs_cvpred,
        verbose=0,
    )
    oof_pred_proba[name] = proba

# ── Step 2: plot ROC + PRC for each positive class ────────────────────────
n_rows = len(POSITIVE_CLASSES)
fig, axes = plt.subplots(
    n_rows, 2,
    figsize=(14, 5 * n_rows),
    squeeze=False
)

for r, pos_class in enumerate(POSITIVE_CLASSES):
    y_bin = (y == pos_class).astype(int)
    prevalence = y_bin.mean()

    ax_roc = axes[r, 0]
    ax_pr  = axes[r, 1]

    for name, _, _ in PIPELINES_TO_COMPARE:
        proba = oof_pred_proba[name][:, pos_class]

        # ROC
        fpr, tpr, _ = roc_curve(y_bin, proba)
        auroc = roc_auc_score(y_bin, proba)

        # PRC
        precision, recall, _ = precision_recall_curve(y_bin, proba)
        ap = average_precision_score(y_bin, proba)

        ax_roc.plot(fpr, tpr, lw=2, label=f"{name} (AUC={auroc:.3f})")
        ax_pr.plot(recall, precision, lw=2, label=f"{name} (AP={ap:.3f})")

        summary_rows.append({
            "positive_class": pos_class,
            "class_name": CLASS_LABELS.get(pos_class, str(pos_class)),
            "pipeline": name,
            "roc_auc": auroc,
            "average_precision": ap,
            "prevalence": prevalence,
        })

    # ROC cosmetics
    ax_roc.plot([0, 1], [0, 1], linestyle="--", color="gray", lw=1)
    ax_roc.set_title(f"ROC — {CLASS_LABELS.get(pos_class, pos_class)} vs Rest")
    ax_roc.set_xlabel("False Positive Rate")
    ax_roc.set_ylabel("True Positive Rate")
    ax_roc.legend(fontsize=9)
    ax_roc.grid(alpha=0.25)

    # PR cosmetics
    ax_pr.axhline(prevalence, linestyle="--", color="gray", lw=1,
                  label=f"No-skill baseline = {prevalence:.3f}")
    ax_pr.set_title(f"Precision-Recall — {CLASS_LABELS.get(pos_class, pos_class)} vs Rest")
    ax_pr.set_xlabel("Recall")
    ax_pr.set_ylabel("Precision")
    ax_pr.legend(fontsize=9)
    ax_pr.grid(alpha=0.25)

plt.tight_layout()
plt.savefig("roc_prc_comparison.png", dpi=180, bbox_inches="tight")
plt.show()
print("Saved: roc_prc_comparison.png")

# ── Step 3: print numeric summary ──────────────────────────────────────────
results_auc = (
    pd.DataFrame(summary_rows)
      .sort_values(["positive_class", "roc_auc", "average_precision"],
                   ascending=[True, False, False])
      .reset_index(drop=True)
)

print("\n─── AUC / AP summary ─────────────────────────────────────────────")
for pos_class in POSITIVE_CLASSES:
    cls_name = CLASS_LABELS.get(pos_class, str(pos_class))
    sub = results_auc[results_auc["positive_class"] == pos_class].copy()

    print(f"\n{cls_name} vs Rest")
    print(sub[["pipeline", "roc_auc", "average_precision", "prevalence"]]
            .to_string(index=False, float_format=lambda x: f"{x:.4f}"))

# Optional: save table
results_auc.to_csv("roc_prc_auc_summary.csv", index=False)
print("\nSaved: roc_prc_auc_summary.csv")


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Learning curves for all four pipelines
# Default metric = QWK
# Also prints the final train/validation score at the largest train size
# ═══════════════════════════════════════════════════════════════════════════

from sklearn.model_selection import learning_curve
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ── Settings ───────────────────────────────────────────────────────────────
LC_SCORING_KEY = "qwk"   # must exist in your scoring_ordinal dict
LC_SCORE_NAME  = "QWK"
TRAIN_SIZES_FRAC = np.linspace(0.2, 1.0, 5)

PIPELINES_TO_COMPARE = [
    ("RF | MI only",  pipe_rf_mi,     -1),
    ("RF | CL + MI",  pipe_rf_cl_mi,   1),
    ("XGB | MI only", pipe_xgb_mi,    -1),
    ("XGB | CL + MI", pipe_xgb_cl_mi,  1),
]

# ── Helper ────────────────────────────────────────────────────────────────
def compute_learning_curve(name, pipe, n_jobs_lc, scoring_key="qwk"):
    train_sizes, train_scores, val_scores = learning_curve(
        estimator=pipe,
        X=X,
        y=y,
        groups=groups,
        cv=cv,
        scoring=scoring_ordinal[scoring_key],
        train_sizes=TRAIN_SIZES_FRAC,
        n_jobs=n_jobs_lc,
        shuffle=True,
        random_state=42,
        error_score="raise",
    )

    out = {
        "name": name,
        "train_sizes": train_sizes,
        "train_scores": train_scores,
        "val_scores": val_scores,
        "train_mean": train_scores.mean(axis=1),
        "train_std": train_scores.std(axis=1),
        "val_mean": val_scores.mean(axis=1),
        "val_std": val_scores.std(axis=1),
    }
    return out

# ── Compute curves ────────────────────────────────────────────────────────
learning_curve_results = []
print("\n─── Computing learning curves ─────────────────────────────")
for name, pipe, n_jobs_lc in PIPELINES_TO_COMPARE:
    print(f"Running: {name}")
    res = compute_learning_curve(name, pipe, n_jobs_lc, scoring_key=LC_SCORING_KEY)
    learning_curve_results.append(res)

# ── Plot curves ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.ravel()

for ax, res in zip(axes, learning_curve_results):
    ts = res["train_sizes"]
    tr_m, tr_s = res["train_mean"], res["train_std"]
    va_m, va_s = res["val_mean"], res["val_std"]

    ax.plot(ts, tr_m, marker="o", lw=2, label=f"Train {LC_SCORE_NAME}", color="tab:blue")
    ax.fill_between(ts, tr_m - tr_s, tr_m + tr_s, alpha=0.18, color="tab:blue")

    ax.plot(ts, va_m, marker="s", lw=2, label=f"Validation {LC_SCORE_NAME}", color="tab:orange")
    ax.fill_between(ts, va_m - va_s, va_m + va_s, alpha=0.18, color="tab:orange")

    ax.set_title(res["name"])
    ax.set_xlabel("Training samples")
    ax.set_ylabel(LC_SCORE_NAME)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig("learning_curves_qwk.png", dpi=180, bbox_inches="tight")
plt.show()
print("Saved: learning_curves_qwk.png")

# ── Print numeric summary ─────────────────────────────────────────────────
summary_rows = []
for res in learning_curve_results:
    for i, n_train in enumerate(res["train_sizes"]):
        summary_rows.append({
            "pipeline": res["name"],
            "n_train": int(n_train),
            "train_mean": float(res["train_mean"][i]),
            "train_std": float(res["train_std"][i]),
            "val_mean": float(res["val_mean"][i]),
            "val_std": float(res["val_std"][i]),
            "generalization_gap": float(res["train_mean"][i] - res["val_mean"][i]),
        })

lc_df = pd.DataFrame(summary_rows)
lc_df.to_csv("learning_curve_summary_qwk.csv", index=False)
print("Saved: learning_curve_summary_qwk.csv")

print("\n─── Learning curve summary (largest train size) ─────────────────")
final_rows = (
    lc_df.sort_values(["pipeline", "n_train"])
         .groupby("pipeline", as_index=False)
         .tail(1)
         .sort_values("val_mean", ascending=False)
)

for _, row in final_rows.iterrows():
    print(
        f"{row['pipeline']}\n"
        f"  n_train={int(row['n_train'])}\n"
        f"  Train {LC_SCORE_NAME}: {row['train_mean']:.4f} ± {row['train_std']:.4f}\n"
        f"  Valid {LC_SCORE_NAME}: {row['val_mean']:.4f} ± {row['val_std']:.4f}\n"
        f"  Gap: {row['generalization_gap']:.4f}\n"
    )

# ── Optional: quick overfitting flag ──────────────────────────────────────
print("\n─── Quick interpretation ───────────────────────────────────────")
for _, row in final_rows.iterrows():
    gap = row["generalization_gap"]
    if gap > 0.10:
        flag = "possible overfitting"
    elif gap > 0.04:
        flag = "moderate gap"
    else:
        flag = "fairly tight train/validation fit"
    print(f"{row['pipeline']}: {flag}")


#Error Analysis

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# Confusion matrices for all four pipelines
# Produces:
#   1) raw-count confusion matrices
#   2) row-normalized confusion matrices
#   3) printed numeric matrices
# ═══════════════════════════════════════════════════════════════════════════

from sklearn.model_selection import cross_val_predict
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

CLASS_LABELS = ["Amorphous", "Partial", "Crystalline"]
CLASS_IDS = [0, 1, 2]

PIPELINES_TO_COMPARE = [
    ("RF | MI only",  pipe_rf_mi,     -1),
    ("RF | CL + MI",  pipe_rf_cl_mi,   1),
    ("XGB | MI only", pipe_xgb_mi,    -1),
    ("XGB | CL + MI", pipe_xgb_cl_mi,  1),
]

# ── Step 1: collect out-of-fold predictions ───────────────────────────────
oof_preds = {}
print("\n─── Generating out-of-fold predictions for confusion matrices ───")
for name, pipe, n_jobs_cvpred in PIPELINES_TO_COMPARE:
    print(f"Running: {name}")
    y_pred = cross_val_predict(
        pipe,
        X, y,
        cv=cv,
        groups=groups,
        method="predict",
        n_jobs=n_jobs_cvpred,
        verbose=0,
    )
    oof_preds[name] = y_pred

# ── Step 2: raw-count confusion matrices ──────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.ravel()

for ax, (name, _, _) in zip(axes, PIPELINES_TO_COMPARE):
    cm = confusion_matrix(y, oof_preds[name], labels=CLASS_IDS)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_LABELS)
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(f"{name}\nCounts")

plt.tight_layout()
plt.savefig("confusion_matrices_counts.png", dpi=180, bbox_inches="tight")
plt.show()
print("Saved: confusion_matrices_counts.png")

# ── Step 3: normalized confusion matrices (row-normalized) ────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.ravel()

for ax, (name, _, _) in zip(axes, PIPELINES_TO_COMPARE):
    cm_norm = confusion_matrix(y, oof_preds[name], labels=CLASS_IDS, normalize="true")
    disp = ConfusionMatrixDisplay(confusion_matrix=cm_norm, display_labels=CLASS_LABELS)
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format=".2f")
    ax.set_title(f"{name}\nNormalized by true class")

plt.tight_layout()
plt.savefig("confusion_matrices_normalized.png", dpi=180, bbox_inches="tight")
plt.show()
print("Saved: confusion_matrices_normalized.png")

# ── Step 4: print matrices and save tables ────────────────────────────────
count_tables = []
norm_tables = []

print("\n─── Confusion matrices (counts) ─────────────────────────────────")
for name, _, _ in PIPELINES_TO_COMPARE:
    cm = confusion_matrix(y, oof_preds[name], labels=CLASS_IDS)
    df_cm = pd.DataFrame(cm, index=[f"true_{c}" for c in CLASS_LABELS],
                            columns=[f"pred_{c}" for c in CLASS_LABELS])
    print(f"\n{name}")
    print(df_cm.to_string())

    df_long = df_cm.reset_index().rename(columns={"index": "true_label"})
    df_long.insert(0, "pipeline", name)
    count_tables.append(df_long)

print("\n─── Confusion matrices (row-normalized) ─────────────────────────")
for name, _, _ in PIPELINES_TO_COMPARE:
    cm_norm = confusion_matrix(y, oof_preds[name], labels=CLASS_IDS, normalize="true")
    df_cm_norm = pd.DataFrame(cm_norm, index=[f"true_{c}" for c in CLASS_LABELS],
                                 columns=[f"pred_{c}" for c in CLASS_LABELS])
    print(f"\n{name}")
    print(df_cm_norm.to_string(float_format=lambda x: f"{x:.3f}"))

    df_long = df_cm_norm.reset_index().rename(columns={"index": "true_label"})
    df_long.insert(0, "pipeline", name)
    norm_tables.append(df_long)

pd.concat(count_tables, ignore_index=True).to_csv("confusion_matrix_counts.csv", index=False)
pd.concat(norm_tables, ignore_index=True).to_csv("confusion_matrix_normalized.csv", index=False)

print("\nSaved: confusion_matrix_counts.csv")
print("Saved: confusion_matrix_normalized.csv")


In [ ]:
def _cl_only_steps() -> list:
    """
    CL-only pipeline:
        impute -> vt -> cl(embedding only) -> smote
    """
    return [
        ("impute", SimpleImputer(strategy="median")),
        ("vt", VarianceThreshold(threshold=0.0)),
        (
            "cl",
            ContrastiveMITransformer(
                embedding_dim=CL_EMB_DIM,
                hidden_dim=256,
                loss_mode="supcon",
                negative_class="partial",   # used only in triplet mode
                temperature=0.07,
                margin=0.5,
                epochs=15,
                batch_size=128,
                lr=1e-3,
                weight_decay=1e-4,
                n_triplets=4000,
                balanced_batches=True,
                scale_for_cl=True,
                concat_original=False,      # <<< CL-only: no raw features
                device=None,
                random_state=42,
                verbose=False,
            ),
        ),
        (
            "smote",
            SMOTE(
                sampling_strategy=SMOTE_STRATEGY,
                k_neighbors=5,
                random_state=42,
            ),
        ),
    ]
def make_rf_pipe_cl_only(rf_params):
    return ImbPipeline(
        steps=_cl_only_steps() + [
            (
                "ordinal_rf",
                FrankHallOrdinalClassifier(
                    base_estimator=RandomForestClassifier(
                        **rf_params,
                        bootstrap=True,
                        n_jobs=-1,
                        random_state=RANDOM_STATE,
                    )
                ),
            )
        ]
    )


def make_xgb_pipe_cl_only(xgb_params):
    return ImbPipeline(
        steps=_cl_only_steps() + [
            (
                "ordinal_xgb",
                FrankHallOrdinalClassifier(
                    base_estimator=XGBClassifier(
                        **xgb_params,
                        **XGB_FIXED,
                    )
                ),
            )
        ]
    )
# ═══════════════════════════════════════════════════════════════════════════
# Optuna — XGBoost (CL-only)
#     Same search space and CV as MI-only, but pipeline = CL-only.
# ═══════════════════════════════════════════════════════════════════════════

def objective_xgb_cl_only(trial):
    param = {
        "n_estimators":     trial.suggest_int("n_estimators", 100, 800),
        "max_depth":        trial.suggest_int("max_depth", 3, 6),
        "learning_rate":    trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
        "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 3, 10),
        "gamma":            trial.suggest_float("gamma", 0.0, 2.0),
        "reg_alpha":        trial.suggest_float("reg_alpha", 1e-5, 15.0, log=True),
        "reg_lambda":       trial.suggest_float("reg_lambda", 1e-5, 15.0, log=True),
    }

    pipe = make_xgb_pipe_cl_only(param)  # CL-only feature pipeline

    cv_res = cross_validate(
        pipe,
        X, y,
        cv=cv,
        groups=groups,
        scoring=scoring_ordinal,
        n_jobs=1,                 # CL uses PyTorch → avoid multiprocessing
        return_train_score=False,
    )

    fold_qwk = cv_res["test_qwk"]
    fold_mae = cv_res["test_mae"]
    fold_acc = cv_res["test_exact_acc"]
    mean_qwk = np.mean(fold_qwk)

    try:
        is_best = mean_qwk > trial.study.best_value
    except ValueError:
        is_best = True

    if is_best:
        print(f"\n  ↳ [CL-only XGB] Trial {trial.number} per-fold:")
        print(f"    {'Fold':>6} {'QWK':>8} {'MAE':>8} {'ExactAcc':>10} {'Partial_n':>10}")
        for i, (qwk, mae, acc) in enumerate(zip(fold_qwk, fold_mae, fold_acc)):
            val_idx   = list(cv.split(X, y, groups))[i][1]
            n_partial = (y[val_idx] == 1).sum()
            print(f"    {i+1:>6d} {qwk:>8.4f} {mae:>8.4f} {acc:>10.4f} {n_partial:>10d}")
        print(f"    {'Mean':>6} {mean_qwk:>8.4f} {np.mean(fold_mae):>8.4f} "
              f"{np.mean(fold_acc):>10.4f}")
    return mean_qwk


def progress_callback_cl_xgb(study, trial):
    marker = " ◄ NEW BEST" if trial.value == study.best_value else ""
    print(f"[CL-only XGB] Trial {trial.number:3d} | QWK={trial.value:.4f} "
          f"| Best={study.best_value:.4f}{marker}")


study_xgb_cl_only = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    study_name="xgb_cl_mi"
)

# You can adjust n_trials; 30–50 is a reasonable starting point.
study_xgb_cl_only.optimize(objective_xgb_cl_only, n_trials=50,
                           callbacks=[progress_callback_cl_xgb])

print("\nBest XGB (CL-only) QWK:", study_xgb_cl_only.best_value)
print("Best XGB (CL-only) Params:", study_xgb_cl_only.best_params)

best_xgb_cl_only_params = {
    k: v for k, v in study_xgb_cl_only.best_params.items() if k in XGB_TUNED_KEYS
}
# ═══════════════════════════════════════════════════════════════════════════
# Optuna — Random Forest (CL-only)
#     Same RF search space and CV, pipeline = CL-only.
# ═══════════════════════════════════════════════════════════════════════════

def objective_rf_cl_only(trial):
    param = {
        "n_estimators":      trial.suggest_int("n_estimators", 100, 600),
        "max_depth":         trial.suggest_int("max_depth", 5, 30),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 15),
        "min_samples_leaf":  trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features":      trial.suggest_categorical(
            "max_features", ["sqrt", "log2"]
        ),
    }

    pipe = make_rf_pipe_cl_only(param)

    scores = cross_val_score(
        pipe,
        X, y,
        cv=cv,
        groups=groups,
        scoring=scoring_ordinal["qwk"],
        n_jobs=1,   # CL → keep to 1
    )
    return np.mean(scores)


def progress_callback_cl_rf(study, trial):
    marker = " ◄ NEW BEST" if trial.value == study.best_value else ""
    print(f"[CL-only RF] Trial {trial.number:3d} | QWK={trial.value:.4f} "
          f"| Best={study.best_value:.4f}{marker}")


study_rf_cl_only = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    study_name="rf_cl_mi"
)
study_rf_cl_only.optimize(objective_rf_cl_only, n_trials=100,
                          callbacks=[progress_callback_cl_rf])

print("\nBest RF (CL-only) QWK:", study_rf_cl_only.best_value)
print("Best RF (CL-only) Params:", study_rf_cl_only.best_params)

best_rf_cl_only_params = study_rf_cl_only.best_params
pipe_rf_cl_only_tuned  = make_rf_pipe_cl_only(best_rf_cl_only_params)
pipe_xgb_cl_only_tuned = make_xgb_pipe_cl_only(best_xgb_cl_only_params)

print("\n─── FINAL COMPARISON (with tuned CL-only) ───────────────────────")

eval_pipe("RF  | CL only (opt)", pipe_rf_cl_only_tuned, n_jobs=1)
eval_pipe("XGB | CL only (opt)", pipe_xgb_cl_only_tuned, n_jobs=1)


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# ROC + PRC comparison block
# Uses out-of-fold predicted probabilities from cross_val_predict
# Default: one-vs-rest for class 2 (Crystalline)
# Change POSITIVE_CLASSES to [1, 2] or [0, 1, 2] if you want more panels
# ═══════════════════════════════════════════════════════════════════════════

from sklearn.model_selection import cross_val_predict
from sklearn.metrics import (
    roc_curve, roc_auc_score,
    precision_recall_curve, average_precision_score
)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ── Settings ───────────────────────────────────────────────────────────────
POSITIVE_CLASSES = [2]   # default: crystalline vs rest
CLASS_LABELS = {0: "Amorphous", 1: "Partial", 2: "Crystalline"}

# Use n_jobs=1 for CL pipelines because they train PyTorch inside CV
PIPELINES_TO_COMPARE = [
    ("RF | MI only",   pipe_rf_mi,       -1),
    ("RF | CL + MI",   pipe_rf_cl_mi,     1),
    ("RF | CL only",   pipe_rf_cl_only_tuned,   1),
    ("XGB | MI only",  pipe_xgb_mi,      -1),
    ("XGB | CL + MI",  pipe_xgb_cl_mi,    1),
    ("XGB | CL only",  pipe_xgb_cl_only_tuned,  1),
]


# ── Step 1: get out-of-fold probabilities for each pipeline ───────────────
oof_pred_proba = {}
summary_rows = []

print("\n─── Generating out-of-fold probabilities ─────────────────────────")
for name, pipe, n_jobs_cvpred in PIPELINES_TO_COMPARE:
    print(f"Running: {name}")
    proba = cross_val_predict(
        pipe,
        X, y,
        cv=cv,
        groups=groups,
        method="predict_proba",
        n_jobs=n_jobs_cvpred,
        verbose=0,
    )
    oof_pred_proba[name] = proba

# ── Step 2: plot ROC + PRC for each positive class ────────────────────────
n_rows = len(POSITIVE_CLASSES)
fig, axes = plt.subplots(
    n_rows, 2,
    figsize=(14, 5 * n_rows),
    squeeze=False
)

for r, pos_class in enumerate(POSITIVE_CLASSES):
    y_bin = (y == pos_class).astype(int)
    prevalence = y_bin.mean()

    ax_roc = axes[r, 0]
    ax_pr  = axes[r, 1]

    for name, _, _ in PIPELINES_TO_COMPARE:
        proba = oof_pred_proba[name][:, pos_class]

        # ROC
        fpr, tpr, _ = roc_curve(y_bin, proba)
        auroc = roc_auc_score(y_bin, proba)

        # PRC
        precision, recall, _ = precision_recall_curve(y_bin, proba)
        ap = average_precision_score(y_bin, proba)

        ax_roc.plot(fpr, tpr, lw=2, label=f"{name} (AUC={auroc:.3f})")
        ax_pr.plot(recall, precision, lw=2, label=f"{name} (AP={ap:.3f})")

        summary_rows.append({
            "positive_class": pos_class,
            "class_name": CLASS_LABELS.get(pos_class, str(pos_class)),
            "pipeline": name,
            "roc_auc": auroc,
            "average_precision": ap,
            "prevalence": prevalence,
        })

    # ROC cosmetics
    ax_roc.plot([0, 1], [0, 1], linestyle="--", color="gray", lw=1)
    ax_roc.set_title(f"ROC — {CLASS_LABELS.get(pos_class, pos_class)} vs Rest")
    ax_roc.set_xlabel("False Positive Rate")
    ax_roc.set_ylabel("True Positive Rate")
    ax_roc.legend(fontsize=9)
    ax_roc.grid(alpha=0.25)

    # PR cosmetics
    ax_pr.axhline(prevalence, linestyle="--", color="gray", lw=1,
                  label=f"No-skill baseline = {prevalence:.3f}")
    ax_pr.set_title(f"Precision-Recall — {CLASS_LABELS.get(pos_class, pos_class)} vs Rest")
    ax_pr.set_xlabel("Recall")
    ax_pr.set_ylabel("Precision")
    ax_pr.legend(fontsize=9)
    ax_pr.grid(alpha=0.25)

plt.tight_layout()
plt.savefig("roc_prc_comparison.png", dpi=180, bbox_inches="tight")
plt.show()
print("Saved: roc_prc_comparison.png")

# ── Step 3: print numeric summary ──────────────────────────────────────────
results_auc = (
    pd.DataFrame(summary_rows)
      .sort_values(["positive_class", "roc_auc", "average_precision"],
                   ascending=[True, False, False])
      .reset_index(drop=True)
)

print("\n─── AUC / AP summary ─────────────────────────────────────────────")
for pos_class in POSITIVE_CLASSES:
    cls_name = CLASS_LABELS.get(pos_class, str(pos_class))
    sub = results_auc[results_auc["positive_class"] == pos_class].copy()

    print(f"\n{cls_name} vs Rest")
    print(sub[["pipeline", "roc_auc", "average_precision", "prevalence"]]
            .to_string(index=False, float_format=lambda x: f"{x:.4f}"))

# Optional: save table
results_auc.to_csv("roc_prc_auc_summary.csv", index=False)
print("\nSaved: roc_prc_auc_summary.csv")
